# 05 · Corrida del Modelo Hidrológico con Escenarios Futuros

Este notebook ejecuta el **modelo hidrológico distribuido + módulo de embalse** alimentado con:

- Precipitación corregida QDM (umbral 0.5 mm).
- ET Hargreaves corregida.
- LULC + textura de suelo + shapefiles.
- Las 5 series de **descargas proyectadas** (ESC_00 … ESC_100).

**Combinaciones simuladas:**

| Eje | Valores |
|---|---|
| Modelos CMIP6 | `MPI-ESM1-2-LR`, `EC-Earth3-Veg-LR`, `EC-Earth3` |
| SSP | `ssp245`, `ssp585` |
| Umbral P | `u0p5` |
| Descarga | `ESC_00`, `ESC_25`, `ESC_50`, `ESC_75`, `ESC_100` |

Salidas: un **Excel por combinación** con la serie diaria del embalse, más temperaturas anexas.


## 5.1 · Simulación multi-escenario sobre la serie histórica (verificación)

Antes de proyectar al futuro, se valida el flujo multi-escenario sobre la serie 2010-2019 (5 variantes de descarga).


In [ ]:
# ==========================================================
# SCRIPT 2 MODIFICADO
# SIMULACIÓN DISTRIBUIDA + EMBALSE
# MULTI-ESCENARIO (5 ESCENARIOS FÍSICAMENTE REPRESENTATIVOS)
# SERIE COMPLETA 2010-2019
# CALIBRACIÓN / VALIDACIÓN
# EXCLUYENDO UN PERIODO SOLO EN MÉTRICAS Y PLOT OBSERVADO
# + GUARDADO DE RESULTADOS
# + BANDA SOMBREADA ±15% DE LA SERIE SIMULADA
# + MÉTRICAS EN LA GRÁFICA
# + COMPARACIÓN ENTRE ESCENARIOS
# ==========================================================

# ==========================================================
# 0) CONFIGURACIÓN DE FECHAS
# ==========================================================
CAL_START = "2010-01-01"
CAL_END   = "2017-01-01"

VAL_START = "2017-01-02"
VAL_END   = "2019-12-31"

SERIE_START = "2010-01-01"
SERIE_END   = "2019-12-31"

EXC_START = "2014-01-01"
EXC_END   = "2015-06-07"


# ==========================================================
# 1) LIBRERÍAS
# ==========================================================
import os
import glob
import json
import calendar
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)


# ==========================================================
# 2) RUTAS / PARÁMETROS GENERALES
# ==========================================================
BASE_DIR = "/content/gdrive/MyDrive/Project001"

CHIRPS_DIR = os.path.join(BASE_DIR, "CHIRPS_PPT_DAILY_2009_2020_WGS84")
ET_DIR     = os.path.join(BASE_DIR, "ET_HARGREAVES_DAILY_2009_2020_WGS84")

LULC_DIR = os.path.join(BASE_DIR, "LULC_LC_Type1_WGS84")
SOIL_DIR = os.path.join(BASE_DIR, "SOIL_TEXTURE_b0_WGS84")

SHP_DIR = os.path.join(BASE_DIR, "shp")
AOI_CUENCA_PATH  = os.path.join(SHP_DIR, "Cuenca-es.geojson")
AOI_EMBALSE_PATH = os.path.join(SHP_DIR, "Embalse.geojson")

QM_MODEL_PATH = os.path.join(BASE_DIR, "QM_model_monthly.csv")
DESCARGAS_CSV = os.path.join(BASE_DIR, "descargas2.csv")

OUT_DIR = os.path.join(BASE_DIR, "resultados_modelo_5_escenarios")
os.makedirs(OUT_DIR, exist_ok=True)

START_YEAR = 2010
END_YEAR   = 2019

VOL_MUERTO = 41e6
VOL_MAX    = 148.80e6
AREA_CUENCA_TOTAL = 196.33e6
V_INICIAL = 57.80e6


# ==========================================================
# 3) ESCENARIOS SELECCIONADOS
# ==========================================================
ESCENARIOS = {
    "ESC_01_G13_I4": {
        "descripcion": "GEN 13/20 - IND 4/42",
        "g7": 1,
        "g9": 1,
        "n": 2,
        "K": 2.0,
        "k_infil": 0.6592839619082883,
        "k_et": 0.8291335176411617,
        "Vs": 0.00918186471522207,
        "CN10_g7": 95.72124246470727,
        "CN16_g7": 85.47696321777651,
        "CN10_g9": 95.97957958640941,
        "CN16_g9": 64.39680341414072,
        "k_effP": 1.7661125944094203,
        "k_q": 1.7070861885202886,
        "lambda_ia": 0.0273135448843369,
    },
    "ESC_02_G20_I22": {
        "descripcion": "GEN 20/20 - IND 22/42",
        "g7": 1,
        "g9": 1,
        "n": 2,
        "K": 4.76206781048957,
        "k_infil": 0.6953880721605649,
        "k_et": 0.7409106141853031,
        "Vs": 0.01,
        "CN10_g7": 94.55309804004582,
        "CN16_g7": 90.23220646594083,
        "CN10_g9": 96.31605148608226,
        "CN16_g9": 70.97588497632263,
        "k_effP": 1.773605143462856,
        "k_q": 1.7782617794119575,
        "lambda_ia": 0.03317415120962344,
    },
    "ESC_03_G14_I18": {
        "descripcion": "GEN 14/20 - IND 18/42",
        "g7": 2,
        "g9": 2,
        "n": 2,
        "K": 5.003550931,
        "k_infil": 0.695388072,
        "k_et": 0.773892589,
        "Vs": 0.00959497,
        "CN10_g7": 93.4369078,
        "CN16_g7": 90.15223472,
        "CN10_g9": 96.50469944,
        "CN16_g9": 62.84461363,
        "k_effP": 1.800085404,
        "k_q": 1.782850951,
        "lambda_ia": 0.005,
    },
    "ESC_04_G20_I21": {
        "descripcion": "GEN 20/20 - IND 21/42",
        "g7": 1,
        "g9": 2,
        "n": 2,
        "K": 4.76206781048957,
        "k_infil": 0.7890003663248961,
        "k_et": 0.6532412536523772,
        "Vs": 0.009945498730034889,
        "CN10_g7": 94.55309804004582,
        "CN16_g7": 89.58997744975092,
        "CN10_g9": 96.3003091768483,
        "CN16_g9": 66.89345682860497,
        "k_effP": 1.817156163223742,
        "k_q": 1.7679524705766723,
        "lambda_ia": 0.005,
    },
    "ESC_05_G13_I6": {
        "descripcion": "GEN 13/20 - IND 6/42",
        "g7": 1,
        "g9": 2,
        "n": 4,
        "K": 4.313107873728896,
        "k_infil": 0.6953880721605649,
        "k_et": 0.8082241252469766,
        "Vs": 0.009594970137013482,
        "CN10_g7": 95.72124246470727,
        "CN16_g7": 85.71535662377057,
        "CN10_g9": 96.31605148608226,
        "CN16_g9": 66.28439128813265,
        "k_effP": 1.812739987216763,
        "k_q": 1.7089490774160854,
        "lambda_ia": 0.03914373443714243,
    },
}


# ==========================================================
# 4) UTILIDADES GENERALES
# ==========================================================
def load_geoms(geojson_path: str):
    if not os.path.exists(geojson_path):
        raise FileNotFoundError(f"No existe AOI: {geojson_path}")
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        raise ValueError(f"AOI vacío: {geojson_path}")
    geom = gdf.geometry.union_all()
    return [geom]


def read_clip_stack(tif_path: str, geoms):
    if not os.path.exists(tif_path):
        raise FileNotFoundError(f"No existe raster: {tif_path}")
    with rasterio.open(tif_path) as src:
        out_img, _ = mask(src, geoms, crop=True, filled=True)
        nodata = src.nodata
    return out_img, nodata


def nanmean_masked(a: np.ndarray, nodata=None) -> float:
    a = a.astype("float64", copy=False)
    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)
    return float(np.nanmean(a))


def find_month_file(folder: str, prefix: str, year: int, month: int) -> str:
    pattern = os.path.join(folder, f"{prefix}_{year}_{month:02d}_WGS84*.*")
    hits = sorted(glob.glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No encontré archivo con patrón:\n  {pattern}")
    return hits[0]


def find_single_tif(folder: str) -> str:
    hits = sorted(glob.glob(os.path.join(folder, "*.tif"))) + sorted(glob.glob(os.path.join(folder, "*.tiff")))
    if not hits:
        raise FileNotFoundError(f"No encontré tif/tiff en: {folder}")
    return hits[0]


# ==========================================================
# 5) QM MENSUAL
# ==========================================================
def load_qm_table(csv_path: str):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No existe QM CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"mes", "x", "y"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"QM_model_monthly.csv debe tener columnas {required}. Tiene: {list(df.columns)}")

    qm = {}
    for _, r in df.iterrows():
        mes = int(r["mes"])
        x = json.loads(r["x"]) if isinstance(r["x"], str) else r["x"]
        y = json.loads(r["y"]) if isinstance(r["y"], str) else r["y"]

        x = np.array(x, dtype=float)
        y = np.array(y, dtype=float)

        if x.size < 2 or y.size < 2 or x.size != y.size:
            raise ValueError(f"QM inválido en mes {mes}: x={x.size}, y={y.size}")

        qm[mes] = (x, y)

    faltantes = [m for m in range(1, 13) if m not in qm]
    if faltantes:
        raise ValueError(f"Faltan meses en QM: {faltantes}")

    return qm


def qm_correct_array(arr: np.ndarray, month: int, QM_TABLE):
    x, y = QM_TABLE[month]
    idx = np.argsort(x)
    x2 = x[idx]
    y2 = y[idx]
    flat = arr.reshape(-1).astype("float64", copy=False)
    out = np.interp(flat, x2, y2).reshape(arr.shape)
    return np.maximum(out, 0.0)


# ==========================================================
# 6) SCS-CN DISTRIBUIDO
# ==========================================================
CN_TABLE = {
    1: {1:35, 2:25, 3:45, 4:39, 5:45, 6:49, 7:68, 8:36, 9:45, 10:30, 11:95, 12:67, 13:72, 14:63, 15:100, 16:74, 17:100},
    2: {1:50, 2:55, 3:66, 4:61, 5:66, 6:69, 7:79, 8:60, 9:66, 10:58, 11:95, 12:78, 13:82, 14:75, 15:100, 16:84, 17:100},
    3: {1:73, 2:70, 3:77, 4:74, 5:77, 6:79, 7:86, 8:73, 9:77, 10:71, 11:95, 12:85, 13:87, 14:83, 15:100, 16:90, 17:100},
    4: {1:79, 2:77, 3:83, 4:80, 5:83, 6:89, 7:89, 8:79, 9:83, 10:78, 11:95, 12:89, 13:89, 14:87, 15:100, 16:92, 17:100},
}


def soil_group_from_texture(soil_class, g7=2, g9=2):
    soil_grp = np.zeros_like(soil_class, dtype="int16")
    soil_grp = np.where(soil_class > 10, 1, soil_grp)
    soil_grp = np.where((soil_class > 4) & (soil_class <= 10), 2, soil_grp)
    soil_grp = np.where((soil_class > 1) & (soil_class <= 4), 3, soil_grp)
    soil_grp = np.where((soil_class > 0) & (soil_class <= 1), 4, soil_grp)

    soil_grp = np.where(soil_class == 7, int(g7), soil_grp)
    soil_grp = np.where(soil_class == 9, int(g9), soil_grp)
    return soil_grp


def reemplazar_cn_lulc10_16(CN10_g7, CN16_g7, CN10_g9, CN16_g9, g7=2, g9=2):
    cn_table_local = {k: v.copy() for k, v in CN_TABLE.items()}
    cn_table_local[int(g7)][10] = float(CN10_g7)
    cn_table_local[int(g7)][16] = float(CN16_g7)
    cn_table_local[int(g9)][10] = float(CN10_g9)
    cn_table_local[int(g9)][16] = float(CN16_g9)
    return cn_table_local


def build_cn_s_from_cache(LULC_CUENCA, SOIL_CUENCA, cn_table_local, g7=2, g9=2, f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4):
    lulc = LULC_CUENCA.astype("float64")
    soil = SOIL_CUENCA.astype("float64")
    soil_grp = soil_group_from_texture(soil, g7=g7, g9=g9)

    CN2 = np.zeros_like(lulc, dtype="float64")
    CN2 = np.where(soil_grp == 0, 100.0, CN2)

    for grp, lut in cn_table_local.items():
        for lc, cn in lut.items():
            CN2 = np.where((soil_grp == grp) & (lulc == lc), float(cn), CN2)

    CN2 = np.minimum(CN2 * float(f_cn2), 100.0)

    CN1 = CN2 / (2.281 - (CN2 * 0.0128))
    CN3 = CN2 / (0.427 + (CN2 * 0.00573))

    S1 = (25400.0 / CN1 - 254.0) * float(f_s1)
    S2 = (25400.0 / CN2 - 254.0) * float(f_s2)
    S3 = (25400.0 / CN3 - 254.0) * float(f_s3)

    return S1, S2, S3


def rolling_sum_5d(stack_TRC: np.ndarray, prev4=None):
    if prev4 is None:
        prev4 = np.zeros((0,) + stack_TRC.shape[1:], dtype=stack_TRC.dtype)

    combo = np.concatenate([prev4, stack_TRC], axis=0)
    cs = np.cumsum(combo, axis=0)

    amc = np.empty_like(combo, dtype="float64")
    for i in range(combo.shape[0]):
        if i < 4:
            amc[i] = np.sum(combo[:i+1], axis=0)
        else:
            amc[i] = cs[i] - cs[i-5]

    tail4 = combo[-4:] if combo.shape[0] >= 4 else combo
    return amc[-stack_TRC.shape[0]:], tail4


def runoff_scs(ppt, amc5, S1, S2, S3, lambda_ia=0.2):
    S = np.where(amc5 <= 13, S1, S2)
    S = np.where(amc5 > 28, S3, S)

    lam = float(np.clip(lambda_ia, 0.001, 0.8))
    Ia = lam * S

    numer = np.power(ppt - Ia, 2)
    denom = (ppt - Ia) + S

    with np.errstate(divide="ignore", invalid="ignore"):
        Q = np.where(ppt < Ia, 0.0, numer / denom)

    return np.where(np.isfinite(Q), Q, 0.0)


# ==========================================================
# 7) RUTEO
# ==========================================================
def route_linear_reservoir(runoff_mm, K=2.0):
    r = np.asarray(runoff_mm, dtype=float)
    q = np.zeros_like(r)
    alpha = np.exp(-1.0 / max(K, 1e-6))

    for t in range(len(r)):
        q[t] = (1 - alpha) * r[t] if t == 0 else alpha * q[t-1] + (1 - alpha) * r[t]

    return q


def route_nash_cascade(runoff_mm, n=3, K=2.0):
    q = np.asarray(runoff_mm, dtype=float)
    for _ in range(int(n)):
        q = route_linear_reservoir(q, K=K)
    return q


# ==========================================================
# 8) EMBALSE
# ==========================================================
def area_embalse(V):
    D366 = V / 1e6
    area_km2 = (
        8.91946288801159E-08 * D366**4
        - 0.0000346583770904819 * D366**3
        + 0.003919249094071 * D366**2
        - 0.0306477700093546 * D366
        + 2.02308232795992
    )
    return max(area_km2, 0) * 1e6


def volumen_auxiliar(V_prev, esc_mm, A_c, Vd):
    A_emb = area_embalse(V_prev)
    A_aporte = max(A_c - A_emb, 0)
    esc_m = esc_mm / 1000.0
    V_esc = esc_m * A_aporte
    return V_prev + V_esc - Vd


def area_media(V_prev, V_aux):
    return 0.5 * (area_embalse(V_prev) + area_embalse(V_aux))


def aplicar_restricciones(B_i):
    if B_i > VOL_MAX:
        return VOL_MAX, B_i - VOL_MAX
    if B_i < VOL_MUERTO:
        return VOL_MUERTO, 0.0
    return B_i, 0.0


def balance_embalse_diario(V_prev, esc_mm, precip_mm, et_mm, A_c, Vd, Vs_m_d, k_infil=1.0, k_et=1.0):
    V_aux = volumen_auxiliar(V_prev, esc_mm, A_c, Vd)
    A_i = area_media(V_prev, V_aux)

    esc_m = esc_mm / 1000.0
    precip_m = precip_mm / 1000.0
    et_m = (et_mm * k_et) / 1000.0

    V_esc = esc_m * max(A_c - A_i, 0.0)
    V_clima = (precip_m - et_m) * A_i
    Vs_vol = (float(Vs_m_d) * float(k_infil)) * A_i

    B_i = V_prev + V_esc + V_clima - Vd - Vs_vol
    V_emb, V_vertido = aplicar_restricciones(B_i)

    return V_emb, V_vertido, A_i, V_clima, V_esc, Vs_vol, B_i, Vd


# ==========================================================
# 9) CARGA DE DATOS FIJOS
# ==========================================================
roi_cuenca_geoms  = load_geoms(AOI_CUENCA_PATH)
roi_embalse_geoms = load_geoms(AOI_EMBALSE_PATH)

QM_TABLE = load_qm_table(QM_MODEL_PATH)

LULC_TIF = find_single_tif(LULC_DIR)
SOIL_TIF = find_single_tif(SOIL_DIR)

Vdescarga = pd.read_csv(DESCARGAS_CSV, sep=";", encoding="latin-1")
Vdescarga = Vdescarga.rename(columns={
    Vdescarga.columns[0]: "fecha",
    Vdescarga.columns[1]: "descarga_m3_s",
    Vdescarga.columns[2]: "V_emb_real_hm3",
})

Vdescarga["fecha"] = pd.to_datetime(Vdescarga["fecha"], dayfirst=True, errors="coerce").dt.floor("D")
Vdescarga["descarga_m3_s"] = pd.to_numeric(Vdescarga["descarga_m3_s"], errors="coerce")
Vdescarga["V_emb_real_hm3"] = pd.to_numeric(Vdescarga["V_emb_real_hm3"], errors="coerce")

Vdescarga = Vdescarga[
    (Vdescarga["fecha"] >= SERIE_START) &
    (Vdescarga["fecha"] <= SERIE_END)
].dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)

Vdescarga = Vdescarga.drop_duplicates(subset=["fecha"], keep="last")


# ==========================================================
# 10) CACHE RASTERS ESTÁTICOS
# ==========================================================
lulc_stack, _ = read_clip_stack(LULC_TIF, roi_cuenca_geoms)
soil_stack, _ = read_clip_stack(SOIL_TIF, roi_cuenca_geoms)

LULC_CUENCA = lulc_stack[0].astype("int16")
SOIL_CUENCA = soil_stack[0].astype("int16")

print("LULC únicos (cuenca):", np.unique(LULC_CUENCA))
print("SOIL únicos (cuenca):", np.unique(SOIL_CUENCA))


# ==========================================================
# 11) MÉTRICAS
# ==========================================================
def _remove_excluded_period(df):
    d = df.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.floor("D")

    exc_s = pd.to_datetime(EXC_START)
    exc_e = pd.to_datetime(EXC_END)

    return d[~((d["fecha"] >= exc_s) & (d["fecha"] <= exc_e))].reset_index(drop=True)


def _compute_metrics(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    m = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[m]
    obs = obs[m]

    if len(obs) < 10:
        return {
            "n": int(len(obs)),
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan
        }

    if np.std(obs) > 0 and np.std(sim) > 0:
        r = float(np.corrcoef(obs, sim)[0, 1])
        R2 = float(r ** 2)
    else:
        r = 0.0
        R2 = 0.0

    RMSE = float(np.sqrt(np.mean((sim - obs) ** 2)))
    denom = float(np.sum((obs - np.mean(obs)) ** 2))
    NSE = float(1 - np.sum((sim - obs) ** 2) / denom) if denom > 0 else np.nan

    sigma = float(np.std(obs))
    RMSE_n = float(RMSE / sigma) if sigma > 0 else np.nan

    mu_o = float(np.mean(obs))
    mu_s = float(np.mean(sim))
    Bias = float((mu_s - mu_o) / mu_o) if abs(mu_o) > 1e-12 else np.nan
    PBIAS = float(Bias * 100.0) if np.isfinite(Bias) else np.nan

    alpha = float(np.std(sim) / np.std(obs)) if np.std(obs) > 0 else np.nan
    beta = float(mu_s / mu_o) if abs(mu_o) > 1e-12 else np.nan

    if np.isfinite(r) and np.isfinite(alpha) and np.isfinite(beta):
        KGE = float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))
    else:
        KGE = np.nan

    return {
        "n": int(len(obs)),
        "R2": R2,
        "RMSE": RMSE,
        "NSE": NSE,
        "RMSE_n": RMSE_n,
        "Bias": Bias,
        "PBIAS": PBIAS,
        "KGE": KGE
    }


# ==========================================================
# 12) FILTROS DE FECHAS
# ==========================================================
def _slice_df_by_dates(df_res, start, end):
    s = pd.to_datetime(start)
    e = pd.to_datetime(end)

    d = df_res.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.floor("D")

    return d[(d["fecha"] >= s) & (d["fecha"] <= e)].reset_index(drop=True)


def _build_calibration_df(df_res, cal_start, cal_end):
    d = _slice_df_by_dates(df_res, cal_start, cal_end)
    d = _remove_excluded_period(d)
    return d


def _build_validation_df(df_res, val_start, val_end):
    d = _slice_df_by_dates(df_res, val_start, val_end)
    d = _remove_excluded_period(d)
    return d


# ==========================================================
# 13) SIMULACIÓN COMPLETA CON SALIDAS DETALLADAS
# ==========================================================
def simular_detallado(params, metric_start=None, metric_end=None):
    cn_table_local = reemplazar_cn_lulc10_16(
        params["CN10_g7"], params["CN16_g7"],
        params["CN10_g9"], params["CN16_g9"],
        g7=params["g7"], g9=params["g9"]
    )

    S1_CU, S2_CU, S3_CU = build_cn_s_from_cache(
        LULC_CUENCA, SOIL_CUENCA,
        cn_table_local=cn_table_local,
        g7=params["g7"], g9=params["g9"],
        f_cn2=1.0,
        f_s1=1.3, f_s2=1.2, f_s3=1.4
    )

    rows = []
    for year in range(START_YEAR, END_YEAR + 1):
        prev4 = None
        for month in range(1, 13):
            chirps_path = find_month_file(CHIRPS_DIR, "CHIRPS_PPT_DAILY", year, month)
            et_path     = find_month_file(ET_DIR, "ET_HARGREAVES_DAILY", year, month)

            ch_cu_stack, _ = read_clip_stack(chirps_path, roi_cuenca_geoms)
            ppt_cu = qm_correct_array(ch_cu_stack.astype("float64"), month, QM_TABLE)
            ppt_cu_eff = ppt_cu * float(params["k_effP"])

            amc5, prev4 = rolling_sum_5d(ppt_cu_eff, prev4=prev4)
            q_cu = runoff_scs(
                ppt_cu_eff, amc5, S1_CU, S2_CU, S3_CU,
                lambda_ia=float(params["lambda_ia"])
            )

            q_mean = [nanmean_masked(q_cu[i], None) for i in range(q_cu.shape[0])]
            q_mean = [v * float(params["k_q"]) for v in q_mean]

            ppt_cu_mean = [nanmean_masked(ppt_cu_eff[i], None) for i in range(ppt_cu_eff.shape[0])]

            ch_em_stack, ch_em_nodata = read_clip_stack(chirps_path, roi_embalse_geoms)
            ppt_em = qm_correct_array(ch_em_stack.astype("float64"), month, QM_TABLE)
            ppt_em_mean = [nanmean_masked(ppt_em[i], ch_em_nodata) for i in range(ppt_em.shape[0])]
            ppt_em_mean = [v * float(params["k_effP"]) for v in ppt_em_mean]

            et_em_stack, et_em_nodata = read_clip_stack(et_path, roi_embalse_geoms)
            et_em = et_em_stack.astype("float64")
            et_em_mean = [nanmean_masked(et_em[i], et_em_nodata) for i in range(et_em.shape[0])]

            ndays = calendar.monthrange(year, month)[1]
            T = min(ndays, len(q_mean), len(ppt_em_mean), len(et_em_mean), len(ppt_cu_mean))

            for d in range(T):
                rows.append({
                    "fecha": datetime(year, month, d + 1),
                    "ppt_cuenca_eff_mm": float(ppt_cu_mean[d]),
                    "esc_mm": float(q_mean[d]),
                    "precip_mm": float(ppt_em_mean[d]),
                    "et_mm": float(et_em_mean[d]),
                })

    df_forz = pd.DataFrame(rows).sort_values("fecha").reset_index(drop=True)
    df_forz["esc_ruteada_mm"] = route_nash_cascade(
        df_forz["esc_mm"].to_numpy(),
        n=int(params["n"]),
        K=float(params["K"])
    )

    df_forz["fecha"] = pd.to_datetime(df_forz["fecha"]).dt.floor("D")
    df_forz = df_forz.drop_duplicates(subset=["fecha"], keep="last")

    df_join = pd.merge(
        df_forz[["fecha", "ppt_cuenca_eff_mm", "esc_mm", "esc_ruteada_mm", "precip_mm", "et_mm"]],
        Vdescarga[["fecha", "descarga_m3_s", "V_emb_real_hm3"]],
        on="fecha",
        how="inner"
    ).sort_values("fecha").reset_index(drop=True)

    if len(df_join) < 100:
        df_res_full = pd.DataFrame({"fecha": [], "V_emb": [], "V_emb_real": []})
        met = {
            "R2": np.nan, "RMSE": np.nan, "NSE": np.nan,
            "RMSE_n": np.nan, "Bias": np.nan, "PBIAS": np.nan, "KGE": np.nan, "n": 0
        }
        return df_res_full, met

    resultados = []
    V_emb = None

    for i in range(len(df_join)):
        V_prev = V_INICIAL if i == 0 else V_emb

        fecha = df_join.loc[i, "fecha"]
        Vd = float(df_join.loc[i, "descarga_m3_s"]) * 86400.0
        V_obs = float(df_join.loc[i, "V_emb_real_hm3"]) * 1_000_000.0

        esc_mm = float(df_join.loc[i, "esc_ruteada_mm"])
        p_mm   = float(df_join.loc[i, "precip_mm"])
        et_mm  = float(df_join.loc[i, "et_mm"])

        V_emb, V_vert, A_i, V_clima, V_esc, Vs_vol, B_i, Vd = balance_embalse_diario(
            V_prev, esc_mm, p_mm, et_mm,
            AREA_CUENCA_TOTAL, Vd,
            Vs_m_d=float(params["Vs"]),
            k_infil=float(params["k_infil"]),
            k_et=float(params["k_et"])
        )

        resultados.append({
            "fecha": fecha,
            "ppt_cuenca_eff_mm": float(df_join.loc[i, "ppt_cuenca_eff_mm"]),
            "esc_mm_sin_ruteo": float(df_join.loc[i, "esc_mm"]),
            "esc_mm_ruteada": float(df_join.loc[i, "esc_ruteada_mm"]),
            "precip_mm_embalse": p_mm,
            "et_mm_embalse": et_mm,
            "descarga_m3_s": float(df_join.loc[i, "descarga_m3_s"]),
            "descarga_m3_dia": Vd,
            "V_emb": V_emb,
            "V_emb_hm3": V_emb / 1e6,
            "V_emb_real": V_obs,
            "V_emb_real_hm3": V_obs / 1e6,
            "V_vertido": V_vert,
            "area_embalse_m2": A_i,
            "V_clima": V_clima,
            "V_esc": V_esc,
            "Vs_vol": Vs_vol,
            "balance_bruto": B_i
        })

    df_res_full = pd.DataFrame(resultados)
    df_res_full["fecha"] = pd.to_datetime(df_res_full["fecha"]).dt.floor("D")

    if metric_start is None or metric_end is None:
        dmet = df_res_full.copy()
    else:
        dmet = _slice_df_by_dates(df_res_full, metric_start, metric_end)

    dmet = _remove_excluded_period(dmet)
    met = _compute_metrics(dmet["V_emb"].values, dmet["V_emb_real"].values)

    return df_res_full, met


# ==========================================================
# 14) GRAFICADO DE UN ESCENARIO
# ==========================================================
def graficar_escenario(df_full, met_cal, met_val, nombre_esc, descripcion, out_png):
    plt.figure(figsize=(16, 5))

    df_obs_plot = df_full.copy()
    exc_s = pd.to_datetime(EXC_START)
    exc_e = pd.to_datetime(EXC_END)
    mask_exc = (df_obs_plot["fecha"] >= exc_s) & (df_obs_plot["fecha"] <= exc_e)
    df_obs_plot.loc[mask_exc, "V_emb_real"] = np.nan

    sim = df_full["V_emb"].to_numpy(dtype=float)
    sim_low = sim * 0.85
    sim_high = sim * 1.15

    plt.fill_between(
        df_full["fecha"],
        sim_low,
        sim_high,
        alpha=0.18,
        label="Banda ±15% Sim"
    )

    plt.plot(df_obs_plot["fecha"], df_obs_plot["V_emb_real"], linewidth=1.5, label="Obs")
    plt.plot(df_full["fecha"], df_full["V_emb"], linewidth=1.8, label="Sim")

    plt.axvline(pd.to_datetime(CAL_START), linestyle="--", linewidth=1.2)
    plt.axvline(pd.to_datetime(CAL_END), linestyle="--", linewidth=1.2)
    plt.axvline(pd.to_datetime(VAL_START), linestyle="--", linewidth=1.2)

    texto_cal = (
        "CALIBRACIÓN\n"
        f"R²={met_cal['R2']:.3f}\n"
        f"RMSE={met_cal['RMSE']:.2e}\n"
        f"NSE={met_cal['NSE']:.3f}\n"
        f"KGE={met_cal['KGE']:.3f}\n"
        f"Bias={met_cal['Bias']:.4f}"
    )

    texto_val = (
        "VALIDACIÓN\n"
        f"R²={met_val['R2']:.3f}\n"
        f"RMSE={met_val['RMSE']:.2e}\n"
        f"NSE={met_val['NSE']:.3f}\n"
        f"KGE={met_val['KGE']:.3f}\n"
        f"Bias={met_val['Bias']:.4f}"
    )

    plt.text(
        0.015, 0.97, texto_cal,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.text(
        0.165, 0.97, texto_val,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.title(
        f"Volumen embalse 2010–2019 | {nombre_esc}\n"
        f"{descripcion}\n"
        f"Calibración: {CAL_START} a {CAL_END} | Validación: {VAL_START} a {VAL_END}"
    )
    plt.xlabel("Fecha")
    plt.ylabel("Volumen (m3)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.show()


# ==========================================================
# 15) EJECUCIÓN MULTI-ESCENARIO
# ==========================================================
print("=== MODELO MULTI-ESCENARIO ===")

resumen_global = []
params_global = []
series_all = []

for nombre_esc, params in ESCENARIOS.items():
    print("\n" + "=" * 80)
    print(f"Ejecutando {nombre_esc}")
    print(params["descripcion"])
    print("=" * 80)
    print(params)

    df_full, _ = simular_detallado(
        params,
        metric_start=CAL_START,
        metric_end=CAL_END
    )

    df_full["fecha"] = pd.to_datetime(df_full["fecha"]).dt.floor("D")

    df_cal = _build_calibration_df(df_full, CAL_START, CAL_END)
    df_val = _build_validation_df(df_full, VAL_START, VAL_END)

    met_cal = _compute_metrics(df_cal["V_emb"], df_cal["V_emb_real"])
    met_val = _compute_metrics(df_val["V_emb"], df_val["V_emb_real"])

    print("\n--- MÉTRICAS CALIBRACIÓN ---")
    print(met_cal)
    print("--- MÉTRICAS VALIDACIÓN ---")
    print(met_val)

    # Guardar serie individual CSV
    out_csv = os.path.join(OUT_DIR, f"serie_{nombre_esc}.csv")
    df_out = df_full.copy()
    df_out["escenario"] = nombre_esc
    df_out["descripcion"] = params["descripcion"]
    df_out.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # Gráfico individual
    out_png = os.path.join(OUT_DIR, f"grafica_{nombre_esc}.png")
    graficar_escenario(df_full, met_cal, met_val, nombre_esc, params["descripcion"], out_png)

    # Acumulados resumen
    resumen_global.append({
        "escenario": nombre_esc,
        "descripcion": params["descripcion"],
        "etapa": "CALIBRACION",
        **met_cal
    })
    resumen_global.append({
        "escenario": nombre_esc,
        "descripcion": params["descripcion"],
        "etapa": "VALIDACION",
        **met_val
    })

    for k, v in params.items():
        params_global.append({
            "escenario": nombre_esc,
            "descripcion": params["descripcion"],
            "parametro": k,
            "valor": v
        })

    df_long = df_full.copy()
    df_long["escenario"] = nombre_esc
    df_long["descripcion"] = params["descripcion"]
    series_all.append(df_long)


# ==========================================================
# 16) UNIFICAR RESULTADOS
# ==========================================================
df_resumen = pd.DataFrame(resumen_global)
df_params = pd.DataFrame(params_global)
df_series = pd.concat(series_all, ignore_index=True)

propiedades_modelo = pd.DataFrame([
    {"Propiedad": "VOL_MUERTO_m3", "Valor": VOL_MUERTO},
    {"Propiedad": "VOL_MAX_m3", "Valor": VOL_MAX},
    {"Propiedad": "AREA_CUENCA_TOTAL_m2", "Valor": AREA_CUENCA_TOTAL},
    {"Propiedad": "V_INICIAL_m3", "Valor": V_INICIAL},
    {"Propiedad": "CAL_START", "Valor": CAL_START},
    {"Propiedad": "CAL_END", "Valor": CAL_END},
    {"Propiedad": "VAL_START", "Valor": VAL_START},
    {"Propiedad": "VAL_END", "Valor": VAL_END},
    {"Propiedad": "SERIE_START", "Valor": SERIE_START},
    {"Propiedad": "SERIE_END", "Valor": SERIE_END},
    {"Propiedad": "EXC_START", "Valor": EXC_START},
    {"Propiedad": "EXC_END", "Valor": EXC_END},
])

# Resumen ancho por escenario
df_resumen_wide = (
    df_resumen
    .pivot_table(
        index=["escenario", "descripcion"],
        columns="etapa",
        values=["R2", "RMSE", "NSE", "KGE", "Bias", "PBIAS", "n"],
        aggfunc="first"
    )
)

df_resumen_wide.columns = [f"{a}_{b}" for a, b in df_resumen_wide.columns]
df_resumen_wide = df_resumen_wide.reset_index()

out_xlsx = os.path.join(OUT_DIR, "resultados_5_escenarios.xlsx")

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    df_resumen.to_excel(writer, sheet_name="metricas_largo", index=False)
    df_resumen_wide.to_excel(writer, sheet_name="metricas_resumen", index=False)
    df_params.to_excel(writer, sheet_name="parametros", index=False)
    propiedades_modelo.to_excel(writer, sheet_name="propiedades_modelo", index=False)
    df_series.to_excel(writer, sheet_name="series_completas", index=False)


# ==========================================================
# 17) GRÁFICA COMPARATIVA CONJUNTA
# ==========================================================
plt.figure(figsize=(18, 6))

# Observado único
df_obs = (
    df_series[["fecha", "V_emb_real"]]
    .drop_duplicates(subset=["fecha"])
    .sort_values("fecha")
    .reset_index(drop=True)
)

exc_s = pd.to_datetime(EXC_START)
exc_e = pd.to_datetime(EXC_END)
mask_exc = (df_obs["fecha"] >= exc_s) & (df_obs["fecha"] <= exc_e)
df_obs_plot = df_obs.copy()
df_obs_plot.loc[mask_exc, "V_emb_real"] = np.nan

plt.plot(df_obs_plot["fecha"], df_obs_plot["V_emb_real"], linewidth=2.2, label="Obs")

for nombre_esc in ESCENARIOS.keys():
    d = (
        df_series[df_series["escenario"] == nombre_esc]
        .sort_values("fecha")
        .reset_index(drop=True)
    )
    plt.plot(d["fecha"], d["V_emb"], linewidth=1.5, label=nombre_esc)

plt.axvline(pd.to_datetime(CAL_START), linestyle="--", linewidth=1.2)
plt.axvline(pd.to_datetime(CAL_END), linestyle="--", linewidth=1.2)
plt.axvline(pd.to_datetime(VAL_START), linestyle="--", linewidth=1.2)

plt.title(
    "Comparación de 5 escenarios físicamente representativos\n"
    "Volumen del embalse 2010–2019"
)
plt.xlabel("Fecha")
plt.ylabel("Volumen (m3)")
plt.legend(ncol=3)
plt.tight_layout()

out_png_comp = os.path.join(OUT_DIR, "comparacion_5_escenarios.png")
plt.savefig(out_png_comp, dpi=200, bbox_inches="tight")
plt.show()


# ==========================================================
# 18) TABLA FINAL EN CONSOLA
# ==========================================================
print("\n=== RESUMEN FINAL DE MÉTRICAS ===")
print(df_resumen_wide.to_string(index=False))

print("\n=== CORRESPONDENCIA DE ESCENARIOS ===")
for nombre_esc, params in ESCENARIOS.items():
    print(f"{nombre_esc} --> {params['descripcion']}")

print("\n=== ARCHIVOS GENERADOS ===")
print("Directorio principal :", OUT_DIR)
print("Excel resumen        :", out_xlsx)
print("Gráfico comparativo  :", out_png_comp)
print("CSV por escenario    : serie_ESC_*.csv")
print("PNG por escenario    : grafica_ESC_*.png")

### 5.1.1 · Variantes de configuración multi-escenario


In [ ]:
# ==========================================================
# SCRIPT 2 MODIFICADO
# SIMULACIÓN DISTRIBUIDA + EMBALSE
# MULTI-ESCENARIO (5 ESCENARIOS FÍSICAMENTE REPRESENTATIVOS)
# SERIE COMPLETA 2010-2019
# CALIBRACIÓN / VALIDACIÓN
# EXCLUYENDO UN PERIODO SOLO EN MÉTRICAS Y PLOT OBSERVADO
# + GUARDADO DE RESULTADOS
# + BANDA SOMBREADA ±15% DE LA SERIE SIMULADA
# + MÉTRICAS EN LA GRÁFICA
# + COMPARACIÓN ENTRE ESCENARIOS
# ==========================================================

# ==========================================================
# 0) CONFIGURACIÓN DE FECHAS
# ==========================================================
CAL_START = "2010-01-01"
CAL_END   = "2017-01-01"

VAL_START = "2017-01-02"
VAL_END   = "2019-12-31"

SERIE_START = "2010-01-01"
SERIE_END   = "2019-12-31"

EXC_START = "2014-01-01"
EXC_END   = "2015-06-07"


# ==========================================================
# 1) LIBRERÍAS
# ==========================================================
import os
import glob
import json
import calendar
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)


# ==========================================================
# 2) RUTAS / PARÁMETROS GENERALES
# ==========================================================
BASE_DIR = "/content/gdrive/MyDrive/Project001"

CHIRPS_DIR = os.path.join(BASE_DIR, "CHIRPS_PPT_DAILY_2009_2020_WGS84")
ET_DIR     = os.path.join(BASE_DIR, "ET_HARGREAVES_DAILY_2009_2020_WGS84")

LULC_DIR = os.path.join(BASE_DIR, "LULC_LC_Type1_WGS84")
SOIL_DIR = os.path.join(BASE_DIR, "SOIL_TEXTURE_b0_WGS84")

SHP_DIR = os.path.join(BASE_DIR, "shp")
AOI_CUENCA_PATH  = os.path.join(SHP_DIR, "Cuenca-es.geojson")
AOI_EMBALSE_PATH = os.path.join(SHP_DIR, "Embalse.geojson")

QM_MODEL_PATH = os.path.join(BASE_DIR, "QM_model_monthly.csv")
DESCARGAS_CSV = os.path.join(BASE_DIR, "descargas2.csv")

OUT_DIR = os.path.join(BASE_DIR, "resultados_modelo_5_escenarios")
os.makedirs(OUT_DIR, exist_ok=True)

START_YEAR = 2010
END_YEAR   = 2019

VOL_MUERTO = 41e6
VOL_MAX    = 148.80e6
AREA_CUENCA_TOTAL = 196.33e6
V_INICIAL = 57.80e6


# ==========================================================
# 3) ESCENARIOS SELECCIONADOS
# ==========================================================
ESCENARIOS = {
    "ESC_01_BALANCED": {
        "descripcion": "GEN 5 - IND 37: Equilibrio Físico Global (El Santo Grial)",
        "g7": 2, "g9": 1, "n": 2, "K": 3.57679534,
        "k_infil": 0.695388072, "k_et": 0.898202363, "Vs": 0.004280656,
        "CN10_g7": 95.99353731, "CN16_g7": 84.92690705,
        "CN10_g9": 93.85918907, "CN16_g9": 47.79133283,
        "k_effP": 1.496379124, "k_q": 2.138215482, "lambda_ia": 0.091920364,
    },
    "ESC_02_THIN_SOIL": {
        "descripcion": "GEN 1 - IND 13: Respuesta Rápida (Suelos Delgados/Roca)",
        "g7": 1, "g9": 1, "n": 3, "K": 7.274480151,
        "k_infil": 0.544780972, "k_et": 0.926009419, "Vs": 0.009274802,
        "CN10_g7": 88.39049765, "CN16_g7": 79.7635199,
        "CN10_g9": 97.38171396, "CN16_g9": 73.37569833,
        "k_effP": 1.294433958, "k_q": 2.349457801, "lambda_ia": 0.076576546,
    },
    "ESC_03_VOLUME_ACC": {
        "descripcion": "GEN 2 - IND 14: Máxima Precisión de Volumen (Masa)",
        "g7": 1, "g9": 1, "n": 2, "K": 3.57679534,
        "k_infil": 0.695388072, "k_et": 0.771095249, "Vs": 0.004280656,
        "CN10_g7": 94.23137497, "CN16_g7": 87.17258884,
        "CN10_g9": 92.39351112, "CN16_g9": 69.33066334,
        "k_effP": 1.786451523, "k_q": 2.138215482, "lambda_ia": 0.143691826,
    },
    "ESC_04_DRY_PUNA": {
        "descripcion": "GEN 1 - IND 5: Condición de Retención (Suelo Seco/Puna)",
        "g7": 2, "g9": 1, "n": 3, "K": 2.025448874,
        "k_infil": 0.695388072, "k_et": 0.887936542, "Vs": 0.004272988,
        "CN10_g7": 95.03184225, "CN16_g7": 89.17929878,
        "CN10_g9": 92.1286713, "CN16_g9": 69.33066334,
        "k_effP": 1.549640725, "k_q": 2.138215482, "lambda_ia": 0.143691826,
    },
    "ESC_05_REGIONAL": {
        "descripcion": "GEN 3 - IND 27: Ajuste Regional (Valores Tesis Chili/Pañe)",
        "g7": 1, "g9": 1, "n": 2, "K": 3.57679534,
        "k_infil": 0.240352353, "k_et": 0.949124867, "Vs": 0.007179887,
        "CN10_g7": 90.84810541, "CN16_g7": 98.0,
        "CN10_g9": 93.16244962, "CN16_g9": 47.79133283,
        "k_effP": 1.496379124, "k_q": 2.472455086, "lambda_ia": 0.091920364,
    },
}

# ==========================================================
# 4) UTILIDADES GENERALES
# ==========================================================
def load_geoms(geojson_path: str):
    if not os.path.exists(geojson_path):
        raise FileNotFoundError(f"No existe AOI: {geojson_path}")
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        raise ValueError(f"AOI vacío: {geojson_path}")
    geom = gdf.geometry.union_all()
    return [geom]


def read_clip_stack(tif_path: str, geoms):
    if not os.path.exists(tif_path):
        raise FileNotFoundError(f"No existe raster: {tif_path}")
    with rasterio.open(tif_path) as src:
        out_img, _ = mask(src, geoms, crop=True, filled=True)
        nodata = src.nodata
    return out_img, nodata


def nanmean_masked(a: np.ndarray, nodata=None) -> float:
    a = a.astype("float64", copy=False)
    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)
    return float(np.nanmean(a))


def find_month_file(folder: str, prefix: str, year: int, month: int) -> str:
    pattern = os.path.join(folder, f"{prefix}_{year}_{month:02d}_WGS84*.*")
    hits = sorted(glob.glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No encontré archivo con patrón:\n  {pattern}")
    return hits[0]


def find_single_tif(folder: str) -> str:
    hits = sorted(glob.glob(os.path.join(folder, "*.tif"))) + sorted(glob.glob(os.path.join(folder, "*.tiff")))
    if not hits:
        raise FileNotFoundError(f"No encontré tif/tiff en: {folder}")
    return hits[0]


# ==========================================================
# 5) QM MENSUAL
# ==========================================================
def load_qm_table(csv_path: str):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No existe QM CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"mes", "x", "y"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"QM_model_monthly.csv debe tener columnas {required}. Tiene: {list(df.columns)}")

    qm = {}
    for _, r in df.iterrows():
        mes = int(r["mes"])
        x = json.loads(r["x"]) if isinstance(r["x"], str) else r["x"]
        y = json.loads(r["y"]) if isinstance(r["y"], str) else r["y"]

        x = np.array(x, dtype=float)
        y = np.array(y, dtype=float)

        if x.size < 2 or y.size < 2 or x.size != y.size:
            raise ValueError(f"QM inválido en mes {mes}: x={x.size}, y={y.size}")

        qm[mes] = (x, y)

    faltantes = [m for m in range(1, 13) if m not in qm]
    if faltantes:
        raise ValueError(f"Faltan meses en QM: {faltantes}")

    return qm


def qm_correct_array(arr: np.ndarray, month: int, QM_TABLE):
    x, y = QM_TABLE[month]
    idx = np.argsort(x)
    x2 = x[idx]
    y2 = y[idx]
    flat = arr.reshape(-1).astype("float64", copy=False)
    out = np.interp(flat, x2, y2).reshape(arr.shape)
    return np.maximum(out, 0.0)


# ==========================================================
# 6) SCS-CN DISTRIBUIDO
# ==========================================================
CN_TABLE = {
    1: {1:35, 2:25, 3:45, 4:39, 5:45, 6:49, 7:68, 8:36, 9:45, 10:30, 11:95, 12:67, 13:72, 14:63, 15:100, 16:74, 17:100},
    2: {1:50, 2:55, 3:66, 4:61, 5:66, 6:69, 7:79, 8:60, 9:66, 10:58, 11:95, 12:78, 13:82, 14:75, 15:100, 16:84, 17:100},
    3: {1:73, 2:70, 3:77, 4:74, 5:77, 6:79, 7:86, 8:73, 9:77, 10:71, 11:95, 12:85, 13:87, 14:83, 15:100, 16:90, 17:100},
    4: {1:79, 2:77, 3:83, 4:80, 5:83, 6:89, 7:89, 8:79, 9:83, 10:78, 11:95, 12:89, 13:89, 14:87, 15:100, 16:92, 17:100},
}


def soil_group_from_texture(soil_class, g7=2, g9=2):
    soil_grp = np.zeros_like(soil_class, dtype="int16")
    soil_grp = np.where(soil_class > 10, 1, soil_grp)
    soil_grp = np.where((soil_class > 4) & (soil_class <= 10), 2, soil_grp)
    soil_grp = np.where((soil_class > 1) & (soil_class <= 4), 3, soil_grp)
    soil_grp = np.where((soil_class > 0) & (soil_class <= 1), 4, soil_grp)

    soil_grp = np.where(soil_class == 7, int(g7), soil_grp)
    soil_grp = np.where(soil_class == 9, int(g9), soil_grp)
    return soil_grp


def reemplazar_cn_lulc10_16(CN10_g7, CN16_g7, CN10_g9, CN16_g9, g7=2, g9=2):
    cn_table_local = {k: v.copy() for k, v in CN_TABLE.items()}
    cn_table_local[int(g7)][10] = float(CN10_g7)
    cn_table_local[int(g7)][16] = float(CN16_g7)
    cn_table_local[int(g9)][10] = float(CN10_g9)
    cn_table_local[int(g9)][16] = float(CN16_g9)
    return cn_table_local


def build_cn_s_from_cache(LULC_CUENCA, SOIL_CUENCA, cn_table_local, g7=2, g9=2, f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4):
    lulc = LULC_CUENCA.astype("float64")
    soil = SOIL_CUENCA.astype("float64")
    soil_grp = soil_group_from_texture(soil, g7=g7, g9=g9)

    CN2 = np.zeros_like(lulc, dtype="float64")
    CN2 = np.where(soil_grp == 0, 100.0, CN2)

    for grp, lut in cn_table_local.items():
        for lc, cn in lut.items():
            CN2 = np.where((soil_grp == grp) & (lulc == lc), float(cn), CN2)

    CN2 = np.minimum(CN2 * float(f_cn2), 100.0)

    CN1 = CN2 / (2.281 - (CN2 * 0.0128))
    CN3 = CN2 / (0.427 + (CN2 * 0.00573))

    S1 = (25400.0 / CN1 - 254.0) * float(f_s1)
    S2 = (25400.0 / CN2 - 254.0) * float(f_s2)
    S3 = (25400.0 / CN3 - 254.0) * float(f_s3)

    return S1, S2, S3


def rolling_sum_5d(stack_TRC: np.ndarray, prev4=None):
    if prev4 is None:
        prev4 = np.zeros((0,) + stack_TRC.shape[1:], dtype=stack_TRC.dtype)

    combo = np.concatenate([prev4, stack_TRC], axis=0)
    cs = np.cumsum(combo, axis=0)

    amc = np.empty_like(combo, dtype="float64")
    for i in range(combo.shape[0]):
        if i < 4:
            amc[i] = np.sum(combo[:i+1], axis=0)
        else:
            amc[i] = cs[i] - cs[i-5]

    tail4 = combo[-4:] if combo.shape[0] >= 4 else combo
    return amc[-stack_TRC.shape[0]:], tail4


def runoff_scs(ppt, amc5, S1, S2, S3, lambda_ia=0.2):
    S = np.where(amc5 <= 13, S1, S2)
    S = np.where(amc5 > 28, S3, S)

    lam = float(np.clip(lambda_ia, 0.001, 0.8))
    Ia = lam * S

    numer = np.power(ppt - Ia, 2)
    denom = (ppt - Ia) + S

    with np.errstate(divide="ignore", invalid="ignore"):
        Q = np.where(ppt < Ia, 0.0, numer / denom)

    return np.where(np.isfinite(Q), Q, 0.0)


# ==========================================================
# 7) RUTEO
# ==========================================================
def route_linear_reservoir(runoff_mm, K=2.0):
    r = np.asarray(runoff_mm, dtype=float)
    q = np.zeros_like(r)
    alpha = np.exp(-1.0 / max(K, 1e-6))

    for t in range(len(r)):
        q[t] = (1 - alpha) * r[t] if t == 0 else alpha * q[t-1] + (1 - alpha) * r[t]

    return q


def route_nash_cascade(runoff_mm, n=3, K=2.0):
    q = np.asarray(runoff_mm, dtype=float)
    for _ in range(int(n)):
        q = route_linear_reservoir(q, K=K)
    return q


# ==========================================================
# 8) EMBALSE
# ==========================================================
def area_embalse(V):
    D366 = V / 1e6
    area_km2 = (
        8.91946288801159E-08 * D366**4
        - 0.0000346583770904819 * D366**3
        + 0.003919249094071 * D366**2
        - 0.0306477700093546 * D366
        + 2.02308232795992
    )
    return max(area_km2, 0) * 1e6


def volumen_auxiliar(V_prev, esc_mm, A_c, Vd):
    A_emb = area_embalse(V_prev)
    A_aporte = max(A_c - A_emb, 0)
    esc_m = esc_mm / 1000.0
    V_esc = esc_m * A_aporte
    return V_prev + V_esc - Vd


def area_media(V_prev, V_aux):
    return 0.5 * (area_embalse(V_prev) + area_embalse(V_aux))


def aplicar_restricciones(B_i):
    if B_i > VOL_MAX:
        return VOL_MAX, B_i - VOL_MAX
    if B_i < VOL_MUERTO:
        return VOL_MUERTO, 0.0
    return B_i, 0.0


def balance_embalse_diario(V_prev, esc_mm, precip_mm, et_mm, A_c, Vd, Vs_m_d, k_infil=1.0, k_et=1.0):
    V_aux = volumen_auxiliar(V_prev, esc_mm, A_c, Vd)
    A_i = area_media(V_prev, V_aux)

    esc_m = esc_mm / 1000.0
    precip_m = precip_mm / 1000.0
    et_m = (et_mm * k_et) / 1000.0

    V_esc = esc_m * max(A_c - A_i, 0.0)
    V_clima = (precip_m - et_m) * A_i
    Vs_vol = (float(Vs_m_d) * float(k_infil)) * A_i

    B_i = V_prev + V_esc + V_clima - Vd - Vs_vol
    V_emb, V_vertido = aplicar_restricciones(B_i)

    return V_emb, V_vertido, A_i, V_clima, V_esc, Vs_vol, B_i, Vd


# ==========================================================
# 9) CARGA DE DATOS FIJOS
# ==========================================================
roi_cuenca_geoms  = load_geoms(AOI_CUENCA_PATH)
roi_embalse_geoms = load_geoms(AOI_EMBALSE_PATH)

QM_TABLE = load_qm_table(QM_MODEL_PATH)

LULC_TIF = find_single_tif(LULC_DIR)
SOIL_TIF = find_single_tif(SOIL_DIR)

Vdescarga = pd.read_csv(DESCARGAS_CSV, sep=";", encoding="latin-1")
Vdescarga = Vdescarga.rename(columns={
    Vdescarga.columns[0]: "fecha",
    Vdescarga.columns[1]: "descarga_m3_s",
    Vdescarga.columns[2]: "V_emb_real_hm3",
})

Vdescarga["fecha"] = pd.to_datetime(Vdescarga["fecha"], dayfirst=True, errors="coerce").dt.floor("D")
Vdescarga["descarga_m3_s"] = pd.to_numeric(Vdescarga["descarga_m3_s"], errors="coerce")
Vdescarga["V_emb_real_hm3"] = pd.to_numeric(Vdescarga["V_emb_real_hm3"], errors="coerce")

Vdescarga = Vdescarga[
    (Vdescarga["fecha"] >= SERIE_START) &
    (Vdescarga["fecha"] <= SERIE_END)
].dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)

Vdescarga = Vdescarga.drop_duplicates(subset=["fecha"], keep="last")


# ==========================================================
# 10) CACHE RASTERS ESTÁTICOS
# ==========================================================
lulc_stack, _ = read_clip_stack(LULC_TIF, roi_cuenca_geoms)
soil_stack, _ = read_clip_stack(SOIL_TIF, roi_cuenca_geoms)

LULC_CUENCA = lulc_stack[0].astype("int16")
SOIL_CUENCA = soil_stack[0].astype("int16")

print("LULC únicos (cuenca):", np.unique(LULC_CUENCA))
print("SOIL únicos (cuenca):", np.unique(SOIL_CUENCA))


# ==========================================================
# 11) MÉTRICAS
# ==========================================================
def _remove_excluded_period(df):
    d = df.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.floor("D")

    exc_s = pd.to_datetime(EXC_START)
    exc_e = pd.to_datetime(EXC_END)

    return d[~((d["fecha"] >= exc_s) & (d["fecha"] <= exc_e))].reset_index(drop=True)


def _compute_metrics(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    m = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[m]
    obs = obs[m]

    if len(obs) < 10:
        return {
            "n": int(len(obs)),
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan
        }

    if np.std(obs) > 0 and np.std(sim) > 0:
        r = float(np.corrcoef(obs, sim)[0, 1])
        R2 = float(r ** 2)
    else:
        r = 0.0
        R2 = 0.0

    RMSE = float(np.sqrt(np.mean((sim - obs) ** 2)))
    denom = float(np.sum((obs - np.mean(obs)) ** 2))
    NSE = float(1 - np.sum((sim - obs) ** 2) / denom) if denom > 0 else np.nan

    sigma = float(np.std(obs))
    RMSE_n = float(RMSE / sigma) if sigma > 0 else np.nan

    mu_o = float(np.mean(obs))
    mu_s = float(np.mean(sim))
    Bias = float((mu_s - mu_o) / mu_o) if abs(mu_o) > 1e-12 else np.nan
    PBIAS = float(Bias * 100.0) if np.isfinite(Bias) else np.nan

    alpha = float(np.std(sim) / np.std(obs)) if np.std(obs) > 0 else np.nan
    beta = float(mu_s / mu_o) if abs(mu_o) > 1e-12 else np.nan

    if np.isfinite(r) and np.isfinite(alpha) and np.isfinite(beta):
        KGE = float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))
    else:
        KGE = np.nan

    return {
        "n": int(len(obs)),
        "R2": R2,
        "RMSE": RMSE,
        "NSE": NSE,
        "RMSE_n": RMSE_n,
        "Bias": Bias,
        "PBIAS": PBIAS,
        "KGE": KGE
    }


# ==========================================================
# 12) FILTROS DE FECHAS
# ==========================================================
def _slice_df_by_dates(df_res, start, end):
    s = pd.to_datetime(start)
    e = pd.to_datetime(end)

    d = df_res.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.floor("D")

    return d[(d["fecha"] >= s) & (d["fecha"] <= e)].reset_index(drop=True)


def _build_calibration_df(df_res, cal_start, cal_end):
    d = _slice_df_by_dates(df_res, cal_start, cal_end)
    d = _remove_excluded_period(d)
    return d


def _build_validation_df(df_res, val_start, val_end):
    d = _slice_df_by_dates(df_res, val_start, val_end)
    d = _remove_excluded_period(d)
    return d


# ==========================================================
# 13) SIMULACIÓN COMPLETA CON SALIDAS DETALLADAS
# ==========================================================
def simular_detallado(params, metric_start=None, metric_end=None):
    cn_table_local = reemplazar_cn_lulc10_16(
        params["CN10_g7"], params["CN16_g7"],
        params["CN10_g9"], params["CN16_g9"],
        g7=params["g7"], g9=params["g9"]
    )

    S1_CU, S2_CU, S3_CU = build_cn_s_from_cache(
        LULC_CUENCA, SOIL_CUENCA,
        cn_table_local=cn_table_local,
        g7=params["g7"], g9=params["g9"],
        f_cn2=1.0,
        f_s1=1.3, f_s2=1.2, f_s3=1.4
    )

    rows = []
    for year in range(START_YEAR, END_YEAR + 1):
        prev4 = None
        for month in range(1, 13):
            chirps_path = find_month_file(CHIRPS_DIR, "CHIRPS_PPT_DAILY", year, month)
            et_path     = find_month_file(ET_DIR, "ET_HARGREAVES_DAILY", year, month)

            ch_cu_stack, _ = read_clip_stack(chirps_path, roi_cuenca_geoms)
            ppt_cu = qm_correct_array(ch_cu_stack.astype("float64"), month, QM_TABLE)
            ppt_cu_eff = ppt_cu * float(params["k_effP"])

            amc5, prev4 = rolling_sum_5d(ppt_cu_eff, prev4=prev4)
            q_cu = runoff_scs(
                ppt_cu_eff, amc5, S1_CU, S2_CU, S3_CU,
                lambda_ia=float(params["lambda_ia"])
            )

            q_mean = [nanmean_masked(q_cu[i], None) for i in range(q_cu.shape[0])]
            q_mean = [v * float(params["k_q"]) for v in q_mean]

            ppt_cu_mean = [nanmean_masked(ppt_cu_eff[i], None) for i in range(ppt_cu_eff.shape[0])]

            ch_em_stack, ch_em_nodata = read_clip_stack(chirps_path, roi_embalse_geoms)
            ppt_em = qm_correct_array(ch_em_stack.astype("float64"), month, QM_TABLE)
            ppt_em_mean = [nanmean_masked(ppt_em[i], ch_em_nodata) for i in range(ppt_em.shape[0])]
            ppt_em_mean = [v * float(params["k_effP"]) for v in ppt_em_mean]

            et_em_stack, et_em_nodata = read_clip_stack(et_path, roi_embalse_geoms)
            et_em = et_em_stack.astype("float64")
            et_em_mean = [nanmean_masked(et_em[i], et_em_nodata) for i in range(et_em.shape[0])]

            ndays = calendar.monthrange(year, month)[1]
            T = min(ndays, len(q_mean), len(ppt_em_mean), len(et_em_mean), len(ppt_cu_mean))

            for d in range(T):
                rows.append({
                    "fecha": datetime(year, month, d + 1),
                    "ppt_cuenca_eff_mm": float(ppt_cu_mean[d]),
                    "esc_mm": float(q_mean[d]),
                    "precip_mm": float(ppt_em_mean[d]),
                    "et_mm": float(et_em_mean[d]),
                })

    df_forz = pd.DataFrame(rows).sort_values("fecha").reset_index(drop=True)
    df_forz["esc_ruteada_mm"] = route_nash_cascade(
        df_forz["esc_mm"].to_numpy(),
        n=int(params["n"]),
        K=float(params["K"])
    )

    df_forz["fecha"] = pd.to_datetime(df_forz["fecha"]).dt.floor("D")
    df_forz = df_forz.drop_duplicates(subset=["fecha"], keep="last")

    df_join = pd.merge(
        df_forz[["fecha", "ppt_cuenca_eff_mm", "esc_mm", "esc_ruteada_mm", "precip_mm", "et_mm"]],
        Vdescarga[["fecha", "descarga_m3_s", "V_emb_real_hm3"]],
        on="fecha",
        how="inner"
    ).sort_values("fecha").reset_index(drop=True)

    if len(df_join) < 100:
        df_res_full = pd.DataFrame({"fecha": [], "V_emb": [], "V_emb_real": []})
        met = {
            "R2": np.nan, "RMSE": np.nan, "NSE": np.nan,
            "RMSE_n": np.nan, "Bias": np.nan, "PBIAS": np.nan, "KGE": np.nan, "n": 0
        }
        return df_res_full, met

    resultados = []
    V_emb = None

    for i in range(len(df_join)):
        V_prev = V_INICIAL if i == 0 else V_emb

        fecha = df_join.loc[i, "fecha"]
        Vd = float(df_join.loc[i, "descarga_m3_s"]) * 86400.0
        V_obs = float(df_join.loc[i, "V_emb_real_hm3"]) * 1_000_000.0

        esc_mm = float(df_join.loc[i, "esc_ruteada_mm"])
        p_mm   = float(df_join.loc[i, "precip_mm"])
        et_mm  = float(df_join.loc[i, "et_mm"])

        V_emb, V_vert, A_i, V_clima, V_esc, Vs_vol, B_i, Vd = balance_embalse_diario(
            V_prev, esc_mm, p_mm, et_mm,
            AREA_CUENCA_TOTAL, Vd,
            Vs_m_d=float(params["Vs"]),
            k_infil=float(params["k_infil"]),
            k_et=float(params["k_et"])
        )

        resultados.append({
            "fecha": fecha,
            "ppt_cuenca_eff_mm": float(df_join.loc[i, "ppt_cuenca_eff_mm"]),
            "esc_mm_sin_ruteo": float(df_join.loc[i, "esc_mm"]),
            "esc_mm_ruteada": float(df_join.loc[i, "esc_ruteada_mm"]),
            "precip_mm_embalse": p_mm,
            "et_mm_embalse": et_mm,
            "descarga_m3_s": float(df_join.loc[i, "descarga_m3_s"]),
            "descarga_m3_dia": Vd,
            "V_emb": V_emb,
            "V_emb_hm3": V_emb / 1e6,
            "V_emb_real": V_obs,
            "V_emb_real_hm3": V_obs / 1e6,
            "V_vertido": V_vert,
            "area_embalse_m2": A_i,
            "V_clima": V_clima,
            "V_esc": V_esc,
            "Vs_vol": Vs_vol,
            "balance_bruto": B_i
        })

    df_res_full = pd.DataFrame(resultados)
    df_res_full["fecha"] = pd.to_datetime(df_res_full["fecha"]).dt.floor("D")

    if metric_start is None or metric_end is None:
        dmet = df_res_full.copy()
    else:
        dmet = _slice_df_by_dates(df_res_full, metric_start, metric_end)

    dmet = _remove_excluded_period(dmet)
    met = _compute_metrics(dmet["V_emb"].values, dmet["V_emb_real"].values)

    return df_res_full, met


# ==========================================================
# 14) GRAFICADO DE UN ESCENARIO
# ==========================================================
def graficar_escenario(df_full, met_cal, met_val, nombre_esc, descripcion, out_png):
    plt.figure(figsize=(16, 5))

    df_obs_plot = df_full.copy()
    exc_s = pd.to_datetime(EXC_START)
    exc_e = pd.to_datetime(EXC_END)
    mask_exc = (df_obs_plot["fecha"] >= exc_s) & (df_obs_plot["fecha"] <= exc_e)
    df_obs_plot.loc[mask_exc, "V_emb_real"] = np.nan

    sim = df_full["V_emb"].to_numpy(dtype=float)
    sim_low = sim * 0.85
    sim_high = sim * 1.15

    plt.fill_between(
        df_full["fecha"],
        sim_low,
        sim_high,
        alpha=0.18,
        label="Banda ±15% Sim"
    )

    plt.plot(df_obs_plot["fecha"], df_obs_plot["V_emb_real"], linewidth=1.5, label="Obs")
    plt.plot(df_full["fecha"], df_full["V_emb"], linewidth=1.8, label="Sim")

    plt.axvline(pd.to_datetime(CAL_START), linestyle="--", linewidth=1.2)
    plt.axvline(pd.to_datetime(CAL_END), linestyle="--", linewidth=1.2)
    plt.axvline(pd.to_datetime(VAL_START), linestyle="--", linewidth=1.2)

    texto_cal = (
        "CALIBRACIÓN\n"
        f"R²={met_cal['R2']:.3f}\n"
        f"RMSE={met_cal['RMSE']:.2e}\n"
        f"NSE={met_cal['NSE']:.3f}\n"
        f"KGE={met_cal['KGE']:.3f}\n"
        f"Bias={met_cal['Bias']:.4f}"
    )

    texto_val = (
        "VALIDACIÓN\n"
        f"R²={met_val['R2']:.3f}\n"
        f"RMSE={met_val['RMSE']:.2e}\n"
        f"NSE={met_val['NSE']:.3f}\n"
        f"KGE={met_val['KGE']:.3f}\n"
        f"Bias={met_val['Bias']:.4f}"
    )

    plt.text(
        0.015, 0.97, texto_cal,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.text(
        0.165, 0.97, texto_val,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.title(
        f"Volumen embalse 2010–2019 | {nombre_esc}\n"
        f"{descripcion}\n"
        f"Calibración: {CAL_START} a {CAL_END} | Validación: {VAL_START} a {VAL_END}"
    )
    plt.xlabel("Fecha")
    plt.ylabel("Volumen (m3)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.show()


# ==========================================================
# 15) EJECUCIÓN MULTI-ESCENARIO
# ==========================================================
print("=== MODELO MULTI-ESCENARIO ===")

resumen_global = []
params_global = []
series_all = []

for nombre_esc, params in ESCENARIOS.items():
    print("\n" + "=" * 80)
    print(f"Ejecutando {nombre_esc}")
    print(params["descripcion"])
    print("=" * 80)
    print(params)

    df_full, _ = simular_detallado(
        params,
        metric_start=CAL_START,
        metric_end=CAL_END
    )

    df_full["fecha"] = pd.to_datetime(df_full["fecha"]).dt.floor("D")

    df_cal = _build_calibration_df(df_full, CAL_START, CAL_END)
    df_val = _build_validation_df(df_full, VAL_START, VAL_END)

    met_cal = _compute_metrics(df_cal["V_emb"], df_cal["V_emb_real"])
    met_val = _compute_metrics(df_val["V_emb"], df_val["V_emb_real"])

    print("\n--- MÉTRICAS CALIBRACIÓN ---")
    print(met_cal)
    print("--- MÉTRICAS VALIDACIÓN ---")
    print(met_val)

    # Guardar serie individual CSV
    out_csv = os.path.join(OUT_DIR, f"serie_{nombre_esc}.csv")
    df_out = df_full.copy()
    df_out["escenario"] = nombre_esc
    df_out["descripcion"] = params["descripcion"]
    df_out.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # Gráfico individual
    out_png = os.path.join(OUT_DIR, f"grafica_{nombre_esc}.png")
    graficar_escenario(df_full, met_cal, met_val, nombre_esc, params["descripcion"], out_png)

    # Acumulados resumen
    resumen_global.append({
        "escenario": nombre_esc,
        "descripcion": params["descripcion"],
        "etapa": "CALIBRACION",
        **met_cal
    })
    resumen_global.append({
        "escenario": nombre_esc,
        "descripcion": params["descripcion"],
        "etapa": "VALIDACION",
        **met_val
    })

    for k, v in params.items():
        params_global.append({
            "escenario": nombre_esc,
            "descripcion": params["descripcion"],
            "parametro": k,
            "valor": v
        })

    df_long = df_full.copy()
    df_long["escenario"] = nombre_esc
    df_long["descripcion"] = params["descripcion"]
    series_all.append(df_long)


# ==========================================================
# 16) UNIFICAR RESULTADOS
# ==========================================================
df_resumen = pd.DataFrame(resumen_global)
df_params = pd.DataFrame(params_global)
df_series = pd.concat(series_all, ignore_index=True)

propiedades_modelo = pd.DataFrame([
    {"Propiedad": "VOL_MUERTO_m3", "Valor": VOL_MUERTO},
    {"Propiedad": "VOL_MAX_m3", "Valor": VOL_MAX},
    {"Propiedad": "AREA_CUENCA_TOTAL_m2", "Valor": AREA_CUENCA_TOTAL},
    {"Propiedad": "V_INICIAL_m3", "Valor": V_INICIAL},
    {"Propiedad": "CAL_START", "Valor": CAL_START},
    {"Propiedad": "CAL_END", "Valor": CAL_END},
    {"Propiedad": "VAL_START", "Valor": VAL_START},
    {"Propiedad": "VAL_END", "Valor": VAL_END},
    {"Propiedad": "SERIE_START", "Valor": SERIE_START},
    {"Propiedad": "SERIE_END", "Valor": SERIE_END},
    {"Propiedad": "EXC_START", "Valor": EXC_START},
    {"Propiedad": "EXC_END", "Valor": EXC_END},
])

# Resumen ancho por escenario
df_resumen_wide = (
    df_resumen
    .pivot_table(
        index=["escenario", "descripcion"],
        columns="etapa",
        values=["R2", "RMSE", "NSE", "KGE", "Bias", "PBIAS", "n"],
        aggfunc="first"
    )
)

df_resumen_wide.columns = [f"{a}_{b}" for a, b in df_resumen_wide.columns]
df_resumen_wide = df_resumen_wide.reset_index()

out_xlsx = os.path.join(OUT_DIR, "resultados_5_escenarios.xlsx")

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    df_resumen.to_excel(writer, sheet_name="metricas_largo", index=False)
    df_resumen_wide.to_excel(writer, sheet_name="metricas_resumen", index=False)
    df_params.to_excel(writer, sheet_name="parametros", index=False)
    propiedades_modelo.to_excel(writer, sheet_name="propiedades_modelo", index=False)
    df_series.to_excel(writer, sheet_name="series_completas", index=False)


# ==========================================================
# 17) GRÁFICA COMPARATIVA CONJUNTA
# ==========================================================
plt.figure(figsize=(18, 6))

# Observado único
df_obs = (
    df_series[["fecha", "V_emb_real"]]
    .drop_duplicates(subset=["fecha"])
    .sort_values("fecha")
    .reset_index(drop=True)
)

exc_s = pd.to_datetime(EXC_START)
exc_e = pd.to_datetime(EXC_END)
mask_exc = (df_obs["fecha"] >= exc_s) & (df_obs["fecha"] <= exc_e)
df_obs_plot = df_obs.copy()
df_obs_plot.loc[mask_exc, "V_emb_real"] = np.nan

plt.plot(df_obs_plot["fecha"], df_obs_plot["V_emb_real"], linewidth=2.2, label="Obs")

for nombre_esc in ESCENARIOS.keys():
    d = (
        df_series[df_series["escenario"] == nombre_esc]
        .sort_values("fecha")
        .reset_index(drop=True)
    )
    plt.plot(d["fecha"], d["V_emb"], linewidth=1.5, label=nombre_esc)

plt.axvline(pd.to_datetime(CAL_START), linestyle="--", linewidth=1.2)
plt.axvline(pd.to_datetime(CAL_END), linestyle="--", linewidth=1.2)
plt.axvline(pd.to_datetime(VAL_START), linestyle="--", linewidth=1.2)

plt.title(
    "Comparación de 5 escenarios físicamente representativos\n"
    "Volumen del embalse 2010–2019"
)
plt.xlabel("Fecha")
plt.ylabel("Volumen (m3)")
plt.legend(ncol=3)
plt.tight_layout()

out_png_comp = os.path.join(OUT_DIR, "comparacion_5_escenarios.png")
plt.savefig(out_png_comp, dpi=200, bbox_inches="tight")
plt.show()


# ==========================================================
# 18) TABLA FINAL EN CONSOLA
# ==========================================================
print("\n=== RESUMEN FINAL DE MÉTRICAS ===")
print(df_resumen_wide.to_string(index=False))

print("\n=== CORRESPONDENCIA DE ESCENARIOS ===")
for nombre_esc, params in ESCENARIOS.items():
    print(f"{nombre_esc} --> {params['descripcion']}")

print("\n=== ARCHIVOS GENERADOS ===")
print("Directorio principal :", OUT_DIR)
print("Excel resumen        :", out_xlsx)
print("Gráfico comparativo  :", out_png_comp)
print("CSV por escenario    : serie_ESC_*.csv")
print("PNG por escenario    : grafica_ESC_*.png")

In [ ]:
# ==========================================================
# SCRIPT 2 MODIFICADO
# SIMULACIÓN DISTRIBUIDA + EMBALSE
# MULTI-ESCENARIO (5 ESCENARIOS FÍSICAMENTE REPRESENTATIVOS)
# SERIE COMPLETA 2010-2019
# CALIBRACIÓN / VALIDACIÓN
# EXCLUYENDO UN PERIODO SOLO EN MÉTRICAS Y PLOT OBSERVADO
# + GUARDADO DE RESULTADOS
# + BANDA SOMBREADA ±15% DE LA SERIE SIMULADA
# + MÉTRICAS EN LA GRÁFICA
# + COMPARACIÓN ENTRE ESCENARIOS
# ==========================================================

# ==========================================================
# 0) CONFIGURACIÓN DE FECHAS
# ==========================================================
CAL_START = "2010-01-01"
CAL_END   = "2017-01-01"

VAL_START = "2017-01-02"
VAL_END   = "2019-12-31"

SERIE_START = "2010-01-01"
SERIE_END   = "2019-12-31"

EXC_START = "2014-01-01"
EXC_END   = "2015-06-07"


# ==========================================================
# 1) LIBRERÍAS
# ==========================================================
import os
import glob
import json
import calendar
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)


# ==========================================================
# 2) RUTAS / PARÁMETROS GENERALES
# ==========================================================
BASE_DIR = "/content/drive/MyDrive/Project001"

CHIRPS_DIR = os.path.join(BASE_DIR, "CHIRPS_PPT_DAILY_2009_2020_WGS84")
ET_DIR     = os.path.join(BASE_DIR, "ET_HARGREAVES_DAILY_2009_2020_WGS84")

LULC_DIR = os.path.join(BASE_DIR, "LULC_LC_Type1_WGS84")
SOIL_DIR = os.path.join(BASE_DIR, "SOIL_TEXTURE_b0_WGS84")

SHP_DIR = os.path.join(BASE_DIR, "shp")
AOI_CUENCA_PATH  = os.path.join(SHP_DIR, "Cuenca-es.geojson")
AOI_EMBALSE_PATH = os.path.join(SHP_DIR, "Embalse.geojson")

QM_MODEL_PATH = os.path.join(BASE_DIR, "QM_model_monthly.csv")
DESCARGAS_CSV = os.path.join(BASE_DIR, "descargas2.csv")

OUT_DIR = os.path.join(BASE_DIR, "resultados_modelo_5_escenarios")
os.makedirs(OUT_DIR, exist_ok=True)

START_YEAR = 2010
END_YEAR   = 2019

VOL_MUERTO = 41e6
VOL_MAX    = 148.80e6
AREA_CUENCA_TOTAL = 196.33e6
V_INICIAL = 57.80e6


# ==========================================================
# 3) ESCENARIOS SELECCIONADOS
# ==========================================================
# ==========================================================
# ESCENARIOS SELECCIONADOS — CUENCA EL PAÑE
# Basado en análisis de representatividad física del archivo
# de calibración genética (metricas.txt)
# ==========================================================

ESCENARIOS = {

    # ──────────────────────────────────────────────────────
    # ESC_01 | GEN 6/20 – IND 27/42
    # Proceso dominante: K lento + k_ET más alto del grupo
    # → Almacenamiento profundo (acuífero volcánico / glaciar
    #   de escombros). NSE=0.906 | KGE=0.935 | fit=0.5100
    # ──────────────────────────────────────────────────────
    "ESC_01_G6_I27": {
        "descripcion": "GEN 6/20 - IND 27/42",
        "g7": 2,
        "g9": 1,
        "n": 3,
        "K": 5.623508257867263,
        "k_infil": 0.8466222840708213,
        "k_et": 0.9534142233928005,
        "Vs": 0.009475350265408508,
        "CN10_g7": 95.6255497746404,
        "CN16_g7": 93.24570404930925,
        "CN10_g9": 97.25874176007405,
        "CN16_g9": 63.87935857626448,
        "k_effP": 1.861843159699172,
        "k_q": 1.81601367210348,
        "lambda_ia": 0.04260303328216147,
    },

    # ──────────────────────────────────────────────────────
    # ESC_02 | GEN 7/20 – IND 1/42  (mejor élite generación 7)
    # Proceso dominante: n=4 → ÚNICA simulación con rugosidad
    # alta (laderas disectadas y afloramientos rocosos del Pañe)
    # NSE=0.906 | KGE=0.935 | fit=0.5100
    # ──────────────────────────────────────────────────────
    "ESC_02_G7_I1": {
        "descripcion": "GEN 7/20 - IND 1/42",
        "g7": 1,
        "g9": 1,
        "n": 4,
        "K": 3.172593028372119,
        "k_infil": 0.6953880721605649,
        "k_et": 0.7383128058977291,
        "Vs": 0.009594970137013482,
        "CN10_g7": 95.72124246470727,
        "CN16_g7": 85.71535662377057,
        "CN10_g9": 96.00364074025487,
        "CN16_g9": 64.9361690734845,
        "k_effP": 1.7885834590885665,
        "k_q": 1.7117817529966601,
        "lambda_ia": 0.0273135448843369,
    },

    # ──────────────────────────────────────────────────────
    # ESC_03 | GEN 5/20 – IND 22/42
    # Proceso dominante: k_infil más alto → Bofedales activos
    # con alta infiltración y bajo umbral de escorrentía.
    # MEJOR KGE DE TODA LA CALIBRACIÓN: 0.946
    # NSE=0.900 | KGE=0.946 | fit=0.4934
    # ──────────────────────────────────────────────────────
    "ESC_03_G5_I22": {
        "descripcion": "GEN 5/20 - IND 22/42",
        "g7": 1,
        "g9": 1,
        "n": 3,
        "K": 3.480499775607945,
        "k_infil": 0.9485890504556826,
        "k_et": 0.8082241252469766,
        "Vs": 0.004926338656781407,
        "CN10_g7": 94.23137497412777,
        "CN16_g7": 89.58997744975092,
        "CN10_g9": 93.58212039124682,
        "CN16_g9": 65.88151003572375,
        "k_effP": 1.675596903886662,
        "k_q": 2.0796434963515664,
        "lambda_ia": 0.014808840897843296,
    },

    # ──────────────────────────────────────────────────────
    # ESC_04 | GEN 6/20 – IND 21/42
    # Proceso dominante: k_ET extremo (0.968) + CN10_g9=98.0
    # → Máxima demanda ET atmosférica en puna + suelo casi
    #   impermeable en condición húmeda. λ ≈ 0.05 (Woodward 2003)
    # NSE=0.898 | KGE=0.919 | fit=0.4887
    # ──────────────────────────────────────────────────────
    "ESC_04_G6_I21": {
        "descripcion": "GEN 6/20 - IND 21/42",
        "g7": 1,
        "g9": 1,
        "n": 3,
        "K": 4.113599777489887,
        "k_infil": 0.8466222840708213,
        "k_et": 0.9677483310758396,
        "Vs": 0.009594970137013482,
        "CN10_g7": 95.72124246470727,
        "CN16_g7": 89.88214089159406,
        "CN10_g9": 98.0,
        "CN16_g9": 66.78377348638809,
        "k_effP": 1.675596903886662,
        "k_q": 1.770322706116105,
        "lambda_ia": 0.0495655039855523,
    },

    # ──────────────────────────────────────────────────────
    # ESC_05 | GEN 20/20 – IND 1/42  (MEJOR FIT DE TODA LA CALIBRACIÓN)
    # Proceso dominante: λ=0.005 mínimo → escorrentía directa
    # e inmediata (suelo costroso / eventos convectivos intensos)
    # MEJOR NSE: 0.910 | KGE=0.929 | fit=0.5206 (máximo global)
    # ──────────────────────────────────────────────────────
    "ESC_05_G20_I1": {
        "descripcion": "GEN 20/20 - IND 1/42",
        "g7": 1,
        "g9": 2,
        "n": 2,
        "K": 5.029109891588228,
        "k_infil": 0.6953880721605649,
        "k_et": 0.761905229683596,
        "Vs": 0.009792343333038724,
        "CN10_g7": 94.55309804004582,
        "CN16_g7": 89.58997744975092,
        "CN10_g9": 95.60732483794743,
        "CN16_g9": 66.89345682860497,
        "k_effP": 1.817156163223742,
        "k_q": 1.7679524705766723,
        "lambda_ia": 0.005,
    },
}


# ==========================================================
# 4) UTILIDADES GENERALES
# ==========================================================
def load_geoms(geojson_path: str):
    if not os.path.exists(geojson_path):
        raise FileNotFoundError(f"No existe AOI: {geojson_path}")
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        raise ValueError(f"AOI vacío: {geojson_path}")
    geom = gdf.geometry.union_all()
    return [geom]


def read_clip_stack(tif_path: str, geoms):
    if not os.path.exists(tif_path):
        raise FileNotFoundError(f"No existe raster: {tif_path}")
    with rasterio.open(tif_path) as src:
        out_img, _ = mask(src, geoms, crop=True, filled=True)
        nodata = src.nodata
    return out_img, nodata


def nanmean_masked(a: np.ndarray, nodata=None) -> float:
    a = a.astype("float64", copy=False)
    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)
    return float(np.nanmean(a))


def find_month_file(folder: str, prefix: str, year: int, month: int) -> str:
    pattern = os.path.join(folder, f"{prefix}_{year}_{month:02d}_WGS84*.*")
    hits = sorted(glob.glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No encontré archivo con patrón:\n  {pattern}")
    return hits[0]


def find_single_tif(folder: str) -> str:
    hits = sorted(glob.glob(os.path.join(folder, "*.tif"))) + sorted(glob.glob(os.path.join(folder, "*.tiff")))
    if not hits:
        raise FileNotFoundError(f"No encontré tif/tiff en: {folder}")
    return hits[0]


# ==========================================================
# 5) QM MENSUAL
# ==========================================================
def load_qm_table(csv_path: str):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No existe QM CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"mes", "x", "y"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"QM_model_monthly.csv debe tener columnas {required}. Tiene: {list(df.columns)}")

    qm = {}
    for _, r in df.iterrows():
        mes = int(r["mes"])
        x = json.loads(r["x"]) if isinstance(r["x"], str) else r["x"]
        y = json.loads(r["y"]) if isinstance(r["y"], str) else r["y"]

        x = np.array(x, dtype=float)
        y = np.array(y, dtype=float)

        if x.size < 2 or y.size < 2 or x.size != y.size:
            raise ValueError(f"QM inválido en mes {mes}: x={x.size}, y={y.size}")

        qm[mes] = (x, y)

    faltantes = [m for m in range(1, 13) if m not in qm]
    if faltantes:
        raise ValueError(f"Faltan meses en QM: {faltantes}")

    return qm


def qm_correct_array(arr: np.ndarray, month: int, QM_TABLE):
    x, y = QM_TABLE[month]
    idx = np.argsort(x)
    x2 = x[idx]
    y2 = y[idx]
    flat = arr.reshape(-1).astype("float64", copy=False)
    out = np.interp(flat, x2, y2).reshape(arr.shape)
    return np.maximum(out, 0.0)


# ==========================================================
# 6) SCS-CN DISTRIBUIDO
# ==========================================================
CN_TABLE = {
    1: {1:35, 2:25, 3:45, 4:39, 5:45, 6:49, 7:68, 8:36, 9:45, 10:30, 11:95, 12:67, 13:72, 14:63, 15:100, 16:74, 17:100},
    2: {1:50, 2:55, 3:66, 4:61, 5:66, 6:69, 7:79, 8:60, 9:66, 10:58, 11:95, 12:78, 13:82, 14:75, 15:100, 16:84, 17:100},
    3: {1:73, 2:70, 3:77, 4:74, 5:77, 6:79, 7:86, 8:73, 9:77, 10:71, 11:95, 12:85, 13:87, 14:83, 15:100, 16:90, 17:100},
    4: {1:79, 2:77, 3:83, 4:80, 5:83, 6:89, 7:89, 8:79, 9:83, 10:78, 11:95, 12:89, 13:89, 14:87, 15:100, 16:92, 17:100},
}


def soil_group_from_texture(soil_class, g7=2, g9=2):
    soil_grp = np.zeros_like(soil_class, dtype="int16")
    soil_grp = np.where(soil_class > 10, 1, soil_grp)
    soil_grp = np.where((soil_class > 4) & (soil_class <= 10), 2, soil_grp)
    soil_grp = np.where((soil_class > 1) & (soil_class <= 4), 3, soil_grp)
    soil_grp = np.where((soil_class > 0) & (soil_class <= 1), 4, soil_grp)

    soil_grp = np.where(soil_class == 7, int(g7), soil_grp)
    soil_grp = np.where(soil_class == 9, int(g9), soil_grp)
    return soil_grp


def reemplazar_cn_lulc10_16(CN10_g7, CN16_g7, CN10_g9, CN16_g9, g7=2, g9=2):
    cn_table_local = {k: v.copy() for k, v in CN_TABLE.items()}
    cn_table_local[int(g7)][10] = float(CN10_g7)
    cn_table_local[int(g7)][16] = float(CN16_g7)
    cn_table_local[int(g9)][10] = float(CN10_g9)
    cn_table_local[int(g9)][16] = float(CN16_g9)
    return cn_table_local


def build_cn_s_from_cache(LULC_CUENCA, SOIL_CUENCA, cn_table_local, g7=2, g9=2, f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4):
    lulc = LULC_CUENCA.astype("float64")
    soil = SOIL_CUENCA.astype("float64")
    soil_grp = soil_group_from_texture(soil, g7=g7, g9=g9)

    CN2 = np.zeros_like(lulc, dtype="float64")
    CN2 = np.where(soil_grp == 0, 100.0, CN2)

    for grp, lut in cn_table_local.items():
        for lc, cn in lut.items():
            CN2 = np.where((soil_grp == grp) & (lulc == lc), float(cn), CN2)

    CN2 = np.minimum(CN2 * float(f_cn2), 100.0)

    CN1 = CN2 / (2.281 - (CN2 * 0.0128))
    CN3 = CN2 / (0.427 + (CN2 * 0.00573))

    S1 = (25400.0 / CN1 - 254.0) * float(f_s1)
    S2 = (25400.0 / CN2 - 254.0) * float(f_s2)
    S3 = (25400.0 / CN3 - 254.0) * float(f_s3)

    return S1, S2, S3


def rolling_sum_5d(stack_TRC: np.ndarray, prev4=None):
    if prev4 is None:
        prev4 = np.zeros((0,) + stack_TRC.shape[1:], dtype=stack_TRC.dtype)

    combo = np.concatenate([prev4, stack_TRC], axis=0)
    cs = np.cumsum(combo, axis=0)

    amc = np.empty_like(combo, dtype="float64")
    for i in range(combo.shape[0]):
        if i < 4:
            amc[i] = np.sum(combo[:i+1], axis=0)
        else:
            amc[i] = cs[i] - cs[i-5]

    tail4 = combo[-4:] if combo.shape[0] >= 4 else combo
    return amc[-stack_TRC.shape[0]:], tail4


def runoff_scs(ppt, amc5, S1, S2, S3, lambda_ia=0.2):
    S = np.where(amc5 <= 13, S1, S2)
    S = np.where(amc5 > 28, S3, S)

    lam = float(np.clip(lambda_ia, 0.001, 0.8))
    Ia = lam * S

    numer = np.power(ppt - Ia, 2)
    denom = (ppt - Ia) + S

    with np.errstate(divide="ignore", invalid="ignore"):
        Q = np.where(ppt < Ia, 0.0, numer / denom)

    return np.where(np.isfinite(Q), Q, 0.0)


# ==========================================================
# 7) RUTEO
# ==========================================================
def route_linear_reservoir(runoff_mm, K=2.0):
    r = np.asarray(runoff_mm, dtype=float)
    q = np.zeros_like(r)
    alpha = np.exp(-1.0 / max(K, 1e-6))

    for t in range(len(r)):
        q[t] = (1 - alpha) * r[t] if t == 0 else alpha * q[t-1] + (1 - alpha) * r[t]

    return q


def route_nash_cascade(runoff_mm, n=3, K=2.0):
    q = np.asarray(runoff_mm, dtype=float)
    for _ in range(int(n)):
        q = route_linear_reservoir(q, K=K)
    return q


# ==========================================================
# 8) EMBALSE
# ==========================================================
def area_embalse(V):
    D366 = V / 1e6
    area_km2 = (
        8.91946288801159E-08 * D366**4
        - 0.0000346583770904819 * D366**3
        + 0.003919249094071 * D366**2
        - 0.0306477700093546 * D366
        + 2.02308232795992
    )
    return max(area_km2, 0) * 1e6


def volumen_auxiliar(V_prev, esc_mm, A_c, Vd):
    A_emb = area_embalse(V_prev)
    A_aporte = max(A_c - A_emb, 0)
    esc_m = esc_mm / 1000.0
    V_esc = esc_m * A_aporte
    return V_prev + V_esc - Vd


def area_media(V_prev, V_aux):
    return 0.5 * (area_embalse(V_prev) + area_embalse(V_aux))


def aplicar_restricciones(B_i):
    if B_i > VOL_MAX:
        return VOL_MAX, B_i - VOL_MAX
    if B_i < VOL_MUERTO:
        return VOL_MUERTO, 0.0
    return B_i, 0.0


def balance_embalse_diario(V_prev, esc_mm, precip_mm, et_mm, A_c, Vd, Vs_m_d, k_infil=1.0, k_et=1.0):
    V_aux = volumen_auxiliar(V_prev, esc_mm, A_c, Vd)
    A_i = area_media(V_prev, V_aux)

    esc_m = esc_mm / 1000.0
    precip_m = precip_mm / 1000.0
    et_m = (et_mm * k_et) / 1000.0

    V_esc = esc_m * max(A_c - A_i, 0.0)
    V_clima = (precip_m - et_m) * A_i
    Vs_vol = (float(Vs_m_d) * float(k_infil)) * A_i

    B_i = V_prev + V_esc + V_clima - Vd - Vs_vol
    V_emb, V_vertido = aplicar_restricciones(B_i)

    return V_emb, V_vertido, A_i, V_clima, V_esc, Vs_vol, B_i, Vd


# ==========================================================
# 9) CARGA DE DATOS FIJOS
# ==========================================================
roi_cuenca_geoms  = load_geoms(AOI_CUENCA_PATH)
roi_embalse_geoms = load_geoms(AOI_EMBALSE_PATH)

QM_TABLE = load_qm_table(QM_MODEL_PATH)

LULC_TIF = find_single_tif(LULC_DIR)
SOIL_TIF = find_single_tif(SOIL_DIR)

Vdescarga = pd.read_csv(DESCARGAS_CSV, sep=";", encoding="latin-1")
Vdescarga = Vdescarga.rename(columns={
    Vdescarga.columns[0]: "fecha",
    Vdescarga.columns[1]: "descarga_m3_s",
    Vdescarga.columns[2]: "V_emb_real_hm3",
})

Vdescarga["fecha"] = pd.to_datetime(Vdescarga["fecha"], dayfirst=True, errors="coerce").dt.floor("D")
Vdescarga["descarga_m3_s"] = pd.to_numeric(Vdescarga["descarga_m3_s"], errors="coerce")
Vdescarga["V_emb_real_hm3"] = pd.to_numeric(Vdescarga["V_emb_real_hm3"], errors="coerce")

Vdescarga = Vdescarga[
    (Vdescarga["fecha"] >= SERIE_START) &
    (Vdescarga["fecha"] <= SERIE_END)
].dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)

Vdescarga = Vdescarga.drop_duplicates(subset=["fecha"], keep="last")


# ==========================================================
# 10) CACHE RASTERS ESTÁTICOS
# ==========================================================
lulc_stack, _ = read_clip_stack(LULC_TIF, roi_cuenca_geoms)
soil_stack, _ = read_clip_stack(SOIL_TIF, roi_cuenca_geoms)

LULC_CUENCA = lulc_stack[0].astype("int16")
SOIL_CUENCA = soil_stack[0].astype("int16")

print("LULC únicos (cuenca):", np.unique(LULC_CUENCA))
print("SOIL únicos (cuenca):", np.unique(SOIL_CUENCA))


# ==========================================================
# 11) MÉTRICAS
# ==========================================================
def _remove_excluded_period(df):
    d = df.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.floor("D")

    exc_s = pd.to_datetime(EXC_START)
    exc_e = pd.to_datetime(EXC_END)

    return d[~((d["fecha"] >= exc_s) & (d["fecha"] <= exc_e))].reset_index(drop=True)


def _compute_metrics(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    m = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[m]
    obs = obs[m]

    if len(obs) < 10:
        return {
            "n": int(len(obs)),
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan
        }

    if np.std(obs) > 0 and np.std(sim) > 0:
        r = float(np.corrcoef(obs, sim)[0, 1])
        R2 = float(r ** 2)
    else:
        r = 0.0
        R2 = 0.0

    RMSE = float(np.sqrt(np.mean((sim - obs) ** 2)))
    denom = float(np.sum((obs - np.mean(obs)) ** 2))
    NSE = float(1 - np.sum((sim - obs) ** 2) / denom) if denom > 0 else np.nan

    sigma = float(np.std(obs))
    RMSE_n = float(RMSE / sigma) if sigma > 0 else np.nan

    mu_o = float(np.mean(obs))
    mu_s = float(np.mean(sim))
    Bias = float((mu_s - mu_o) / mu_o) if abs(mu_o) > 1e-12 else np.nan
    PBIAS = float(Bias * 100.0) if np.isfinite(Bias) else np.nan

    alpha = float(np.std(sim) / np.std(obs)) if np.std(obs) > 0 else np.nan
    beta = float(mu_s / mu_o) if abs(mu_o) > 1e-12 else np.nan

    if np.isfinite(r) and np.isfinite(alpha) and np.isfinite(beta):
        KGE = float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))
    else:
        KGE = np.nan

    return {
        "n": int(len(obs)),
        "R2": R2,
        "RMSE": RMSE,
        "NSE": NSE,
        "RMSE_n": RMSE_n,
        "Bias": Bias,
        "PBIAS": PBIAS,
        "KGE": KGE
    }


# ==========================================================
# 12) FILTROS DE FECHAS
# ==========================================================
def _slice_df_by_dates(df_res, start, end):
    s = pd.to_datetime(start)
    e = pd.to_datetime(end)

    d = df_res.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.floor("D")

    return d[(d["fecha"] >= s) & (d["fecha"] <= e)].reset_index(drop=True)


def _build_calibration_df(df_res, cal_start, cal_end):
    d = _slice_df_by_dates(df_res, cal_start, cal_end)
    d = _remove_excluded_period(d)
    return d


def _build_validation_df(df_res, val_start, val_end):
    d = _slice_df_by_dates(df_res, val_start, val_end)
    d = _remove_excluded_period(d)
    return d


# ==========================================================
# 13) SIMULACIÓN COMPLETA CON SALIDAS DETALLADAS
# ==========================================================
def simular_detallado(params, metric_start=None, metric_end=None):
    cn_table_local = reemplazar_cn_lulc10_16(
        params["CN10_g7"], params["CN16_g7"],
        params["CN10_g9"], params["CN16_g9"],
        g7=params["g7"], g9=params["g9"]
    )

    S1_CU, S2_CU, S3_CU = build_cn_s_from_cache(
        LULC_CUENCA, SOIL_CUENCA,
        cn_table_local=cn_table_local,
        g7=params["g7"], g9=params["g9"],
        f_cn2=1.0,
        f_s1=1.3, f_s2=1.2, f_s3=1.4
    )

    rows = []
    for year in range(START_YEAR, END_YEAR + 1):
        prev4 = None
        for month in range(1, 13):
            chirps_path = find_month_file(CHIRPS_DIR, "CHIRPS_PPT_DAILY", year, month)
            et_path     = find_month_file(ET_DIR, "ET_HARGREAVES_DAILY", year, month)

            ch_cu_stack, _ = read_clip_stack(chirps_path, roi_cuenca_geoms)
            ppt_cu = qm_correct_array(ch_cu_stack.astype("float64"), month, QM_TABLE)
            ppt_cu_eff = ppt_cu * float(params["k_effP"])

            amc5, prev4 = rolling_sum_5d(ppt_cu_eff, prev4=prev4)
            q_cu = runoff_scs(
                ppt_cu_eff, amc5, S1_CU, S2_CU, S3_CU,
                lambda_ia=float(params["lambda_ia"])
            )

            q_mean = [nanmean_masked(q_cu[i], None) for i in range(q_cu.shape[0])]
            q_mean = [v * float(params["k_q"]) for v in q_mean]

            ppt_cu_mean = [nanmean_masked(ppt_cu_eff[i], None) for i in range(ppt_cu_eff.shape[0])]

            ch_em_stack, ch_em_nodata = read_clip_stack(chirps_path, roi_embalse_geoms)
            ppt_em = qm_correct_array(ch_em_stack.astype("float64"), month, QM_TABLE)
            ppt_em_mean = [nanmean_masked(ppt_em[i], ch_em_nodata) for i in range(ppt_em.shape[0])]
            ppt_em_mean = [v * float(params["k_effP"]) for v in ppt_em_mean]

            et_em_stack, et_em_nodata = read_clip_stack(et_path, roi_embalse_geoms)
            et_em = et_em_stack.astype("float64")
            et_em_mean = [nanmean_masked(et_em[i], et_em_nodata) for i in range(et_em.shape[0])]

            ndays = calendar.monthrange(year, month)[1]
            T = min(ndays, len(q_mean), len(ppt_em_mean), len(et_em_mean), len(ppt_cu_mean))

            for d in range(T):
                rows.append({
                    "fecha": datetime(year, month, d + 1),
                    "ppt_cuenca_eff_mm": float(ppt_cu_mean[d]),
                    "esc_mm": float(q_mean[d]),
                    "precip_mm": float(ppt_em_mean[d]),
                    "et_mm": float(et_em_mean[d]),
                })

    df_forz = pd.DataFrame(rows).sort_values("fecha").reset_index(drop=True)
    df_forz["esc_ruteada_mm"] = route_nash_cascade(
        df_forz["esc_mm"].to_numpy(),
        n=int(params["n"]),
        K=float(params["K"])
    )

    df_forz["fecha"] = pd.to_datetime(df_forz["fecha"]).dt.floor("D")
    df_forz = df_forz.drop_duplicates(subset=["fecha"], keep="last")

    df_join = pd.merge(
        df_forz[["fecha", "ppt_cuenca_eff_mm", "esc_mm", "esc_ruteada_mm", "precip_mm", "et_mm"]],
        Vdescarga[["fecha", "descarga_m3_s", "V_emb_real_hm3"]],
        on="fecha",
        how="inner"
    ).sort_values("fecha").reset_index(drop=True)

    if len(df_join) < 100:
        df_res_full = pd.DataFrame({"fecha": [], "V_emb": [], "V_emb_real": []})
        met = {
            "R2": np.nan, "RMSE": np.nan, "NSE": np.nan,
            "RMSE_n": np.nan, "Bias": np.nan, "PBIAS": np.nan, "KGE": np.nan, "n": 0
        }
        return df_res_full, met

    resultados = []
    V_emb = None

    for i in range(len(df_join)):
        V_prev = V_INICIAL if i == 0 else V_emb

        fecha = df_join.loc[i, "fecha"]
        Vd = float(df_join.loc[i, "descarga_m3_s"]) * 86400.0
        V_obs = float(df_join.loc[i, "V_emb_real_hm3"]) * 1_000_000.0

        esc_mm = float(df_join.loc[i, "esc_ruteada_mm"])
        p_mm   = float(df_join.loc[i, "precip_mm"])
        et_mm  = float(df_join.loc[i, "et_mm"])

        V_emb, V_vert, A_i, V_clima, V_esc, Vs_vol, B_i, Vd = balance_embalse_diario(
            V_prev, esc_mm, p_mm, et_mm,
            AREA_CUENCA_TOTAL, Vd,
            Vs_m_d=float(params["Vs"]),
            k_infil=float(params["k_infil"]),
            k_et=float(params["k_et"])
        )

        resultados.append({
            "fecha": fecha,
            "ppt_cuenca_eff_mm": float(df_join.loc[i, "ppt_cuenca_eff_mm"]),
            "esc_mm_sin_ruteo": float(df_join.loc[i, "esc_mm"]),
            "esc_mm_ruteada": float(df_join.loc[i, "esc_ruteada_mm"]),
            "precip_mm_embalse": p_mm,
            "et_mm_embalse": et_mm,
            "descarga_m3_s": float(df_join.loc[i, "descarga_m3_s"]),
            "descarga_m3_dia": Vd,
            "V_emb": V_emb,
            "V_emb_hm3": V_emb / 1e6,
            "V_emb_real": V_obs,
            "V_emb_real_hm3": V_obs / 1e6,
            "V_vertido": V_vert,
            "area_embalse_m2": A_i,
            "V_clima": V_clima,
            "V_esc": V_esc,
            "Vs_vol": Vs_vol,
            "balance_bruto": B_i
        })

    df_res_full = pd.DataFrame(resultados)
    df_res_full["fecha"] = pd.to_datetime(df_res_full["fecha"]).dt.floor("D")

    if metric_start is None or metric_end is None:
        dmet = df_res_full.copy()
    else:
        dmet = _slice_df_by_dates(df_res_full, metric_start, metric_end)

    dmet = _remove_excluded_period(dmet)
    met = _compute_metrics(dmet["V_emb"].values, dmet["V_emb_real"].values)

    return df_res_full, met


# ==========================================================
# 14) GRAFICADO DE UN ESCENARIO
# ==========================================================
def graficar_escenario(df_full, met_cal, met_val, nombre_esc, descripcion, out_png):
    plt.figure(figsize=(16, 5))

    df_obs_plot = df_full.copy()
    exc_s = pd.to_datetime(EXC_START)
    exc_e = pd.to_datetime(EXC_END)
    mask_exc = (df_obs_plot["fecha"] >= exc_s) & (df_obs_plot["fecha"] <= exc_e)
    df_obs_plot.loc[mask_exc, "V_emb_real"] = np.nan

    sim = df_full["V_emb"].to_numpy(dtype=float)
    sim_low = sim * 0.85
    sim_high = sim * 1.15

    plt.fill_between(
        df_full["fecha"],
        sim_low,
        sim_high,
        alpha=0.18,
        label="Banda ±15% Sim"
    )

    plt.plot(df_obs_plot["fecha"], df_obs_plot["V_emb_real"], linewidth=1.5, label="Obs")
    plt.plot(df_full["fecha"], df_full["V_emb"], linewidth=1.8, label="Sim")

    plt.axvline(pd.to_datetime(CAL_START), linestyle="--", linewidth=1.2)
    plt.axvline(pd.to_datetime(CAL_END), linestyle="--", linewidth=1.2)
    plt.axvline(pd.to_datetime(VAL_START), linestyle="--", linewidth=1.2)

    texto_cal = (
        "CALIBRACIÓN\n"
        f"R²={met_cal['R2']:.3f}\n"
        f"RMSE={met_cal['RMSE']:.2e}\n"
        f"NSE={met_cal['NSE']:.3f}\n"
        f"KGE={met_cal['KGE']:.3f}\n"
        f"Bias={met_cal['Bias']:.4f}"
    )

    texto_val = (
        "VALIDACIÓN\n"
        f"R²={met_val['R2']:.3f}\n"
        f"RMSE={met_val['RMSE']:.2e}\n"
        f"NSE={met_val['NSE']:.3f}\n"
        f"KGE={met_val['KGE']:.3f}\n"
        f"Bias={met_val['Bias']:.4f}"
    )

    plt.text(
        0.015, 0.97, texto_cal,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.text(
        0.165, 0.97, texto_val,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.title(
        f"Volumen embalse 2010–2019 | {nombre_esc}\n"
        f"{descripcion}\n"
        f"Calibración: {CAL_START} a {CAL_END} | Validación: {VAL_START} a {VAL_END}"
    )
    plt.xlabel("Fecha")
    plt.ylabel("Volumen (m3)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.show()


# ==========================================================
# 15) EJECUCIÓN MULTI-ESCENARIO
# ==========================================================
print("=== MODELO MULTI-ESCENARIO ===")

resumen_global = []
params_global = []
series_all = []

for nombre_esc, params in ESCENARIOS.items():
    print("\n" + "=" * 80)
    print(f"Ejecutando {nombre_esc}")
    print(params["descripcion"])
    print("=" * 80)
    print(params)

    df_full, _ = simular_detallado(
        params,
        metric_start=CAL_START,
        metric_end=CAL_END
    )

    df_full["fecha"] = pd.to_datetime(df_full["fecha"]).dt.floor("D")

    df_cal = _build_calibration_df(df_full, CAL_START, CAL_END)
    df_val = _build_validation_df(df_full, VAL_START, VAL_END)

    met_cal = _compute_metrics(df_cal["V_emb"], df_cal["V_emb_real"])
    met_val = _compute_metrics(df_val["V_emb"], df_val["V_emb_real"])

    print("\n--- MÉTRICAS CALIBRACIÓN ---")
    print(met_cal)
    print("--- MÉTRICAS VALIDACIÓN ---")
    print(met_val)

    # Guardar serie individual CSV
    out_csv = os.path.join(OUT_DIR, f"serie_{nombre_esc}.csv")
    df_out = df_full.copy()
    df_out["escenario"] = nombre_esc
    df_out["descripcion"] = params["descripcion"]
    df_out.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # Gráfico individual
    out_png = os.path.join(OUT_DIR, f"grafica_{nombre_esc}.png")
    graficar_escenario(df_full, met_cal, met_val, nombre_esc, params["descripcion"], out_png)

    # Acumulados resumen
    resumen_global.append({
        "escenario": nombre_esc,
        "descripcion": params["descripcion"],
        "etapa": "CALIBRACION",
        **met_cal
    })
    resumen_global.append({
        "escenario": nombre_esc,
        "descripcion": params["descripcion"],
        "etapa": "VALIDACION",
        **met_val
    })

    for k, v in params.items():
        params_global.append({
            "escenario": nombre_esc,
            "descripcion": params["descripcion"],
            "parametro": k,
            "valor": v
        })

    df_long = df_full.copy()
    df_long["escenario"] = nombre_esc
    df_long["descripcion"] = params["descripcion"]
    series_all.append(df_long)


# ==========================================================
# 16) UNIFICAR RESULTADOS
# ==========================================================
df_resumen = pd.DataFrame(resumen_global)
df_params = pd.DataFrame(params_global)
df_series = pd.concat(series_all, ignore_index=True)

propiedades_modelo = pd.DataFrame([
    {"Propiedad": "VOL_MUERTO_m3", "Valor": VOL_MUERTO},
    {"Propiedad": "VOL_MAX_m3", "Valor": VOL_MAX},
    {"Propiedad": "AREA_CUENCA_TOTAL_m2", "Valor": AREA_CUENCA_TOTAL},
    {"Propiedad": "V_INICIAL_m3", "Valor": V_INICIAL},
    {"Propiedad": "CAL_START", "Valor": CAL_START},
    {"Propiedad": "CAL_END", "Valor": CAL_END},
    {"Propiedad": "VAL_START", "Valor": VAL_START},
    {"Propiedad": "VAL_END", "Valor": VAL_END},
    {"Propiedad": "SERIE_START", "Valor": SERIE_START},
    {"Propiedad": "SERIE_END", "Valor": SERIE_END},
    {"Propiedad": "EXC_START", "Valor": EXC_START},
    {"Propiedad": "EXC_END", "Valor": EXC_END},
])

# Resumen ancho por escenario
df_resumen_wide = (
    df_resumen
    .pivot_table(
        index=["escenario", "descripcion"],
        columns="etapa",
        values=["R2", "RMSE", "NSE", "KGE", "Bias", "PBIAS", "n"],
        aggfunc="first"
    )
)

df_resumen_wide.columns = [f"{a}_{b}" for a, b in df_resumen_wide.columns]
df_resumen_wide = df_resumen_wide.reset_index()

out_xlsx = os.path.join(OUT_DIR, "resultados_5_escenarios.xlsx")

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    df_resumen.to_excel(writer, sheet_name="metricas_largo", index=False)
    df_resumen_wide.to_excel(writer, sheet_name="metricas_resumen", index=False)
    df_params.to_excel(writer, sheet_name="parametros", index=False)
    propiedades_modelo.to_excel(writer, sheet_name="propiedades_modelo", index=False)
    df_series.to_excel(writer, sheet_name="series_completas", index=False)


# ==========================================================
# 17) GRÁFICA COMPARATIVA CONJUNTA
# ==========================================================
plt.figure(figsize=(18, 6))

# Observado único
df_obs = (
    df_series[["fecha", "V_emb_real"]]
    .drop_duplicates(subset=["fecha"])
    .sort_values("fecha")
    .reset_index(drop=True)
)

exc_s = pd.to_datetime(EXC_START)
exc_e = pd.to_datetime(EXC_END)
mask_exc = (df_obs["fecha"] >= exc_s) & (df_obs["fecha"] <= exc_e)
df_obs_plot = df_obs.copy()
df_obs_plot.loc[mask_exc, "V_emb_real"] = np.nan

plt.plot(df_obs_plot["fecha"], df_obs_plot["V_emb_real"], linewidth=2.2, label="Obs")

for nombre_esc in ESCENARIOS.keys():
    d = (
        df_series[df_series["escenario"] == nombre_esc]
        .sort_values("fecha")
        .reset_index(drop=True)
    )
    plt.plot(d["fecha"], d["V_emb"], linewidth=1.5, label=nombre_esc)

plt.axvline(pd.to_datetime(CAL_START), linestyle="--", linewidth=1.2)
plt.axvline(pd.to_datetime(CAL_END), linestyle="--", linewidth=1.2)
plt.axvline(pd.to_datetime(VAL_START), linestyle="--", linewidth=1.2)

plt.title(
    "Comparación de 5 escenarios físicamente representativos\n"
    "Volumen del embalse 2010–2019"
)
plt.xlabel("Fecha")
plt.ylabel("Volumen (m3)")
plt.legend(ncol=3)
plt.tight_layout()

out_png_comp = os.path.join(OUT_DIR, "comparacion_5_escenarios.png")
plt.savefig(out_png_comp, dpi=200, bbox_inches="tight")
plt.show()


# ==========================================================
# 18) TABLA FINAL EN CONSOLA
# ==========================================================
print("\n=== RESUMEN FINAL DE MÉTRICAS ===")
print(df_resumen_wide.to_string(index=False))

print("\n=== CORRESPONDENCIA DE ESCENARIOS ===")
for nombre_esc, params in ESCENARIOS.items():
    print(f"{nombre_esc} --> {params['descripcion']}")

print("\n=== ARCHIVOS GENERADOS ===")
print("Directorio principal :", OUT_DIR)
print("Excel resumen        :", out_xlsx)
print("Gráfico comparativo  :", out_png_comp)
print("CSV por escenario    : serie_ESC_*.csv")
print("PNG por escenario    : grafica_ESC_*.png")

In [ ]:
# ==========================================================
# SCRIPT 2 MODIFICADO
# SIMULACIÓN DISTRIBUIDA + EMBALSE
# MULTI-ESCENARIO (5 ESCENARIOS FÍSICAMENTE REPRESENTATIVOS)
# SERIE COMPLETA 2010-2019
# CALIBRACIÓN / VALIDACIÓN
# EXCLUYENDO UN PERIODO SOLO EN MÉTRICAS Y PLOT OBSERVADO
# + GUARDADO DE RESULTADOS
# + BANDA SOMBREADA ±15% DE LA SERIE SIMULADA
# + MÉTRICAS EN LA GRÁFICA
# + COMPARACIÓN ENTRE ESCENARIOS
# ==========================================================

# ==========================================================
# 0) CONFIGURACIÓN DE FECHAS
# ==========================================================
CAL_START = "2010-01-01"
CAL_END   = "2017-01-01"

VAL_START = "2017-01-02"
VAL_END   = "2019-12-31"

SERIE_START = "2010-01-01"
SERIE_END   = "2019-12-31"

EXC_START = "2014-04-01"
EXC_END   = "2015-06-07"


# ==========================================================
# 1) LIBRERÍAS
# ==========================================================
import os
import glob
import json
import calendar
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)


# ==========================================================
# 2) RUTAS / PARÁMETROS GENERALES
# ==========================================================
BASE_DIR = "/content/drive/MyDrive/Project001"

CHIRPS_DIR = os.path.join(BASE_DIR, "CHIRPS_PPT_DAILY_2009_2020_WGS84")
ET_DIR     = os.path.join(BASE_DIR, "ET_HARGREAVES_DAILY_2009_2020_WGS84")

LULC_DIR = os.path.join(BASE_DIR, "LULC_LC_Type1_WGS84")
SOIL_DIR = os.path.join(BASE_DIR, "SOIL_TEXTURE_b0_WGS84")

SHP_DIR = os.path.join(BASE_DIR, "shp")
AOI_CUENCA_PATH  = os.path.join(SHP_DIR, "Cuenca-es.geojson")
AOI_EMBALSE_PATH = os.path.join(SHP_DIR, "Embalse.geojson")

QM_MODEL_PATH = os.path.join(BASE_DIR, "QM_model_monthly.csv")
DESCARGAS_CSV = os.path.join(BASE_DIR, "descargas2.csv")

OUT_DIR = os.path.join(BASE_DIR, "resultados_modelo_5_escenarios")
os.makedirs(OUT_DIR, exist_ok=True)

START_YEAR = 2010
END_YEAR   = 2019

VOL_MUERTO = 41e6
VOL_MAX    = 148.80e6
AREA_CUENCA_TOTAL = 196.33e6
V_INICIAL = 57.80e6


# ==========================================================
# 3) ESCENARIOS SELECCIONADOS
# ==========================================================
# ==========================================================
# ESCENARIOS SELECCIONADOS — CUENCA EL PAÑE
# Basado en análisis de representatividad física del archivo
# de calibración genética (metricas.txt)
# ==========================================================

# ==========================================================
# 3) ESCENARIOS SELECCIONADOS
# ==========================================================
ESCENARIOS = {

    "ESC_05_G13_I6": {
        "descripcion": "GEN 13/20 - IND 6/42",
        "g7": 1,
        "g9": 2,
        "n": 4,
        "K": 4.313107873728896,
        "k_infil": 0.6953880721605649,
        "k_et": 0.8082241252469766,
        "Vs": 0.009594970137013482,
        "CN10_g7": 95.72124246470727,
        "CN16_g7": 85.71535662377057,
        "CN10_g9": 96.31605148608226,
        "CN16_g9": 66.28439128813265,
        "k_effP": 1.812739987216763,
        "k_q": 1.7089490774160854,
        "lambda_ia": 0.03914373443714243,
    }
}

# ==========================================================
# 4) UTILIDADES GENERALES
# ==========================================================
def load_geoms(geojson_path: str):
    if not os.path.exists(geojson_path):
        raise FileNotFoundError(f"No existe AOI: {geojson_path}")
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        raise ValueError(f"AOI vacío: {geojson_path}")
    geom = gdf.geometry.union_all()
    return [geom]


def read_clip_stack(tif_path: str, geoms):
    if not os.path.exists(tif_path):
        raise FileNotFoundError(f"No existe raster: {tif_path}")
    with rasterio.open(tif_path) as src:
        out_img, _ = mask(src, geoms, crop=True, filled=True)
        nodata = src.nodata
    return out_img, nodata


def nanmean_masked(a: np.ndarray, nodata=None) -> float:
    a = a.astype("float64", copy=False)
    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)
    return float(np.nanmean(a))


def find_month_file(folder: str, prefix: str, year: int, month: int) -> str:
    pattern = os.path.join(folder, f"{prefix}_{year}_{month:02d}_WGS84*.*")
    hits = sorted(glob.glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No encontré archivo con patrón:\n  {pattern}")
    return hits[0]


def find_single_tif(folder: str) -> str:
    hits = sorted(glob.glob(os.path.join(folder, "*.tif"))) + sorted(glob.glob(os.path.join(folder, "*.tiff")))
    if not hits:
        raise FileNotFoundError(f"No encontré tif/tiff en: {folder}")
    return hits[0]


# ==========================================================
# 5) QM MENSUAL
# ==========================================================
def load_qm_table(csv_path: str):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No existe QM CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"mes", "x", "y"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"QM_model_monthly.csv debe tener columnas {required}. Tiene: {list(df.columns)}")

    qm = {}
    for _, r in df.iterrows():
        mes = int(r["mes"])
        x = json.loads(r["x"]) if isinstance(r["x"], str) else r["x"]
        y = json.loads(r["y"]) if isinstance(r["y"], str) else r["y"]

        x = np.array(x, dtype=float)
        y = np.array(y, dtype=float)

        if x.size < 2 or y.size < 2 or x.size != y.size:
            raise ValueError(f"QM inválido en mes {mes}: x={x.size}, y={y.size}")

        qm[mes] = (x, y)

    faltantes = [m for m in range(1, 13) if m not in qm]
    if faltantes:
        raise ValueError(f"Faltan meses en QM: {faltantes}")

    return qm


def qm_correct_array(arr: np.ndarray, month: int, QM_TABLE):
    x, y = QM_TABLE[month]
    idx = np.argsort(x)
    x2 = x[idx]
    y2 = y[idx]
    flat = arr.reshape(-1).astype("float64", copy=False)
    out = np.interp(flat, x2, y2).reshape(arr.shape)
    return np.maximum(out, 0.0)


# ==========================================================
# 6) SCS-CN DISTRIBUIDO
# ==========================================================
CN_TABLE = {
    1: {1:35, 2:25, 3:45, 4:39, 5:45, 6:49, 7:68, 8:36, 9:45, 10:30, 11:95, 12:67, 13:72, 14:63, 15:100, 16:74, 17:100},
    2: {1:50, 2:55, 3:66, 4:61, 5:66, 6:69, 7:79, 8:60, 9:66, 10:58, 11:95, 12:78, 13:82, 14:75, 15:100, 16:84, 17:100},
    3: {1:73, 2:70, 3:77, 4:74, 5:77, 6:79, 7:86, 8:73, 9:77, 10:71, 11:95, 12:85, 13:87, 14:83, 15:100, 16:90, 17:100},
    4: {1:79, 2:77, 3:83, 4:80, 5:83, 6:89, 7:89, 8:79, 9:83, 10:78, 11:95, 12:89, 13:89, 14:87, 15:100, 16:92, 17:100},
}


def soil_group_from_texture(soil_class, g7=2, g9=2):
    soil_grp = np.zeros_like(soil_class, dtype="int16")
    soil_grp = np.where(soil_class > 10, 1, soil_grp)
    soil_grp = np.where((soil_class > 4) & (soil_class <= 10), 2, soil_grp)
    soil_grp = np.where((soil_class > 1) & (soil_class <= 4), 3, soil_grp)
    soil_grp = np.where((soil_class > 0) & (soil_class <= 1), 4, soil_grp)

    soil_grp = np.where(soil_class == 7, int(g7), soil_grp)
    soil_grp = np.where(soil_class == 9, int(g9), soil_grp)
    return soil_grp


def reemplazar_cn_lulc10_16(CN10_g7, CN16_g7, CN10_g9, CN16_g9, g7=2, g9=2):
    cn_table_local = {k: v.copy() for k, v in CN_TABLE.items()}
    cn_table_local[int(g7)][10] = float(CN10_g7)
    cn_table_local[int(g7)][16] = float(CN16_g7)
    cn_table_local[int(g9)][10] = float(CN10_g9)
    cn_table_local[int(g9)][16] = float(CN16_g9)
    return cn_table_local


def build_cn_s_from_cache(LULC_CUENCA, SOIL_CUENCA, cn_table_local, g7=2, g9=2, f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4):
    lulc = LULC_CUENCA.astype("float64")
    soil = SOIL_CUENCA.astype("float64")
    soil_grp = soil_group_from_texture(soil, g7=g7, g9=g9)

    CN2 = np.zeros_like(lulc, dtype="float64")
    CN2 = np.where(soil_grp == 0, 100.0, CN2)

    for grp, lut in cn_table_local.items():
        for lc, cn in lut.items():
            CN2 = np.where((soil_grp == grp) & (lulc == lc), float(cn), CN2)

    CN2 = np.minimum(CN2 * float(f_cn2), 100.0)

    CN1 = CN2 / (2.281 - (CN2 * 0.0128))
    CN3 = CN2 / (0.427 + (CN2 * 0.00573))

    S1 = (25400.0 / CN1 - 254.0) * float(f_s1)
    S2 = (25400.0 / CN2 - 254.0) * float(f_s2)
    S3 = (25400.0 / CN3 - 254.0) * float(f_s3)

    return S1, S2, S3


def rolling_sum_5d(stack_TRC: np.ndarray, prev4=None):
    if prev4 is None:
        prev4 = np.zeros((0,) + stack_TRC.shape[1:], dtype=stack_TRC.dtype)

    combo = np.concatenate([prev4, stack_TRC], axis=0)
    cs = np.cumsum(combo, axis=0)

    amc = np.empty_like(combo, dtype="float64")
    for i in range(combo.shape[0]):
        if i < 4:
            amc[i] = np.sum(combo[:i+1], axis=0)
        else:
            amc[i] = cs[i] - cs[i-5]

    tail4 = combo[-4:] if combo.shape[0] >= 4 else combo
    return amc[-stack_TRC.shape[0]:], tail4


def runoff_scs(ppt, amc5, S1, S2, S3, lambda_ia=0.2):
    S = np.where(amc5 <= 13, S1, S2)
    S = np.where(amc5 > 28, S3, S)

    lam = float(np.clip(lambda_ia, 0.001, 0.8))
    Ia = lam * S

    numer = np.power(ppt - Ia, 2)
    denom = (ppt - Ia) + S

    with np.errstate(divide="ignore", invalid="ignore"):
        Q = np.where(ppt < Ia, 0.0, numer / denom)

    return np.where(np.isfinite(Q), Q, 0.0)


# ==========================================================
# 7) RUTEO
# ==========================================================
def route_linear_reservoir(runoff_mm, K=2.0):
    r = np.asarray(runoff_mm, dtype=float)
    q = np.zeros_like(r)
    alpha = np.exp(-1.0 / max(K, 1e-6))

    for t in range(len(r)):
        q[t] = (1 - alpha) * r[t] if t == 0 else alpha * q[t-1] + (1 - alpha) * r[t]

    return q


def route_nash_cascade(runoff_mm, n=3, K=2.0):
    q = np.asarray(runoff_mm, dtype=float)
    for _ in range(int(n)):
        q = route_linear_reservoir(q, K=K)
    return q


# ==========================================================
# 8) EMBALSE
# ==========================================================
def area_embalse(V):
    D366 = V / 1e6
    area_km2 = (
        8.91946288801159E-08 * D366**4
        - 0.0000346583770904819 * D366**3
        + 0.003919249094071 * D366**2
        - 0.0306477700093546 * D366
        + 2.02308232795992
    )
    return max(area_km2, 0) * 1e6


def volumen_auxiliar(V_prev, esc_mm, A_c, Vd):
    A_emb = area_embalse(V_prev)
    A_aporte = max(A_c - A_emb, 0)
    esc_m = esc_mm / 1000.0
    V_esc = esc_m * A_aporte
    return V_prev + V_esc - Vd


def area_media(V_prev, V_aux):
    return 0.5 * (area_embalse(V_prev) + area_embalse(V_aux))


def aplicar_restricciones(B_i):
    if B_i > VOL_MAX:
        return VOL_MAX, B_i - VOL_MAX
    if B_i < VOL_MUERTO:
        return VOL_MUERTO, 0.0
    return B_i, 0.0


def balance_embalse_diario(V_prev, esc_mm, precip_mm, et_mm, A_c, Vd, Vs_m_d, k_infil=1.0, k_et=1.0):
    V_aux = volumen_auxiliar(V_prev, esc_mm, A_c, Vd)
    A_i = area_media(V_prev, V_aux)

    esc_m = esc_mm / 1000.0
    precip_m = precip_mm / 1000.0
    et_m = (et_mm * k_et) / 1000.0

    V_esc = esc_m * max(A_c - A_i, 0.0)
    V_clima = (precip_m - et_m) * A_i
    Vs_vol = (float(Vs_m_d) * float(k_infil)) * A_i

    B_i = V_prev + V_esc + V_clima - Vd - Vs_vol
    V_emb, V_vertido = aplicar_restricciones(B_i)

    return V_emb, V_vertido, A_i, V_clima, V_esc, Vs_vol, B_i, Vd


# ==========================================================
# 9) CARGA DE DATOS FIJOS
# ==========================================================
roi_cuenca_geoms  = load_geoms(AOI_CUENCA_PATH)
roi_embalse_geoms = load_geoms(AOI_EMBALSE_PATH)

QM_TABLE = load_qm_table(QM_MODEL_PATH)

LULC_TIF = find_single_tif(LULC_DIR)
SOIL_TIF = find_single_tif(SOIL_DIR)

Vdescarga = pd.read_csv(DESCARGAS_CSV, sep=";", encoding="latin-1")
Vdescarga = Vdescarga.rename(columns={
    Vdescarga.columns[0]: "fecha",
    Vdescarga.columns[1]: "descarga_m3_s",
    Vdescarga.columns[2]: "V_emb_real_hm3",
})

Vdescarga["fecha"] = pd.to_datetime(Vdescarga["fecha"], dayfirst=True, errors="coerce").dt.floor("D")
Vdescarga["descarga_m3_s"] = pd.to_numeric(Vdescarga["descarga_m3_s"], errors="coerce")
Vdescarga["V_emb_real_hm3"] = pd.to_numeric(Vdescarga["V_emb_real_hm3"], errors="coerce")

Vdescarga = Vdescarga[
    (Vdescarga["fecha"] >= SERIE_START) &
    (Vdescarga["fecha"] <= SERIE_END)
].dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)

Vdescarga = Vdescarga.drop_duplicates(subset=["fecha"], keep="last")


# ==========================================================
# 10) CACHE RASTERS ESTÁTICOS
# ==========================================================
lulc_stack, _ = read_clip_stack(LULC_TIF, roi_cuenca_geoms)
soil_stack, _ = read_clip_stack(SOIL_TIF, roi_cuenca_geoms)

LULC_CUENCA = lulc_stack[0].astype("int16")
SOIL_CUENCA = soil_stack[0].astype("int16")

print("LULC únicos (cuenca):", np.unique(LULC_CUENCA))
print("SOIL únicos (cuenca):", np.unique(SOIL_CUENCA))


# ==========================================================
# 11) MÉTRICAS
# ==========================================================
def _remove_excluded_period(df):
    d = df.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.floor("D")

    exc_s = pd.to_datetime(EXC_START)
    exc_e = pd.to_datetime(EXC_END)

    return d[~((d["fecha"] >= exc_s) & (d["fecha"] <= exc_e))].reset_index(drop=True)


def _compute_metrics(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    m = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[m]
    obs = obs[m]

    if len(obs) < 10:
        return {
            "n": int(len(obs)),
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan
        }

    if np.std(obs) > 0 and np.std(sim) > 0:
        r = float(np.corrcoef(obs, sim)[0, 1])
        R2 = float(r ** 2)
    else:
        r = 0.0
        R2 = 0.0

    RMSE = float(np.sqrt(np.mean((sim - obs) ** 2)))
    denom = float(np.sum((obs - np.mean(obs)) ** 2))
    NSE = float(1 - np.sum((sim - obs) ** 2) / denom) if denom > 0 else np.nan

    sigma = float(np.std(obs))
    RMSE_n = float(RMSE / sigma) if sigma > 0 else np.nan

    mu_o = float(np.mean(obs))
    mu_s = float(np.mean(sim))
    Bias = float((mu_s - mu_o) / mu_o) if abs(mu_o) > 1e-12 else np.nan
    PBIAS = float(Bias * 100.0) if np.isfinite(Bias) else np.nan

    alpha = float(np.std(sim) / np.std(obs)) if np.std(obs) > 0 else np.nan
    beta = float(mu_s / mu_o) if abs(mu_o) > 1e-12 else np.nan

    if np.isfinite(r) and np.isfinite(alpha) and np.isfinite(beta):
        KGE = float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))
    else:
        KGE = np.nan

    return {
        "n": int(len(obs)),
        "R2": R2,
        "RMSE": RMSE,
        "NSE": NSE,
        "RMSE_n": RMSE_n,
        "Bias": Bias,
        "PBIAS": PBIAS,
        "KGE": KGE
    }


# ==========================================================
# 12) FILTROS DE FECHAS
# ==========================================================
def _slice_df_by_dates(df_res, start, end):
    s = pd.to_datetime(start)
    e = pd.to_datetime(end)

    d = df_res.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.floor("D")

    return d[(d["fecha"] >= s) & (d["fecha"] <= e)].reset_index(drop=True)


def _build_calibration_df(df_res, cal_start, cal_end):
    d = _slice_df_by_dates(df_res, cal_start, cal_end)
    d = _remove_excluded_period(d)
    return d


def _build_validation_df(df_res, val_start, val_end):
    d = _slice_df_by_dates(df_res, val_start, val_end)
    d = _remove_excluded_period(d)
    return d


# ==========================================================
# 13) SIMULACIÓN COMPLETA CON SALIDAS DETALLADAS
# ==========================================================
def simular_detallado(params, metric_start=None, metric_end=None):
    cn_table_local = reemplazar_cn_lulc10_16(
        params["CN10_g7"], params["CN16_g7"],
        params["CN10_g9"], params["CN16_g9"],
        g7=params["g7"], g9=params["g9"]
    )

    S1_CU, S2_CU, S3_CU = build_cn_s_from_cache(
        LULC_CUENCA, SOIL_CUENCA,
        cn_table_local=cn_table_local,
        g7=params["g7"], g9=params["g9"],
        f_cn2=1.0,
        f_s1=1.3, f_s2=1.2, f_s3=1.4
    )

    rows = []
    for year in range(START_YEAR, END_YEAR + 1):
        prev4 = None
        for month in range(1, 13):
            chirps_path = find_month_file(CHIRPS_DIR, "CHIRPS_PPT_DAILY", year, month)
            et_path     = find_month_file(ET_DIR, "ET_HARGREAVES_DAILY", year, month)

            ch_cu_stack, _ = read_clip_stack(chirps_path, roi_cuenca_geoms)
            ppt_cu = qm_correct_array(ch_cu_stack.astype("float64"), month, QM_TABLE)
            ppt_cu_eff = ppt_cu * float(params["k_effP"])

            amc5, prev4 = rolling_sum_5d(ppt_cu_eff, prev4=prev4)
            q_cu = runoff_scs(
                ppt_cu_eff, amc5, S1_CU, S2_CU, S3_CU,
                lambda_ia=float(params["lambda_ia"])
            )

            q_mean = [nanmean_masked(q_cu[i], None) for i in range(q_cu.shape[0])]
            q_mean = [v * float(params["k_q"]) for v in q_mean]

            ppt_cu_mean = [nanmean_masked(ppt_cu_eff[i], None) for i in range(ppt_cu_eff.shape[0])]

            ch_em_stack, ch_em_nodata = read_clip_stack(chirps_path, roi_embalse_geoms)
            ppt_em = qm_correct_array(ch_em_stack.astype("float64"), month, QM_TABLE)
            ppt_em_mean = [nanmean_masked(ppt_em[i], ch_em_nodata) for i in range(ppt_em.shape[0])]
            ppt_em_mean = [v * float(params["k_effP"]) for v in ppt_em_mean]

            et_em_stack, et_em_nodata = read_clip_stack(et_path, roi_embalse_geoms)
            et_em = et_em_stack.astype("float64")
            et_em_mean = [nanmean_masked(et_em[i], et_em_nodata) for i in range(et_em.shape[0])]

            ndays = calendar.monthrange(year, month)[1]
            T = min(ndays, len(q_mean), len(ppt_em_mean), len(et_em_mean), len(ppt_cu_mean))

            for d in range(T):
                rows.append({
                    "fecha": datetime(year, month, d + 1),
                    "ppt_cuenca_eff_mm": float(ppt_cu_mean[d]),
                    "esc_mm": float(q_mean[d]),
                    "precip_mm": float(ppt_em_mean[d]),
                    "et_mm": float(et_em_mean[d]),
                })

    df_forz = pd.DataFrame(rows).sort_values("fecha").reset_index(drop=True)
    df_forz["esc_ruteada_mm"] = route_nash_cascade(
        df_forz["esc_mm"].to_numpy(),
        n=int(params["n"]),
        K=float(params["K"])
    )

    df_forz["fecha"] = pd.to_datetime(df_forz["fecha"]).dt.floor("D")
    df_forz = df_forz.drop_duplicates(subset=["fecha"], keep="last")

    df_join = pd.merge(
        df_forz[["fecha", "ppt_cuenca_eff_mm", "esc_mm", "esc_ruteada_mm", "precip_mm", "et_mm"]],
        Vdescarga[["fecha", "descarga_m3_s", "V_emb_real_hm3"]],
        on="fecha",
        how="inner"
    ).sort_values("fecha").reset_index(drop=True)

    if len(df_join) < 100:
        df_res_full = pd.DataFrame({"fecha": [], "V_emb": [], "V_emb_real": []})
        met = {
            "R2": np.nan, "RMSE": np.nan, "NSE": np.nan,
            "RMSE_n": np.nan, "Bias": np.nan, "PBIAS": np.nan, "KGE": np.nan, "n": 0
        }
        return df_res_full, met

    resultados = []
    V_emb = None

    for i in range(len(df_join)):
        V_prev = V_INICIAL if i == 0 else V_emb

        fecha = df_join.loc[i, "fecha"]
        Vd = float(df_join.loc[i, "descarga_m3_s"]) * 86400.0
        V_obs = float(df_join.loc[i, "V_emb_real_hm3"]) * 1_000_000.0

        esc_mm = float(df_join.loc[i, "esc_ruteada_mm"])
        p_mm   = float(df_join.loc[i, "precip_mm"])
        et_mm  = float(df_join.loc[i, "et_mm"])

        V_emb, V_vert, A_i, V_clima, V_esc, Vs_vol, B_i, Vd = balance_embalse_diario(
            V_prev, esc_mm, p_mm, et_mm,
            AREA_CUENCA_TOTAL, Vd,
            Vs_m_d=float(params["Vs"]),
            k_infil=float(params["k_infil"]),
            k_et=float(params["k_et"])
        )

        resultados.append({
            "fecha": fecha,
            "ppt_cuenca_eff_mm": float(df_join.loc[i, "ppt_cuenca_eff_mm"]),
            "esc_mm_sin_ruteo": float(df_join.loc[i, "esc_mm"]),
            "esc_mm_ruteada": float(df_join.loc[i, "esc_ruteada_mm"]),
            "precip_mm_embalse": p_mm,
            "et_mm_embalse": et_mm,
            "descarga_m3_s": float(df_join.loc[i, "descarga_m3_s"]),
            "descarga_m3_dia": Vd,
            "V_emb": V_emb,
            "V_emb_hm3": V_emb / 1e6,
            "V_emb_real": V_obs,
            "V_emb_real_hm3": V_obs / 1e6,
            "V_vertido": V_vert,
            "area_embalse_m2": A_i,
            "V_clima": V_clima,
            "V_esc": V_esc,
            "Vs_vol": Vs_vol,
            "balance_bruto": B_i
        })

    df_res_full = pd.DataFrame(resultados)
    df_res_full["fecha"] = pd.to_datetime(df_res_full["fecha"]).dt.floor("D")

    if metric_start is None or metric_end is None:
        dmet = df_res_full.copy()
    else:
        dmet = _slice_df_by_dates(df_res_full, metric_start, metric_end)

    dmet = _remove_excluded_period(dmet)
    met = _compute_metrics(dmet["V_emb"].values, dmet["V_emb_real"].values)

    return df_res_full, met


# ==========================================================
# 14) GRAFICADO DE UN ESCENARIO
# ==========================================================
def graficar_escenario(df_full, met_cal, met_val, nombre_esc, descripcion, out_png):
    plt.figure(figsize=(16, 5))

    df_obs_plot = df_full.copy()
    exc_s = pd.to_datetime(EXC_START)
    exc_e = pd.to_datetime(EXC_END)
    mask_exc = (df_obs_plot["fecha"] >= exc_s) & (df_obs_plot["fecha"] <= exc_e)
    df_obs_plot.loc[mask_exc, "V_emb_real"] = np.nan

    sim = df_full["V_emb"].to_numpy(dtype=float)
    sim_low = sim * 0.85
    sim_high = sim * 1.15

    plt.fill_between(
        df_full["fecha"],
        sim_low,
        sim_high,
        alpha=0.18,
        label="Banda ±15% Sim"
    )

    plt.plot(df_obs_plot["fecha"], df_obs_plot["V_emb_real"], linewidth=1.5, label="Obs")
    plt.plot(df_full["fecha"], df_full["V_emb"], linewidth=1.8, label="Sim")

    plt.axvline(pd.to_datetime(CAL_START), linestyle="--", linewidth=1.2)
    plt.axvline(pd.to_datetime(CAL_END), linestyle="--", linewidth=1.2)
    plt.axvline(pd.to_datetime(VAL_START), linestyle="--", linewidth=1.2)

    texto_cal = (
        "CALIBRACIÓN\n"
        f"R²={met_cal['R2']:.3f}\n"
        f"RMSE={met_cal['RMSE']:.2e}\n"
        f"NSE={met_cal['NSE']:.3f}\n"
        f"KGE={met_cal['KGE']:.3f}\n"
        f"Bias={met_cal['Bias']:.4f}"
    )

    texto_val = (
        "VALIDACIÓN\n"
        f"R²={met_val['R2']:.3f}\n"
        f"RMSE={met_val['RMSE']:.2e}\n"
        f"NSE={met_val['NSE']:.3f}\n"
        f"KGE={met_val['KGE']:.3f}\n"
        f"Bias={met_val['Bias']:.4f}"
    )

    plt.text(
        0.015, 0.97, texto_cal,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.text(
        0.165, 0.97, texto_val,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.title(
        f"Volumen embalse 2010–2019 | {nombre_esc}\n"
        f"{descripcion}\n"
        f"Calibración: {CAL_START} a {CAL_END} | Validación: {VAL_START} a {VAL_END}"
    )
    plt.xlabel("Fecha")
    plt.ylabel("Volumen (m3)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.show()


# ==========================================================
# 15) EJECUCIÓN MULTI-ESCENARIO
# ==========================================================
print("=== MODELO MULTI-ESCENARIO ===")

resumen_global = []
params_global = []
series_all = []

for nombre_esc, params in ESCENARIOS.items():
    print("\n" + "=" * 80)
    print(f"Ejecutando {nombre_esc}")
    print(params["descripcion"])
    print("=" * 80)
    print(params)

    df_full, _ = simular_detallado(
        params,
        metric_start=CAL_START,
        metric_end=CAL_END
    )

    df_full["fecha"] = pd.to_datetime(df_full["fecha"]).dt.floor("D")

    df_cal = _build_calibration_df(df_full, CAL_START, CAL_END)
    df_val = _build_validation_df(df_full, VAL_START, VAL_END)

    met_cal = _compute_metrics(df_cal["V_emb"], df_cal["V_emb_real"])
    met_val = _compute_metrics(df_val["V_emb"], df_val["V_emb_real"])

    print("\n--- MÉTRICAS CALIBRACIÓN ---")
    print(met_cal)
    print("--- MÉTRICAS VALIDACIÓN ---")
    print(met_val)

    # Guardar serie individual CSV
    out_csv = os.path.join(OUT_DIR, f"serie_{nombre_esc}.csv")
    df_out = df_full.copy()
    df_out["escenario"] = nombre_esc
    df_out["descripcion"] = params["descripcion"]
    df_out.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # Gráfico individual
    out_png = os.path.join(OUT_DIR, f"grafica_{nombre_esc}.png")
    graficar_escenario(df_full, met_cal, met_val, nombre_esc, params["descripcion"], out_png)

    # Acumulados resumen
    resumen_global.append({
        "escenario": nombre_esc,
        "descripcion": params["descripcion"],
        "etapa": "CALIBRACION",
        **met_cal
    })
    resumen_global.append({
        "escenario": nombre_esc,
        "descripcion": params["descripcion"],
        "etapa": "VALIDACION",
        **met_val
    })

    for k, v in params.items():
        params_global.append({
            "escenario": nombre_esc,
            "descripcion": params["descripcion"],
            "parametro": k,
            "valor": v
        })

    df_long = df_full.copy()
    df_long["escenario"] = nombre_esc
    df_long["descripcion"] = params["descripcion"]
    series_all.append(df_long)


# ==========================================================
# 16) UNIFICAR RESULTADOS
# ==========================================================
df_resumen = pd.DataFrame(resumen_global)
df_params = pd.DataFrame(params_global)
df_series = pd.concat(series_all, ignore_index=True)

propiedades_modelo = pd.DataFrame([
    {"Propiedad": "VOL_MUERTO_m3", "Valor": VOL_MUERTO},
    {"Propiedad": "VOL_MAX_m3", "Valor": VOL_MAX},
    {"Propiedad": "AREA_CUENCA_TOTAL_m2", "Valor": AREA_CUENCA_TOTAL},
    {"Propiedad": "V_INICIAL_m3", "Valor": V_INICIAL},
    {"Propiedad": "CAL_START", "Valor": CAL_START},
    {"Propiedad": "CAL_END", "Valor": CAL_END},
    {"Propiedad": "VAL_START", "Valor": VAL_START},
    {"Propiedad": "VAL_END", "Valor": VAL_END},
    {"Propiedad": "SERIE_START", "Valor": SERIE_START},
    {"Propiedad": "SERIE_END", "Valor": SERIE_END},
    {"Propiedad": "EXC_START", "Valor": EXC_START},
    {"Propiedad": "EXC_END", "Valor": EXC_END},
])

# Resumen ancho por escenario
df_resumen_wide = (
    df_resumen
    .pivot_table(
        index=["escenario", "descripcion"],
        columns="etapa",
        values=["R2", "RMSE", "NSE", "KGE", "Bias", "PBIAS", "n"],
        aggfunc="first"
    )
)

df_resumen_wide.columns = [f"{a}_{b}" for a, b in df_resumen_wide.columns]
df_resumen_wide = df_resumen_wide.reset_index()

out_xlsx = os.path.join(OUT_DIR, "resultados_5_escenarios.xlsx")

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    df_resumen.to_excel(writer, sheet_name="metricas_largo", index=False)
    df_resumen_wide.to_excel(writer, sheet_name="metricas_resumen", index=False)
    df_params.to_excel(writer, sheet_name="parametros", index=False)
    propiedades_modelo.to_excel(writer, sheet_name="propiedades_modelo", index=False)
    df_series.to_excel(writer, sheet_name="series_completas", index=False)


# ==========================================================
# 17) GRÁFICA COMPARATIVA CONJUNTA
# ==========================================================
plt.figure(figsize=(18, 6))

# Observado único
df_obs = (
    df_series[["fecha", "V_emb_real"]]
    .drop_duplicates(subset=["fecha"])
    .sort_values("fecha")
    .reset_index(drop=True)
)

exc_s = pd.to_datetime(EXC_START)
exc_e = pd.to_datetime(EXC_END)
mask_exc = (df_obs["fecha"] >= exc_s) & (df_obs["fecha"] <= exc_e)
df_obs_plot = df_obs.copy()
df_obs_plot.loc[mask_exc, "V_emb_real"] = np.nan

plt.plot(df_obs_plot["fecha"], df_obs_plot["V_emb_real"], linewidth=2.2, label="Obs")

for nombre_esc in ESCENARIOS.keys():
    d = (
        df_series[df_series["escenario"] == nombre_esc]
        .sort_values("fecha")
        .reset_index(drop=True)
    )
    plt.plot(d["fecha"], d["V_emb"], linewidth=1.5, label=nombre_esc)

plt.axvline(pd.to_datetime(CAL_START), linestyle="--", linewidth=1.2)
plt.axvline(pd.to_datetime(CAL_END), linestyle="--", linewidth=1.2)
plt.axvline(pd.to_datetime(VAL_START), linestyle="--", linewidth=1.2)

plt.title(
    "Comparación de 5 escenarios físicamente representativos\n"
    "Volumen del embalse 2010–2019"
)
plt.xlabel("Fecha")
plt.ylabel("Volumen (m3)")
plt.legend(ncol=3)
plt.tight_layout()

out_png_comp = os.path.join(OUT_DIR, "comparacion_5_escenarios.png")
plt.savefig(out_png_comp, dpi=200, bbox_inches="tight")
plt.show()


# ==========================================================
# 18) TABLA FINAL EN CONSOLA
# ==========================================================
print("\n=== RESUMEN FINAL DE MÉTRICAS ===")
print(df_resumen_wide.to_string(index=False))

print("\n=== CORRESPONDENCIA DE ESCENARIOS ===")
for nombre_esc, params in ESCENARIOS.items():
    print(f"{nombre_esc} --> {params['descripcion']}")

print("\n=== ARCHIVOS GENERADOS ===")
print("Directorio principal :", OUT_DIR)
print("Excel resumen        :", out_xlsx)
print("Gráfico comparativo  :", out_png_comp)
print("CSV por escenario    : serie_ESC_*.csv")
print("PNG por escenario    : grafica_ESC_*.png")

### 5.1.2 · Versión sin exclusión de periodo (evalúa toda la serie)


In [ ]:
# ==========================================================
# SCRIPT 2 MODIFICADO
# SIMULACIÓN DISTRIBUIDA + EMBALSE
# MULTI-ESCENARIO (5 ESCENARIOS FÍSICAMENTE REPRESENTATIVOS)
# SERIE COMPLETA 2010-2019
# CALIBRACIÓN / VALIDACIÓN
# SIN EXCLUSIÓN DE NINGÚN PERIODO (TODA LA SERIE EVALUADA)
# + GUARDADO DE RESULTADOS
# + BANDA SOMBREADA ±15% DE LA SERIE SIMULADA
# + MÉTRICAS EN LA GRÁFICA
# + COMPARACIÓN ENTRE ESCENARIOS
# ==========================================================


# ==========================================================
# 0) CONFIGURACIÓN DE FECHAS
# ==========================================================
CAL_START = "2010-01-01"
CAL_END   = "2017-01-01"

VAL_START = "2017-01-02"
VAL_END   = "2019-12-31"

SERIE_START = "2010-01-01"
SERIE_END   = "2019-12-31"

EXC_START = "2014-01-01"
EXC_END   = "2015-06-07"


# ==========================================================
# 1) LIBRERÍAS
# ==========================================================
import os
import glob
import json
import calendar
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)


# ==========================================================
# 2) RUTAS / PARÁMETROS GENERALES
# ==========================================================
BASE_DIR = "/content/drive/MyDrive/Project001"

CHIRPS_DIR = os.path.join(BASE_DIR, "CHIRPS_PPT_DAILY_2009_2020_WGS84")
ET_DIR     = os.path.join(BASE_DIR, "ET_HARGREAVES_DAILY_2009_2020_WGS84")

LULC_DIR = os.path.join(BASE_DIR, "LULC_LC_Type1_WGS84")
SOIL_DIR = os.path.join(BASE_DIR, "SOIL_TEXTURE_b0_WGS84")

SHP_DIR = os.path.join(BASE_DIR, "shp")
AOI_CUENCA_PATH  = os.path.join(SHP_DIR, "Cuenca-es.geojson")
AOI_EMBALSE_PATH = os.path.join(SHP_DIR, "Embalse.geojson")

QM_MODEL_PATH = os.path.join(BASE_DIR, "QM_model_monthly.csv")
DESCARGAS_CSV = os.path.join(BASE_DIR, "descargas2.csv")

OUT_DIR = os.path.join(BASE_DIR, "resultados_modelo_5_escenarios")
os.makedirs(OUT_DIR, exist_ok=True)

START_YEAR = 2010
END_YEAR   = 2019

VOL_MUERTO = 41e6
VOL_MAX    = 148.80e6
AREA_CUENCA_TOTAL = 196.33e6
V_INICIAL = 57.80e6


# ==========================================================
# 3) ESCENARIOS SELECCIONADOS
# ==========================================================
# ==========================================================
# ESCENARIOS SELECCIONADOS — CUENCA EL PAÑE
# Basado en análisis de representatividad física del archivo
# de calibración genética (metricas.txt)
# ==========================================================

# ==========================================================
# 3) ESCENARIOS SELECCIONADOS
# ==========================================================
ESCENARIOS = {

    "ESC_05_G13_I6": {
        "descripcion": "GEN 13/20 - IND 6/42",
        "g7": 1,
        "g9": 2,
        "n": 4,
        "K": 4.313107873728896,
        "k_infil": 0.6953880721605649,
        "k_et": 0.8082241252469766,
        "Vs": 0.009594970137013482,
        "CN10_g7": 95.72124246470727,
        "CN16_g7": 85.71535662377057,
        "CN10_g9": 96.31605148608226,
        "CN16_g9": 66.28439128813265,
        "k_effP": 1.812739987216763,
        "k_q": 1.7089490774160854,
        "lambda_ia": 0.03914373443714243,
    }
}


# ==========================================================
# 4) UTILIDADES GENERALES
# ==========================================================
def load_geoms(geojson_path: str):
    if not os.path.exists(geojson_path):
        raise FileNotFoundError(f"No existe AOI: {geojson_path}")
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        raise ValueError(f"AOI vacío: {geojson_path}")
    geom = gdf.geometry.union_all()
    return [geom]


def read_clip_stack(tif_path: str, geoms):
    if not os.path.exists(tif_path):
        raise FileNotFoundError(f"No existe raster: {tif_path}")
    with rasterio.open(tif_path) as src:
        out_img, _ = mask(src, geoms, crop=True, filled=True)
        nodata = src.nodata
    return out_img, nodata


def nanmean_masked(a: np.ndarray, nodata=None) -> float:
    a = a.astype("float64", copy=False)
    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)
    return float(np.nanmean(a))


def find_month_file(folder: str, prefix: str, year: int, month: int) -> str:
    pattern = os.path.join(folder, f"{prefix}_{year}_{month:02d}_WGS84*.*")
    hits = sorted(glob.glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No encontré archivo con patrón:\n  {pattern}")
    return hits[0]


def find_single_tif(folder: str) -> str:
    hits = sorted(glob.glob(os.path.join(folder, "*.tif"))) + sorted(glob.glob(os.path.join(folder, "*.tiff")))
    if not hits:
        raise FileNotFoundError(f"No encontré tif/tiff en: {folder}")
    return hits[0]


# ==========================================================
# 5) QM MENSUAL
# ==========================================================
def load_qm_table(csv_path: str):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No existe QM CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"mes", "x", "y"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"QM_model_monthly.csv debe tener columnas {required}. Tiene: {list(df.columns)}")

    qm = {}
    for _, r in df.iterrows():
        mes = int(r["mes"])
        x = json.loads(r["x"]) if isinstance(r["x"], str) else r["x"]
        y = json.loads(r["y"]) if isinstance(r["y"], str) else r["y"]

        x = np.array(x, dtype=float)
        y = np.array(y, dtype=float)

        if x.size < 2 or y.size < 2 or x.size != y.size:
            raise ValueError(f"QM inválido en mes {mes}: x={x.size}, y={y.size}")

        qm[mes] = (x, y)

    faltantes = [m for m in range(1, 13) if m not in qm]
    if faltantes:
        raise ValueError(f"Faltan meses en QM: {faltantes}")

    return qm


def qm_correct_array(arr: np.ndarray, month: int, QM_TABLE):
    x, y = QM_TABLE[month]
    idx = np.argsort(x)
    x2 = x[idx]
    y2 = y[idx]
    flat = arr.reshape(-1).astype("float64", copy=False)
    out = np.interp(flat, x2, y2).reshape(arr.shape)
    return np.maximum(out, 0.0)


# ==========================================================
# 6) SCS-CN DISTRIBUIDO
# ==========================================================
CN_TABLE = {
    1: {1:35, 2:25, 3:45, 4:39, 5:45, 6:49, 7:68, 8:36, 9:45, 10:30, 11:95, 12:67, 13:72, 14:63, 15:100, 16:74, 17:100},
    2: {1:50, 2:55, 3:66, 4:61, 5:66, 6:69, 7:79, 8:60, 9:66, 10:58, 11:95, 12:78, 13:82, 14:75, 15:100, 16:84, 17:100},
    3: {1:73, 2:70, 3:77, 4:74, 5:77, 6:79, 7:86, 8:73, 9:77, 10:71, 11:95, 12:85, 13:87, 14:83, 15:100, 16:90, 17:100},
    4: {1:79, 2:77, 3:83, 4:80, 5:83, 6:89, 7:89, 8:79, 9:83, 10:78, 11:95, 12:89, 13:89, 14:87, 15:100, 16:92, 17:100},
}


def soil_group_from_texture(soil_class, g7=2, g9=2):
    soil_grp = np.zeros_like(soil_class, dtype="int16")
    soil_grp = np.where(soil_class > 10, 1, soil_grp)
    soil_grp = np.where((soil_class > 4) & (soil_class <= 10), 2, soil_grp)
    soil_grp = np.where((soil_class > 1) & (soil_class <= 4), 3, soil_grp)
    soil_grp = np.where((soil_class > 0) & (soil_class <= 1), 4, soil_grp)

    soil_grp = np.where(soil_class == 7, int(g7), soil_grp)
    soil_grp = np.where(soil_class == 9, int(g9), soil_grp)
    return soil_grp


def reemplazar_cn_lulc10_16(CN10_g7, CN16_g7, CN10_g9, CN16_g9, g7=2, g9=2):
    cn_table_local = {k: v.copy() for k, v in CN_TABLE.items()}
    cn_table_local[int(g7)][10] = float(CN10_g7)
    cn_table_local[int(g7)][16] = float(CN16_g7)
    cn_table_local[int(g9)][10] = float(CN10_g9)
    cn_table_local[int(g9)][16] = float(CN16_g9)
    return cn_table_local


def build_cn_s_from_cache(LULC_CUENCA, SOIL_CUENCA, cn_table_local, g7=2, g9=2, f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4):
    lulc = LULC_CUENCA.astype("float64")
    soil = SOIL_CUENCA.astype("float64")
    soil_grp = soil_group_from_texture(soil, g7=g7, g9=g9)

    CN2 = np.zeros_like(lulc, dtype="float64")
    CN2 = np.where(soil_grp == 0, 100.0, CN2)

    for grp, lut in cn_table_local.items():
        for lc, cn in lut.items():
            CN2 = np.where((soil_grp == grp) & (lulc == lc), float(cn), CN2)

    CN2 = np.minimum(CN2 * float(f_cn2), 100.0)

    CN1 = CN2 / (2.281 - (CN2 * 0.0128))
    CN3 = CN2 / (0.427 + (CN2 * 0.00573))

    S1 = (25400.0 / CN1 - 254.0) * float(f_s1)
    S2 = (25400.0 / CN2 - 254.0) * float(f_s2)
    S3 = (25400.0 / CN3 - 254.0) * float(f_s3)

    return S1, S2, S3


def rolling_sum_5d(stack_TRC: np.ndarray, prev4=None):
    if prev4 is None:
        prev4 = np.zeros((0,) + stack_TRC.shape[1:], dtype=stack_TRC.dtype)

    combo = np.concatenate([prev4, stack_TRC], axis=0)
    cs = np.cumsum(combo, axis=0)

    amc = np.empty_like(combo, dtype="float64")
    for i in range(combo.shape[0]):
        if i < 4:
            amc[i] = np.sum(combo[:i+1], axis=0)
        else:
            amc[i] = cs[i] - cs[i-5]

    tail4 = combo[-4:] if combo.shape[0] >= 4 else combo
    return amc[-stack_TRC.shape[0]:], tail4


def runoff_scs(ppt, amc5, S1, S2, S3, lambda_ia=0.2):
    S = np.where(amc5 <= 13, S1, S2)
    S = np.where(amc5 > 28, S3, S)

    lam = float(np.clip(lambda_ia, 0.001, 0.8))
    Ia = lam * S

    numer = np.power(ppt - Ia, 2)
    denom = (ppt - Ia) + S

    with np.errstate(divide="ignore", invalid="ignore"):
        Q = np.where(ppt < Ia, 0.0, numer / denom)

    return np.where(np.isfinite(Q), Q, 0.0)


# ==========================================================
# 7) RUTEO
# ==========================================================
def route_linear_reservoir(runoff_mm, K=2.0):
    r = np.asarray(runoff_mm, dtype=float)
    q = np.zeros_like(r)
    alpha = np.exp(-1.0 / max(K, 1e-6))

    for t in range(len(r)):
        q[t] = (1 - alpha) * r[t] if t == 0 else alpha * q[t-1] + (1 - alpha) * r[t]

    return q


def route_nash_cascade(runoff_mm, n=3, K=2.0):
    q = np.asarray(runoff_mm, dtype=float)
    for _ in range(int(n)):
        q = route_linear_reservoir(q, K=K)
    return q


# ==========================================================
# 8) EMBALSE
# ==========================================================
def area_embalse(V):
    D366 = V / 1e6
    area_km2 = (
        8.91946288801159E-08 * D366**4
        - 0.0000346583770904819 * D366**3
        + 0.003919249094071 * D366**2
        - 0.0306477700093546 * D366
        + 2.02308232795992
    )
    return max(area_km2, 0) * 1e6


def volumen_auxiliar(V_prev, esc_mm, A_c, Vd):
    A_emb = area_embalse(V_prev)
    A_aporte = max(A_c - A_emb, 0)
    esc_m = esc_mm / 1000.0
    V_esc = esc_m * A_aporte
    return V_prev + V_esc - Vd


def area_media(V_prev, V_aux):
    return 0.5 * (area_embalse(V_prev) + area_embalse(V_aux))


def aplicar_restricciones(B_i):
    if B_i > VOL_MAX:
        return VOL_MAX, B_i - VOL_MAX
    if B_i < VOL_MUERTO:
        return VOL_MUERTO, 0.0
    return B_i, 0.0


def balance_embalse_diario(V_prev, esc_mm, precip_mm, et_mm, A_c, Vd, Vs_m_d, k_infil=1.0, k_et=1.0):
    V_aux = volumen_auxiliar(V_prev, esc_mm, A_c, Vd)
    A_i = area_media(V_prev, V_aux)

    esc_m = esc_mm / 1000.0
    precip_m = precip_mm / 1000.0
    et_m = (et_mm * k_et) / 1000.0

    V_esc = esc_m * max(A_c - A_i, 0.0)
    V_clima = (precip_m - et_m) * A_i
    Vs_vol = (float(Vs_m_d) * float(k_infil)) * A_i

    B_i = V_prev + V_esc + V_clima - Vd - Vs_vol
    V_emb, V_vertido = aplicar_restricciones(B_i)

    return V_emb, V_vertido, A_i, V_clima, V_esc, Vs_vol, B_i, Vd


# ==========================================================
# 9) CARGA DE DATOS FIJOS
# ==========================================================
roi_cuenca_geoms  = load_geoms(AOI_CUENCA_PATH)
roi_embalse_geoms = load_geoms(AOI_EMBALSE_PATH)

QM_TABLE = load_qm_table(QM_MODEL_PATH)

LULC_TIF = find_single_tif(LULC_DIR)
SOIL_TIF = find_single_tif(SOIL_DIR)

Vdescarga = pd.read_csv(DESCARGAS_CSV, sep=";", encoding="latin-1")
Vdescarga = Vdescarga.rename(columns={
    Vdescarga.columns[0]: "fecha",
    Vdescarga.columns[1]: "descarga_m3_s",
    Vdescarga.columns[2]: "V_emb_real_hm3",
})

Vdescarga["fecha"] = pd.to_datetime(Vdescarga["fecha"], dayfirst=True, errors="coerce").dt.floor("D")
Vdescarga["descarga_m3_s"] = pd.to_numeric(Vdescarga["descarga_m3_s"], errors="coerce")
Vdescarga["V_emb_real_hm3"] = pd.to_numeric(Vdescarga["V_emb_real_hm3"], errors="coerce")

Vdescarga = Vdescarga[
    (Vdescarga["fecha"] >= SERIE_START) &
    (Vdescarga["fecha"] <= SERIE_END)
].dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)

Vdescarga = Vdescarga.drop_duplicates(subset=["fecha"], keep="last")


# ==========================================================
# 10) CACHE RASTERS ESTÁTICOS
# ==========================================================
lulc_stack, _ = read_clip_stack(LULC_TIF, roi_cuenca_geoms)
soil_stack, _ = read_clip_stack(SOIL_TIF, roi_cuenca_geoms)

LULC_CUENCA = lulc_stack[0].astype("int16")
SOIL_CUENCA = soil_stack[0].astype("int16")

print("LULC únicos (cuenca):", np.unique(LULC_CUENCA))
print("SOIL únicos (cuenca):", np.unique(SOIL_CUENCA))


# ==========================================================
# 11) MÉTRICAS
# ==========================================================
def _compute_metrics(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    m = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[m]
    obs = obs[m]

    if len(obs) < 10:
        return {
            "n": int(len(obs)),
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan
        }

    if np.std(obs) > 0 and np.std(sim) > 0:
        r = float(np.corrcoef(obs, sim)[0, 1])
        R2 = float(r ** 2)
    else:
        r = 0.0
        R2 = 0.0

    RMSE = float(np.sqrt(np.mean((sim - obs) ** 2)))
    denom = float(np.sum((obs - np.mean(obs)) ** 2))
    NSE = float(1 - np.sum((sim - obs) ** 2) / denom) if denom > 0 else np.nan

    sigma = float(np.std(obs))
    RMSE_n = float(RMSE / sigma) if sigma > 0 else np.nan

    mu_o = float(np.mean(obs))
    mu_s = float(np.mean(sim))
    Bias = float((mu_s - mu_o) / mu_o) if abs(mu_o) > 1e-12 else np.nan
    PBIAS = float(Bias * 100.0) if np.isfinite(Bias) else np.nan

    alpha = float(np.std(sim) / np.std(obs)) if np.std(obs) > 0 else np.nan
    beta = float(mu_s / mu_o) if abs(mu_o) > 1e-12 else np.nan

    if np.isfinite(r) and np.isfinite(alpha) and np.isfinite(beta):
        KGE = float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))
    else:
        KGE = np.nan

    return {
        "n": int(len(obs)),
        "R2": R2,
        "RMSE": RMSE,
        "NSE": NSE,
        "RMSE_n": RMSE_n,
        "Bias": Bias,
        "PBIAS": PBIAS,
        "KGE": KGE
    }


# ==========================================================
# 12) FILTROS DE FECHAS
# ==========================================================
def _slice_df_by_dates(df_res, start, end):
    s = pd.to_datetime(start)
    e = pd.to_datetime(end)

    d = df_res.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.floor("D")

    return d[(d["fecha"] >= s) & (d["fecha"] <= e)].reset_index(drop=True)


def _build_calibration_df(df_res, cal_start, cal_end):
    # Toda la serie de calibración SIN excluir ningún periodo
    d = _slice_df_by_dates(df_res, cal_start, cal_end)
    return d


def _build_validation_df(df_res, val_start, val_end):
    # Toda la serie de validación SIN excluir ningún periodo
    d = _slice_df_by_dates(df_res, val_start, val_end)
    return d


# ==========================================================
# 13) SIMULACIÓN COMPLETA CON SALIDAS DETALLADAS
# ==========================================================
def simular_detallado(params, metric_start=None, metric_end=None):
    cn_table_local = reemplazar_cn_lulc10_16(
        params["CN10_g7"], params["CN16_g7"],
        params["CN10_g9"], params["CN16_g9"],
        g7=params["g7"], g9=params["g9"]
    )

    S1_CU, S2_CU, S3_CU = build_cn_s_from_cache(
        LULC_CUENCA, SOIL_CUENCA,
        cn_table_local=cn_table_local,
        g7=params["g7"], g9=params["g9"],
        f_cn2=1.0,
        f_s1=1.3, f_s2=1.2, f_s3=1.4
    )

    rows = []
    for year in range(START_YEAR, END_YEAR + 1):
        prev4 = None
        for month in range(1, 13):
            chirps_path = find_month_file(CHIRPS_DIR, "CHIRPS_PPT_DAILY", year, month)
            et_path     = find_month_file(ET_DIR, "ET_HARGREAVES_DAILY", year, month)

            ch_cu_stack, _ = read_clip_stack(chirps_path, roi_cuenca_geoms)
            ppt_cu = qm_correct_array(ch_cu_stack.astype("float64"), month, QM_TABLE)
            ppt_cu_eff = ppt_cu * float(params["k_effP"])

            amc5, prev4 = rolling_sum_5d(ppt_cu_eff, prev4=prev4)
            q_cu = runoff_scs(
                ppt_cu_eff, amc5, S1_CU, S2_CU, S3_CU,
                lambda_ia=float(params["lambda_ia"])
            )

            q_mean = [nanmean_masked(q_cu[i], None) for i in range(q_cu.shape[0])]
            q_mean = [v * float(params["k_q"]) for v in q_mean]

            ppt_cu_mean = [nanmean_masked(ppt_cu_eff[i], None) for i in range(ppt_cu_eff.shape[0])]

            ch_em_stack, ch_em_nodata = read_clip_stack(chirps_path, roi_embalse_geoms)
            ppt_em = qm_correct_array(ch_em_stack.astype("float64"), month, QM_TABLE)
            ppt_em_mean = [nanmean_masked(ppt_em[i], ch_em_nodata) for i in range(ppt_em.shape[0])]
            ppt_em_mean = [v * float(params["k_effP"]) for v in ppt_em_mean]

            et_em_stack, et_em_nodata = read_clip_stack(et_path, roi_embalse_geoms)
            et_em = et_em_stack.astype("float64")
            et_em_mean = [nanmean_masked(et_em[i], et_em_nodata) for i in range(et_em.shape[0])]

            ndays = calendar.monthrange(year, month)[1]
            T = min(ndays, len(q_mean), len(ppt_em_mean), len(et_em_mean), len(ppt_cu_mean))

            for d in range(T):
                rows.append({
                    "fecha": datetime(year, month, d + 1),
                    "ppt_cuenca_eff_mm": float(ppt_cu_mean[d]),
                    "esc_mm": float(q_mean[d]),
                    "precip_mm": float(ppt_em_mean[d]),
                    "et_mm": float(et_em_mean[d]),
                })

    df_forz = pd.DataFrame(rows).sort_values("fecha").reset_index(drop=True)
    df_forz["esc_ruteada_mm"] = route_nash_cascade(
        df_forz["esc_mm"].to_numpy(),
        n=int(params["n"]),
        K=float(params["K"])
    )

    df_forz["fecha"] = pd.to_datetime(df_forz["fecha"]).dt.floor("D")
    df_forz = df_forz.drop_duplicates(subset=["fecha"], keep="last")

    df_join = pd.merge(
        df_forz[["fecha", "ppt_cuenca_eff_mm", "esc_mm", "esc_ruteada_mm", "precip_mm", "et_mm"]],
        Vdescarga[["fecha", "descarga_m3_s", "V_emb_real_hm3"]],
        on="fecha",
        how="inner"
    ).sort_values("fecha").reset_index(drop=True)

    if len(df_join) < 100:
        df_res_full = pd.DataFrame({"fecha": [], "V_emb": [], "V_emb_real": []})
        met = {
            "R2": np.nan, "RMSE": np.nan, "NSE": np.nan,
            "RMSE_n": np.nan, "Bias": np.nan, "PBIAS": np.nan, "KGE": np.nan, "n": 0
        }
        return df_res_full, met

    resultados = []
    V_emb = None

    for i in range(len(df_join)):
        V_prev = V_INICIAL if i == 0 else V_emb

        fecha = df_join.loc[i, "fecha"]
        Vd = float(df_join.loc[i, "descarga_m3_s"]) * 86400.0
        V_obs = float(df_join.loc[i, "V_emb_real_hm3"]) * 1_000_000.0

        esc_mm = float(df_join.loc[i, "esc_ruteada_mm"])
        p_mm   = float(df_join.loc[i, "precip_mm"])
        et_mm  = float(df_join.loc[i, "et_mm"])

        V_emb, V_vert, A_i, V_clima, V_esc, Vs_vol, B_i, Vd = balance_embalse_diario(
            V_prev, esc_mm, p_mm, et_mm,
            AREA_CUENCA_TOTAL, Vd,
            Vs_m_d=float(params["Vs"]),
            k_infil=float(params["k_infil"]),
            k_et=float(params["k_et"])
        )

        resultados.append({
            "fecha": fecha,
            "ppt_cuenca_eff_mm": float(df_join.loc[i, "ppt_cuenca_eff_mm"]),
            "esc_mm_sin_ruteo": float(df_join.loc[i, "esc_mm"]),
            "esc_mm_ruteada": float(df_join.loc[i, "esc_ruteada_mm"]),
            "precip_mm_embalse": p_mm,
            "et_mm_embalse": et_mm,
            "descarga_m3_s": float(df_join.loc[i, "descarga_m3_s"]),
            "descarga_m3_dia": Vd,
            "V_emb": V_emb,
            "V_emb_hm3": V_emb / 1e6,
            "V_emb_real": V_obs,
            "V_emb_real_hm3": V_obs / 1e6,
            "V_vertido": V_vert,
            "area_embalse_m2": A_i,
            "V_clima": V_clima,
            "V_esc": V_esc,
            "Vs_vol": Vs_vol,
            "balance_bruto": B_i
        })

    df_res_full = pd.DataFrame(resultados)
    df_res_full["fecha"] = pd.to_datetime(df_res_full["fecha"]).dt.floor("D")

    # Métricas sobre TODA la serie del periodo solicitado, sin excluir nada
    if metric_start is None or metric_end is None:
        dmet = df_res_full.copy()
    else:
        dmet = _slice_df_by_dates(df_res_full, metric_start, metric_end)

    met = _compute_metrics(dmet["V_emb"].values, dmet["V_emb_real"].values)

    return df_res_full, met


# ==========================================================
# 14) GRAFICADO DE UN ESCENARIO
# ==========================================================
def graficar_escenario(df_full, met_cal, met_val, nombre_esc, descripcion, out_png):
    plt.figure(figsize=(16, 5))

    sim = df_full["V_emb"].to_numpy(dtype=float)
    sim_low = sim * 0.85
    sim_high = sim * 1.15

    plt.fill_between(
        df_full["fecha"],
        sim_low,
        sim_high,
        alpha=0.18,
        label="Banda ±15% Sim"
    )

    # Serie observada completa, sin enmascarar ningún periodo
    plt.plot(df_full["fecha"], df_full["V_emb_real"], linewidth=1.5, label="Obs")
    plt.plot(df_full["fecha"], df_full["V_emb"], linewidth=1.8, label="Sim")

    plt.axvline(pd.to_datetime(CAL_START), linestyle="--", linewidth=1.2)
    plt.axvline(pd.to_datetime(CAL_END), linestyle="--", linewidth=1.2)
    plt.axvline(pd.to_datetime(VAL_START), linestyle="--", linewidth=1.2)

    texto_cal = (
        "CALIBRACIÓN\n"
        f"R²={met_cal['R2']:.3f}\n"
        f"RMSE={met_cal['RMSE']:.2e}\n"
        f"NSE={met_cal['NSE']:.3f}\n"
        f"KGE={met_cal['KGE']:.3f}\n"
        f"Bias={met_cal['Bias']:.4f}"
    )

    texto_val = (
        "VALIDACIÓN\n"
        f"R²={met_val['R2']:.3f}\n"
        f"RMSE={met_val['RMSE']:.2e}\n"
        f"NSE={met_val['NSE']:.3f}\n"
        f"KGE={met_val['KGE']:.3f}\n"
        f"Bias={met_val['Bias']:.4f}"
    )

    plt.text(
        0.015, 0.97, texto_cal,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.text(
        0.165, 0.97, texto_val,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.title(
        f"Volumen embalse 2010–2019 | {nombre_esc}\n"
        f"{descripcion}\n"
        f"Calibración: {CAL_START} a {CAL_END} | Validación: {VAL_START} a {VAL_END}"
    )
    plt.xlabel("Fecha")
    plt.ylabel("Volumen (m3)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.show()


# ==========================================================
# 15) EJECUCIÓN MULTI-ESCENARIO
# ==========================================================
print("=== MODELO MULTI-ESCENARIO ===")

resumen_global = []
params_global = []
series_all = []

for nombre_esc, params in ESCENARIOS.items():
    print("\n" + "=" * 80)
    print(f"Ejecutando {nombre_esc}")
    print(params["descripcion"])
    print("=" * 80)
    print(params)

    df_full, _ = simular_detallado(
        params,
        metric_start=CAL_START,
        metric_end=CAL_END
    )

    df_full["fecha"] = pd.to_datetime(df_full["fecha"]).dt.floor("D")

    df_cal = _build_calibration_df(df_full, CAL_START, CAL_END)
    df_val = _build_validation_df(df_full, VAL_START, VAL_END)

    met_cal = _compute_metrics(df_cal["V_emb"], df_cal["V_emb_real"])
    met_val = _compute_metrics(df_val["V_emb"], df_val["V_emb_real"])

    print("\n--- MÉTRICAS CALIBRACIÓN ---")
    print(met_cal)
    print("--- MÉTRICAS VALIDACIÓN ---")
    print(met_val)

    # Guardar serie individual CSV
    out_csv = os.path.join(OUT_DIR, f"serie_{nombre_esc}.csv")
    df_out = df_full.copy()
    df_out["escenario"] = nombre_esc
    df_out["descripcion"] = params["descripcion"]
    df_out.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # Gráfico individual
    out_png = os.path.join(OUT_DIR, f"grafica_{nombre_esc}.png")
    graficar_escenario(df_full, met_cal, met_val, nombre_esc, params["descripcion"], out_png)

    # Acumulados resumen
    resumen_global.append({
        "escenario": nombre_esc,
        "descripcion": params["descripcion"],
        "etapa": "CALIBRACION",
        **met_cal
    })
    resumen_global.append({
        "escenario": nombre_esc,
        "descripcion": params["descripcion"],
        "etapa": "VALIDACION",
        **met_val
    })

    for k, v in params.items():
        params_global.append({
            "escenario": nombre_esc,
            "descripcion": params["descripcion"],
            "parametro": k,
            "valor": v
        })

    df_long = df_full.copy()
    df_long["escenario"] = nombre_esc
    df_long["descripcion"] = params["descripcion"]
    series_all.append(df_long)


# ==========================================================
# 16) UNIFICAR RESULTADOS
# ==========================================================
df_resumen = pd.DataFrame(resumen_global)
df_params = pd.DataFrame(params_global)
df_series = pd.concat(series_all, ignore_index=True)

propiedades_modelo = pd.DataFrame([
    {"Propiedad": "VOL_MUERTO_m3", "Valor": VOL_MUERTO},
    {"Propiedad": "VOL_MAX_m3", "Valor": VOL_MAX},
    {"Propiedad": "AREA_CUENCA_TOTAL_m2", "Valor": AREA_CUENCA_TOTAL},
    {"Propiedad": "V_INICIAL_m3", "Valor": V_INICIAL},
    {"Propiedad": "CAL_START", "Valor": CAL_START},
    {"Propiedad": "CAL_END", "Valor": CAL_END},
    {"Propiedad": "VAL_START", "Valor": VAL_START},
    {"Propiedad": "VAL_END", "Valor": VAL_END},
    {"Propiedad": "SERIE_START", "Valor": SERIE_START},
    {"Propiedad": "SERIE_END", "Valor": SERIE_END},
    {"Propiedad": "EXC_START", "Valor": EXC_START},
    {"Propiedad": "EXC_END", "Valor": EXC_END},
])

# Resumen ancho por escenario
df_resumen_wide = (
    df_resumen
    .pivot_table(
        index=["escenario", "descripcion"],
        columns="etapa",
        values=["R2", "RMSE", "NSE", "KGE", "Bias", "PBIAS", "n"],
        aggfunc="first"
    )
)

df_resumen_wide.columns = [f"{a}_{b}" for a, b in df_resumen_wide.columns]
df_resumen_wide = df_resumen_wide.reset_index()

out_xlsx = os.path.join(OUT_DIR, "resultados_5_escenarios.xlsx")

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    df_resumen.to_excel(writer, sheet_name="metricas_largo", index=False)
    df_resumen_wide.to_excel(writer, sheet_name="metricas_resumen", index=False)
    df_params.to_excel(writer, sheet_name="parametros", index=False)
    propiedades_modelo.to_excel(writer, sheet_name="propiedades_modelo", index=False)
    df_series.to_excel(writer, sheet_name="series_completas", index=False)


# ==========================================================
# 17) GRÁFICA COMPARATIVA CONJUNTA
# ==========================================================
plt.figure(figsize=(18, 6))

# Observado único — serie completa sin enmascarar
df_obs = (
    df_series[["fecha", "V_emb_real"]]
    .drop_duplicates(subset=["fecha"])
    .sort_values("fecha")
    .reset_index(drop=True)
)

plt.plot(df_obs["fecha"], df_obs["V_emb_real"], linewidth=2.2, label="Obs")

for nombre_esc in ESCENARIOS.keys():
    d = (
        df_series[df_series["escenario"] == nombre_esc]
        .sort_values("fecha")
        .reset_index(drop=True)
    )
    plt.plot(d["fecha"], d["V_emb"], linewidth=1.5, label=nombre_esc)

plt.axvline(pd.to_datetime(CAL_START), linestyle="--", linewidth=1.2)
plt.axvline(pd.to_datetime(CAL_END), linestyle="--", linewidth=1.2)
plt.axvline(pd.to_datetime(VAL_START), linestyle="--", linewidth=1.2)

plt.title(
    "Comparación de 5 escenarios físicamente representativos\n"
    "Volumen del embalse 2010–2019"
)
plt.xlabel("Fecha")
plt.ylabel("Volumen (m3)")
plt.legend(ncol=3)
plt.tight_layout()

out_png_comp = os.path.join(OUT_DIR, "comparacion_5_escenarios.png")
plt.savefig(out_png_comp, dpi=200, bbox_inches="tight")
plt.show()


# ==========================================================
# 18) TABLA FINAL EN CONSOLA
# ==========================================================
print("\n=== RESUMEN FINAL DE MÉTRICAS ===")
print(df_resumen_wide.to_string(index=False))

print("\n=== CORRESPONDENCIA DE ESCENARIOS ===")
for nombre_esc, params in ESCENARIOS.items():
    print(f"{nombre_esc} --> {params['descripcion']}")

print("\n=== ARCHIVOS GENERADOS ===")
print("Directorio principal :", OUT_DIR)
print("Excel resumen        :", out_xlsx)
print("Gráfico comparativo  :", out_png_comp)
print("CSV por escenario    : serie_ESC_*.csv")
print("PNG por escenario    : grafica_ESC_*.png")

## 5.2 · El Pañe Futuro — modelo + embalse (escritura sólo Excel)


In [ ]:
# ==========================================================
# EL PAÑE FUTURO
# MODELO HIDROLÓGICO + EMBALSE
# ----------------------------------------------------------
# COMBINACIONES:
#   modelos  : MPI_ESM1_2_LR, EC_Earth3_Veg_LR, EC_Earth3
#   ssp      : ssp245, ssp585
#   umbral P : solo u0p5
#   descarga : ESC_00, ESC_25, ESC_50, ESC_75, ESC_100
#
# ENTRADAS:
#   - Precipitación corregida QDM (u0p5)
#   - ET corregido
#   - LULC + SOIL + shapefiles
#   - QM_model_monthly.csv
#   - Descargas proyectadas
#
# SALIDAS:
#   - SOLO ARCHIVOS EXCEL
#   - Un Excel por combinación
#   - Un Excel resumen global
#
# REGLA DE CORRECCIÓN DE VOLUMEN:
#   - 2019-12-31 -> 56.00 Hm3 útil
#   - 2025-12-31 -> 36.37 Hm3 útil
#   - Se convierte a volumen total del modelo:
#         V_total = VOL_MUERTO + V_util * 1e6
#   - Luego la simulación continúa normalmente
# ==========================================================

# ==========================================================
# 0) LIBRERÍAS
# ==========================================================
import os
import re
import json
import glob
import calendar
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)


# ==========================================================
# 1) CONFIGURACIÓN GENERAL
# ==========================================================
BASE_DIR = Path("/content/drive/MyDrive/Project001")

# -------------------------
# Carpetas estáticas / base
# -------------------------
LULC_DIR = BASE_DIR / "LULC_LC_Type1_WGS84"
SOIL_DIR = BASE_DIR / "SOIL_TEXTURE_b0_WGS84"
SHP_DIR  = BASE_DIR / "shp"

AOI_CUENCA_PATH  = SHP_DIR / "Cuenca-es.geojson"
AOI_EMBALSE_PATH = SHP_DIR / "Embalse.geojson"

QM_MODEL_PATH = BASE_DIR / "QM_model_monthly.csv"

# -------------------------
# Carpetas futuras corregidas
# -------------------------
CLIMA_DIR = BASE_DIR / "DATOS_CLIMATICOS_TOTALES"
PPT_CORR_DIR = CLIMA_DIR / "PRECIPITACION_CORREGIDA_QDM"
ET_CORR_DIR  = CLIMA_DIR / "ET_HARGREAVES_CORREGIDA_QDM"

# -------------------------
# Descargas proyectadas
# -------------------------
DESC_PROY_DIR = BASE_DIR / "proyeccion_descargas_ciclo_3anios_promedio_no_ceros_2020_2100"

# -------------------------
# Carpeta de salida
# SOLO EXCEL
# -------------------------
OUT_DIR = BASE_DIR / "resultados_pane_futuro_u0p5_excel"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Modelos / escenarios
# -------------------------
MODELOS = [
    "MPI_ESM1_2_LR",
    "EC_Earth3_Veg_LR",
    "EC_Earth3"
]

SSP_LIST = ["ssp245", "ssp585"]
ESCENARIOS_DESCARGA = ["ESC_00", "ESC_25", "ESC_50", "ESC_75", "ESC_100"]

UMBRAL_PRECIP = "u0p5"

# ==========================================================
# 2) PARÁMETROS DEL MODELO HIDROLÓGICO
# Basado en tu escenario ESC_05_G13_I6
# ==========================================================
PARAMS = {
    "descripcion": "GEN 13/20 - IND 6/42",
    "g7": 1,
    "g9": 2,
    "n": 4,
    "K": 4.313107873728896,
    "k_infil": 0.6953880721605649,
    "k_et": 0.8082241252469766,
    "Vs": 0.009594970137013482,
    "CN10_g7": 95.72124246470727,
    "CN16_g7": 85.71535662377057,
    "CN10_g9": 96.31605148608226,
    "CN16_g9": 66.28439128813265,
    "k_effP": 1.812739987216763,
    "k_q": 1.7089490774160854,
    "lambda_ia": 0.03914373443714243,
}

# ==========================================================
# 3) PARÁMETROS DEL EMBALSE
# ==========================================================
VOL_MUERTO = 41e6
VOL_MAX    = 148.80e6
AREA_CUENCA_TOTAL = 196.33e6
V_INICIAL = 57.80e6

# ==========================================================
# 4) FECHAS DE CORRECCIÓN DEL VOLUMEN
# Volumen útil observado -> convertido a volumen total del modelo
# ==========================================================
CORRECCIONES_UTIL_HM3 = {
    pd.Timestamp("2019-12-31"): 56.00,
    pd.Timestamp("2025-12-31"): 36.37,
}

# ==========================================================
# 5) UTILIDADES DE CONSOLA
# ==========================================================
def linea(char="─", n=80):
    print(char * n)

def header(txt, char="═"):
    linea(char)
    print(f"  {txt}")
    linea(char)

def ok(txt):
    print(f"  ✔ {txt}")

def warn(txt):
    print(f"  ⚠ {txt}")

def err(txt):
    print(f"  ✘ {txt}")

def print_paths_for_run(modelo, ssp, esc_desc, ppt_dir, et_dir, desc_csv):
    header(f"USANDO ENTRADAS | {modelo} | {ssp} | {esc_desc}", char="·")
    print(f"  Precipitación corregida : {ppt_dir}")
    print(f"  ET corregido            : {et_dir}")
    print(f"  Descarga proyectada     : {desc_csv}")
    print(f"  QM mensual              : {QM_MODEL_PATH}")
    print(f"  LULC                    : {LULC_DIR}")
    print(f"  SOIL                    : {SOIL_DIR}")
    print(f"  Cuenca                  : {AOI_CUENCA_PATH}")
    print(f"  Embalse                 : {AOI_EMBALSE_PATH}")
    linea("·")

# ==========================================================
# 6) UTILIDADES GENERALES
# ==========================================================
def load_geoms(geojson_path: Path):
    if not geojson_path.exists():
        raise FileNotFoundError(f"No existe AOI: {geojson_path}")
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        raise ValueError(f"AOI vacío: {geojson_path}")
    geom = gdf.geometry.union_all()
    return [geom]

def read_clip_stack(tif_path: Path, geoms):
    if not tif_path.exists():
        raise FileNotFoundError(f"No existe raster: {tif_path}")
    with rasterio.open(tif_path) as src:
        out_img, _ = mask(src, geoms, crop=True, filled=True)
        nodata = src.nodata
    return out_img, nodata

def nanmean_masked(a: np.ndarray, nodata=None) -> float:
    a = a.astype("float64", copy=False)
    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)
    return float(np.nanmean(a))

def find_single_tif(folder: Path) -> Path:
    hits = sorted(folder.glob("*.tif")) + sorted(folder.glob("*.tiff"))
    if not hits:
        raise FileNotFoundError(f"No encontré tif/tiff en: {folder}")
    return hits[0]

def load_qm_table(csv_path: Path):
    if not csv_path.exists():
        raise FileNotFoundError(f"No existe QM CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"mes", "x", "y"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"QM_model_monthly.csv debe tener columnas {required}. Tiene: {list(df.columns)}")

    qm = {}
    for _, r in df.iterrows():
        mes = int(r["mes"])
        x = json.loads(r["x"]) if isinstance(r["x"], str) else r["x"]
        y = json.loads(r["y"]) if isinstance(r["y"], str) else r["y"]

        x = np.array(x, dtype=float)
        y = np.array(y, dtype=float)

        if x.size < 2 or y.size < 2 or x.size != y.size:
            raise ValueError(f"QM inválido en mes {mes}: x={x.size}, y={y.size}")

        qm[mes] = (x, y)

    faltantes = [m for m in range(1, 13) if m not in qm]
    if faltantes:
        raise ValueError(f"Faltan meses en QM: {faltantes}")

    return qm

def qm_correct_array(arr: np.ndarray, month: int, QM_TABLE):
    x, y = QM_TABLE[month]
    idx = np.argsort(x)
    x2 = x[idx]
    y2 = y[idx]
    flat = arr.reshape(-1).astype("float64", copy=False)
    out = np.interp(flat, x2, y2).reshape(arr.shape)
    return np.maximum(out, 0.0)

# ==========================================================
# 7) PARSEO DE NOMBRES DE ARCHIVOS FUTUROS
# ==========================================================
def extraer_anio_mes_desde_nombre(nombre: str):
    m = re.search(r"_(\d{4})_(\d{2})(?:_|\.|$)", nombre)
    if not m:
        return None, None
    anio = int(m.group(1))
    mes = int(m.group(2))
    return anio, mes

def listar_tifs_por_anio_mes(folder: Path, suffix=".tif"):
    d = {}
    for p in sorted(folder.glob(f"*{suffix}")):
        anio, mes = extraer_anio_mes_desde_nombre(p.name)
        if anio is not None and mes is not None:
            d[(anio, mes)] = p
    return d

# ==========================================================
# 8) TABLAS CN
# ==========================================================
CN_TABLE = {
    1: {1:35, 2:25, 3:45, 4:39, 5:45, 6:49, 7:68, 8:36, 9:45, 10:30, 11:95, 12:67, 13:72, 14:63, 15:100, 16:74, 17:100},
    2: {1:50, 2:55, 3:66, 4:61, 5:66, 6:69, 7:79, 8:60, 9:66, 10:58, 11:95, 12:78, 13:82, 14:75, 15:100, 16:84, 17:100},
    3: {1:73, 2:70, 3:77, 4:74, 5:77, 6:79, 7:86, 8:73, 9:77, 10:71, 11:95, 12:85, 13:87, 14:83, 15:100, 16:90, 17:100},
    4: {1:79, 2:77, 3:83, 4:80, 5:83, 6:89, 7:89, 8:79, 9:83, 10:78, 11:95, 12:89, 13:89, 14:87, 15:100, 16:92, 17:100},
}

def soil_group_from_texture(soil_class, g7=2, g9=2):
    soil_grp = np.zeros_like(soil_class, dtype="int16")
    soil_grp = np.where(soil_class > 10, 1, soil_grp)
    soil_grp = np.where((soil_class > 4) & (soil_class <= 10), 2, soil_grp)
    soil_grp = np.where((soil_class > 1) & (soil_class <= 4), 3, soil_grp)
    soil_grp = np.where((soil_class > 0) & (soil_class <= 1), 4, soil_grp)

    soil_grp = np.where(soil_class == 7, int(g7), soil_grp)
    soil_grp = np.where(soil_class == 9, int(g9), soil_grp)
    return soil_grp

def reemplazar_cn_lulc10_16(params):
    cn_table_local = {k: v.copy() for k, v in CN_TABLE.items()}
    cn_table_local[int(params["g7"])][10] = float(params["CN10_g7"])
    cn_table_local[int(params["g7"])][16] = float(params["CN16_g7"])
    cn_table_local[int(params["g9"])][10] = float(params["CN10_g9"])
    cn_table_local[int(params["g9"])][16] = float(params["CN16_g9"])
    return cn_table_local

def build_cn_s_from_cache(LULC_CUENCA, SOIL_CUENCA, cn_table_local, params, f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4):
    lulc = LULC_CUENCA.astype("float64")
    soil = SOIL_CUENCA.astype("float64")
    soil_grp = soil_group_from_texture(soil, g7=params["g7"], g9=params["g9"])

    CN2 = np.zeros_like(lulc, dtype="float64")
    CN2 = np.where(soil_grp == 0, 100.0, CN2)

    for grp, lut in cn_table_local.items():
        for lc, cn in lut.items():
            CN2 = np.where((soil_grp == grp) & (lulc == lc), float(cn), CN2)

    CN2 = np.minimum(CN2 * float(f_cn2), 100.0)

    CN1 = CN2 / (2.281 - (CN2 * 0.0128))
    CN3 = CN2 / (0.427 + (CN2 * 0.00573))

    S1 = (25400.0 / CN1 - 254.0) * float(f_s1)
    S2 = (25400.0 / CN2 - 254.0) * float(f_s2)
    S3 = (25400.0 / CN3 - 254.0) * float(f_s3)

    return S1, S2, S3

# ==========================================================
# 9) ESCORRENTÍA Y RUTEO
# ==========================================================
def rolling_sum_5d(stack_TRC: np.ndarray, prev4=None):
    if prev4 is None:
        prev4 = np.zeros((0,) + stack_TRC.shape[1:], dtype=stack_TRC.dtype)

    combo = np.concatenate([prev4, stack_TRC], axis=0)
    cs = np.cumsum(combo, axis=0)

    amc = np.empty_like(combo, dtype="float64")
    for i in range(combo.shape[0]):
        if i < 4:
            amc[i] = np.sum(combo[:i+1], axis=0)
        else:
            amc[i] = cs[i] - cs[i-5]

    tail4 = combo[-4:] if combo.shape[0] >= 4 else combo
    return amc[-stack_TRC.shape[0]:], tail4

def runoff_scs(ppt, amc5, S1, S2, S3, lambda_ia=0.2):
    S = np.where(amc5 <= 13, S1, S2)
    S = np.where(amc5 > 28, S3, S)

    lam = float(np.clip(lambda_ia, 0.001, 0.8))
    Ia = lam * S

    numer = np.power(ppt - Ia, 2)
    denom = (ppt - Ia) + S

    with np.errstate(divide="ignore", invalid="ignore"):
        Q = np.where(ppt < Ia, 0.0, numer / denom)

    return np.where(np.isfinite(Q), Q, 0.0)

def route_linear_reservoir(runoff_mm, K=2.0):
    r = np.asarray(runoff_mm, dtype=float)
    q = np.zeros_like(r)
    alpha = np.exp(-1.0 / max(K, 1e-6))

    for t in range(len(r)):
        q[t] = (1 - alpha) * r[t] if t == 0 else alpha * q[t-1] + (1 - alpha) * r[t]
    return q

def route_nash_cascade(runoff_mm, n=3, K=2.0):
    q = np.asarray(runoff_mm, dtype=float)
    for _ in range(int(n)):
        q = route_linear_reservoir(q, K=K)
    return q

# ==========================================================
# 10) EMBALSE
# ==========================================================
def area_embalse(V):
    D366 = V / 1e6
    area_km2 = (
        8.91946288801159E-08 * D366**4
        - 0.0000346583770904819 * D366**3
        + 0.003919249094071 * D366**2
        - 0.0306477700093546 * D366
        + 2.02308232795992
    )
    return max(area_km2, 0) * 1e6

def volumen_auxiliar(V_prev, esc_mm, A_c, Vd):
    A_emb = area_embalse(V_prev)
    A_aporte = max(A_c - A_emb, 0)
    esc_m = esc_mm / 1000.0
    V_esc = esc_m * A_aporte
    return V_prev + V_esc - Vd

def area_media(V_prev, V_aux):
    return 0.5 * (area_embalse(V_prev) + area_embalse(V_aux))

def aplicar_restricciones(B_i):
    if B_i > VOL_MAX:
        return VOL_MAX, B_i - VOL_MAX
    if B_i < VOL_MUERTO:
        return VOL_MUERTO, 0.0
    return B_i, 0.0

def balance_embalse_diario(V_prev, esc_mm, precip_mm, et_mm, A_c, Vd, Vs_m_d, k_infil=1.0, k_et=1.0):
    V_aux = volumen_auxiliar(V_prev, esc_mm, A_c, Vd)
    A_i = area_media(V_prev, V_aux)

    esc_m = esc_mm / 1000.0
    precip_m = precip_mm / 1000.0
    et_m = (et_mm * k_et) / 1000.0

    V_esc = esc_m * max(A_c - A_i, 0.0)
    V_clima = (precip_m - et_m) * A_i
    Vs_vol = (float(Vs_m_d) * float(k_infil)) * A_i

    B_i = V_prev + V_esc + V_clima - Vd - Vs_vol
    V_emb, V_vertido = aplicar_restricciones(B_i)

    return V_emb, V_vertido, A_i, V_clima, V_esc, Vs_vol, B_i, Vd

def volumen_util_a_total_m3(v_util_hm3):
    """
    Convierte volumen útil (Hm3) del reporte a volumen total del modelo (m3).
    Suposición:
        V_total = VOL_MUERTO + V_util
    """
    return VOL_MUERTO + (float(v_util_hm3) * 1_000_000.0)

# ==========================================================
# 11) MÉTRICAS
# ==========================================================
def _compute_metrics(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    m = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[m]
    obs = obs[m]

    if len(obs) < 10:
        return {
            "n": int(len(obs)),
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan
        }

    if np.std(obs) > 0 and np.std(sim) > 0:
        r = float(np.corrcoef(obs, sim)[0, 1])
        R2 = float(r ** 2)
    else:
        r = 0.0
        R2 = 0.0

    RMSE = float(np.sqrt(np.mean((sim - obs) ** 2)))
    denom = float(np.sum((obs - np.mean(obs)) ** 2))
    NSE = float(1 - np.sum((sim - obs) ** 2) / denom) if denom > 0 else np.nan

    sigma = float(np.std(obs))
    RMSE_n = float(RMSE / sigma) if sigma > 0 else np.nan

    mu_o = float(np.mean(obs))
    mu_s = float(np.mean(sim))
    Bias = float((mu_s - mu_o) / mu_o) if abs(mu_o) > 1e-12 else np.nan
    PBIAS = float(Bias * 100.0) if np.isfinite(Bias) else np.nan

    alpha = float(np.std(sim) / np.std(obs)) if np.std(obs) > 0 else np.nan
    beta = float(mu_s / mu_o) if abs(mu_o) > 1e-12 else np.nan

    if np.isfinite(r) and np.isfinite(alpha) and np.isfinite(beta):
        KGE = float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))
    else:
        KGE = np.nan

    return {
        "n": int(len(obs)),
        "R2": R2,
        "RMSE": RMSE,
        "NSE": NSE,
        "RMSE_n": RMSE_n,
        "Bias": Bias,
        "PBIAS": PBIAS,
        "KGE": KGE
    }

# ==========================================================
# 12) LECTURA DE DESCARGAS PROYECTADAS
# ==========================================================
def cargar_descarga_proyectada(csv_path: Path):
    if not csv_path.exists():
        raise FileNotFoundError(f"No existe descarga proyectada: {csv_path}")

    df = pd.read_csv(csv_path)
    cols_required = {"fecha", "descarga_proyectada_m3_s"}
    if not cols_required.issubset(df.columns):
        raise ValueError(f"{csv_path.name} debe tener columnas {cols_required}")

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce").dt.floor("D")
    df["descarga_proyectada_m3_s"] = pd.to_numeric(df["descarga_proyectada_m3_s"], errors="coerce")
    df = df.dropna(subset=["fecha", "descarga_proyectada_m3_s"]).sort_values("fecha").reset_index(drop=True)
    return df

# ==========================================================
# 13) CARGA DE DATOS FIJOS
# ==========================================================
header("CARGANDO DATOS FIJOS")

roi_cuenca_geoms  = load_geoms(AOI_CUENCA_PATH)
roi_embalse_geoms = load_geoms(AOI_EMBALSE_PATH)

QM_TABLE = load_qm_table(QM_MODEL_PATH)

LULC_TIF = find_single_tif(LULC_DIR)
SOIL_TIF = find_single_tif(SOIL_DIR)

ok(f"LULC: {LULC_TIF}")
ok(f"SOIL: {SOIL_TIF}")

lulc_stack, _ = read_clip_stack(LULC_TIF, roi_cuenca_geoms)
soil_stack, _ = read_clip_stack(SOIL_TIF, roi_cuenca_geoms)

LULC_CUENCA = lulc_stack[0].astype("int16")
SOIL_CUENCA = soil_stack[0].astype("int16")

print("LULC únicos (cuenca):", np.unique(LULC_CUENCA))
print("SOIL únicos (cuenca):", np.unique(SOIL_CUENCA))

cn_table_local = reemplazar_cn_lulc10_16(PARAMS)
S1_CU, S2_CU, S3_CU = build_cn_s_from_cache(
    LULC_CUENCA, SOIL_CUENCA,
    cn_table_local=cn_table_local,
    params=PARAMS,
    f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4
)

# ==========================================================
# 14) FUNCIÓN PRINCIPAL DE SIMULACIÓN FUTURA
# ==========================================================
def simular_futuro_combinacion(modelo: str, ssp: str, esc_desc: str):
    """
    Ejecuta la simulación para una combinación:
      modelo + ssp + escenario de descarga
    usando:
      - P corregida u0p5
      - ET corregido
      - QM mensual
      - parámetros fijos del modelo hidrológico
    """
    ppt_folder = PPT_CORR_DIR / f"{modelo}_{ssp}_{UMBRAL_PRECIP}"
    et_folder  = ET_CORR_DIR  / f"{modelo}_{ssp}"
    desc_csv   = DESC_PROY_DIR / f"proyeccion_{esc_desc}.csv"

    if not ppt_folder.exists():
        raise FileNotFoundError(f"No existe carpeta de precipitación: {ppt_folder}")
    if not et_folder.exists():
        raise FileNotFoundError(f"No existe carpeta de ET: {et_folder}")
    if not desc_csv.exists():
        raise FileNotFoundError(f"No existe CSV de descarga: {desc_csv}")

    print_paths_for_run(modelo, ssp, esc_desc, ppt_folder, et_folder, desc_csv)

    # Listar TIFF mensuales
    ppt_files = listar_tifs_por_anio_mes(ppt_folder)
    et_files  = listar_tifs_por_anio_mes(et_folder)

    claves = sorted(set(ppt_files.keys()) & set(et_files.keys()))
    if not claves:
        raise RuntimeError(f"No hay meses comunes entre precipitación y ET para {modelo}_{ssp}")

    # Leer descarga proyectada
    df_desc = cargar_descarga_proyectada(desc_csv)

    rows = []
    prev4 = None

    barra_meses = tqdm(claves, desc=f"Simulando {modelo} | {ssp} | {esc_desc}", unit="mes", colour="green")

    for (year, month) in barra_meses:
        ppt_path = ppt_files[(year, month)]
        et_path  = et_files[(year, month)]

        barra_meses.set_postfix(fecha=f"{year}-{month:02d}")

        # Leer precipitación cuenca
        ppt_cu_stack, _ = read_clip_stack(ppt_path, roi_cuenca_geoms)
        ppt_cu = qm_correct_array(ppt_cu_stack.astype("float64"), month, QM_TABLE)
        ppt_cu_eff = ppt_cu * float(PARAMS["k_effP"])

        # AMC + escorrentía cuenca
        amc5, prev4 = rolling_sum_5d(ppt_cu_eff, prev4=prev4)
        q_cu = runoff_scs(
            ppt_cu_eff, amc5, S1_CU, S2_CU, S3_CU,
            lambda_ia=float(PARAMS["lambda_ia"])
        )

        q_mean = [nanmean_masked(q_cu[i], None) for i in range(q_cu.shape[0])]
        q_mean = [v * float(PARAMS["k_q"]) for v in q_mean]

        ppt_cu_mean = [nanmean_masked(ppt_cu_eff[i], None) for i in range(ppt_cu_eff.shape[0])]

        # Leer precipitación embalse desde el mismo raster de precipitación corregida
        ppt_em_stack, ppt_em_nodata = read_clip_stack(ppt_path, roi_embalse_geoms)
        ppt_em = qm_correct_array(ppt_em_stack.astype("float64"), month, QM_TABLE)
        ppt_em_mean = [nanmean_masked(ppt_em[i], ppt_em_nodata) for i in range(ppt_em.shape[0])]
        ppt_em_mean = [v * float(PARAMS["k_effP"]) for v in ppt_em_mean]

        # Leer ET embalse corregido
        et_em_stack, et_em_nodata = read_clip_stack(et_path, roi_embalse_geoms)
        et_em = et_em_stack.astype("float64")
        et_em_mean = [nanmean_masked(et_em[i], et_em_nodata) for i in range(et_em.shape[0])]

        ndays = calendar.monthrange(year, month)[1]
        T = min(ndays, len(q_mean), len(ppt_em_mean), len(et_em_mean), len(ppt_cu_mean))

        for d in range(T):
            rows.append({
                "fecha": datetime(year, month, d + 1),
                "ppt_cuenca_eff_mm": float(ppt_cu_mean[d]),
                "esc_mm": float(q_mean[d]),
                "precip_mm": float(ppt_em_mean[d]),
                "et_mm": float(et_em_mean[d]),
            })

    barra_meses.close()

    # Forzantes
    df_forz = pd.DataFrame(rows).sort_values("fecha").reset_index(drop=True)
    df_forz["esc_ruteada_mm"] = route_nash_cascade(
        df_forz["esc_mm"].to_numpy(),
        n=int(PARAMS["n"]),
        K=float(PARAMS["K"])
    )

    df_forz["fecha"] = pd.to_datetime(df_forz["fecha"]).dt.floor("D")
    df_forz = df_forz.drop_duplicates(subset=["fecha"], keep="last")

    # Cruce con descarga proyectada
    df_join = pd.merge(
        df_forz[["fecha", "ppt_cuenca_eff_mm", "esc_mm", "esc_ruteada_mm", "precip_mm", "et_mm"]],
        df_desc[["fecha", "descarga_proyectada_m3_s"]],
        on="fecha",
        how="inner"
    ).sort_values("fecha").reset_index(drop=True)

    if len(df_join) < 100:
        raise RuntimeError(f"Serie resultante demasiado corta para {modelo}_{ssp}_{esc_desc}")

    # Simulación embalse
    resultados = []
    V_emb = None

    for i in range(len(df_join)):
        fecha = pd.to_datetime(df_join.loc[i, "fecha"]).floor("D")
        V_prev = V_INICIAL if i == 0 else V_emb

        Vd = float(df_join.loc[i, "descarga_proyectada_m3_s"]) * 86400.0
        esc_mm = float(df_join.loc[i, "esc_ruteada_mm"])
        p_mm   = float(df_join.loc[i, "precip_mm"])
        et_mm  = float(df_join.loc[i, "et_mm"])

        V_emb, V_vert, A_i, V_clima, V_esc, Vs_vol, B_i, Vd = balance_embalse_diario(
            V_prev, esc_mm, p_mm, et_mm,
            AREA_CUENCA_TOTAL, Vd,
            Vs_m_d=float(PARAMS["Vs"]),
            k_infil=float(PARAMS["k_infil"]),
            k_et=float(PARAMS["k_et"])
        )

        # --------------------------------------------------
        # Corrección puntual del volumen en fechas clave
        # --------------------------------------------------
        if fecha in CORRECCIONES_UTIL_HM3:
            V_emb = volumen_util_a_total_m3(CORRECCIONES_UTIL_HM3[fecha])
            V_emb = max(VOL_MUERTO, min(V_emb, VOL_MAX))

        resultados.append({
            "fecha": fecha,
            "modelo": modelo,
            "ssp": ssp,
            "umbral_precip": UMBRAL_PRECIP,
            "escenario_descarga": esc_desc,
            "ppt_cuenca_eff_mm": float(df_join.loc[i, "ppt_cuenca_eff_mm"]),
            "esc_mm_sin_ruteo": float(df_join.loc[i, "esc_mm"]),
            "esc_mm_ruteada": float(df_join.loc[i, "esc_ruteada_mm"]),
            "precip_mm_embalse": p_mm,
            "et_mm_embalse": et_mm,
            "descarga_proyectada_m3_s": float(df_join.loc[i, "descarga_proyectada_m3_s"]),
            "descarga_proyectada_m3_dia": Vd,
            "V_emb_m3": V_emb,
            "V_emb_hm3": V_emb / 1e6,
            "V_emb_util_hm3": max((V_emb - VOL_MUERTO) / 1e6, 0.0),
            "V_vertido_m3": V_vert,
            "area_embalse_m2": A_i,
            "V_clima_m3": V_clima,
            "V_esc_m3": V_esc,
            "Vs_vol_m3": Vs_vol,
            "balance_bruto_m3": B_i,
            "corregido_volumen": int(fecha in CORRECCIONES_UTIL_HM3),
            "vol_util_objetivo_hm3": CORRECCIONES_UTIL_HM3.get(fecha, np.nan)
        })

    df_res = pd.DataFrame(resultados)

    # Métricas simples contra la corrección puntual útil
    df_ctrl = df_res[df_res["corregido_volumen"] == 1].copy()
    if len(df_ctrl) >= 1:
        met_ctrl = _compute_metrics(df_ctrl["V_emb_util_hm3"], df_ctrl["vol_util_objetivo_hm3"])
    else:
        met_ctrl = {k: np.nan for k in ["n", "R2", "RMSE", "NSE", "RMSE_n", "Bias", "PBIAS", "KGE"]}

    return df_res, met_ctrl, ppt_folder, et_folder, desc_csv

# ==========================================================
# 15) EJECUCIÓN MULTICOMBINACIÓN
# ==========================================================
header("SIMULACIÓN FUTURA EL PAÑE | u0p5")

combinaciones = []
for modelo in MODELOS:
    for ssp in SSP_LIST:
        for esc_desc in ESCENARIOS_DESCARGA:
            combinaciones.append((modelo, ssp, esc_desc))

resumen_global = []

barra_total = tqdm(combinaciones, desc="Combinaciones totales", unit="combo", colour="blue")

for modelo, ssp, esc_desc in barra_total:
    barra_total.set_postfix(modelo=modelo, ssp=ssp, esc=esc_desc)

    nombre_combo = f"{modelo}_{ssp}_{UMBRAL_PRECIP}_{esc_desc}"
    out_xlsx = OUT_DIR / f"{nombre_combo}.xlsx"

    try:
        df_res, met_ctrl, ppt_folder, et_folder, desc_csv = simular_futuro_combinacion(modelo, ssp, esc_desc)

        # Hojas auxiliares
        df_paths = pd.DataFrame([
            {"entrada": "ppt_corregida", "ruta": str(ppt_folder)},
            {"entrada": "et_corregido", "ruta": str(et_folder)},
            {"entrada": "descarga_proyectada", "ruta": str(desc_csv)},
            {"entrada": "qm_model", "ruta": str(QM_MODEL_PATH)},
            {"entrada": "lulc", "ruta": str(LULC_TIF)},
            {"entrada": "soil", "ruta": str(SOIL_TIF)},
            {"entrada": "cuenca", "ruta": str(AOI_CUENCA_PATH)},
            {"entrada": "embalse", "ruta": str(AOI_EMBALSE_PATH)},
        ])

        df_params = pd.DataFrame([
            {"parametro": k, "valor": v} for k, v in PARAMS.items()
        ])

        df_props = pd.DataFrame([
            {"propiedad": "VOL_MUERTO_m3", "valor": VOL_MUERTO},
            {"propiedad": "VOL_MAX_m3", "valor": VOL_MAX},
            {"propiedad": "AREA_CUENCA_TOTAL_m2", "valor": AREA_CUENCA_TOTAL},
            {"propiedad": "V_INICIAL_m3", "valor": V_INICIAL},
            {"propiedad": "UMBRAL_PRECIP", "valor": UMBRAL_PRECIP},
            {"propiedad": "correccion_2019_12_31_util_hm3", "valor": 56.00},
            {"propiedad": "correccion_2025_12_31_util_hm3", "valor": 36.37},
        ])

        df_metricas = pd.DataFrame([{
            "modelo": modelo,
            "ssp": ssp,
            "umbral_precip": UMBRAL_PRECIP,
            "escenario_descarga": esc_desc,
            **met_ctrl
        }])

        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            df_metricas.to_excel(writer, sheet_name="metricas_control", index=False)
            df_props.to_excel(writer, sheet_name="propiedades_modelo", index=False)
            df_params.to_excel(writer, sheet_name="parametros_hidrologicos", index=False)
            df_paths.to_excel(writer, sheet_name="rutas_usadas", index=False)
            df_res.to_excel(writer, sheet_name="serie_simulada", index=False)

        resumen_global.append({
            "modelo": modelo,
            "ssp": ssp,
            "umbral_precip": UMBRAL_PRECIP,
            "escenario_descarga": esc_desc,
            "excel": str(out_xlsx),
            **met_ctrl
        })

        ok(f"Excel generado: {out_xlsx.name}")

    except Exception as e:
        err(f"{nombre_combo}: {e}")
        resumen_global.append({
            "modelo": modelo,
            "ssp": ssp,
            "umbral_precip": UMBRAL_PRECIP,
            "escenario_descarga": esc_desc,
            "excel": str(out_xlsx),
            "n": np.nan,
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan,
            "error": str(e),
        })

barra_total.close()

# ==========================================================
# 16) RESUMEN GLOBAL
# ==========================================================
df_resumen_global = pd.DataFrame(resumen_global)
out_resumen = OUT_DIR / "resumen_global_u0p5.xlsx"

with pd.ExcelWriter(out_resumen, engine="openpyxl") as writer:
    df_resumen_global.to_excel(writer, sheet_name="resumen_global", index=False)

header("PROCESO COMPLETADO", char="█")
print(f"  Carpeta de salida : {OUT_DIR}")
print(f"  Excel resumen     : {out_resumen}")
print(f"  Total combinaciones: {len(combinaciones)}")

## 5.3 · El Pañe Futuro — versión con bloques de 10 años + Excel final


In [ ]:
# ==========================================================
# EL PAÑE FUTURO
# MODELO HIDROLÓGICO + EMBALSE
# ----------------------------------------------------------
# COMBINACIONES:
#   modelos  : MPI_ESM1_2_LR, EC_Earth3_Veg_LR, EC_Earth3
#   ssp      : ssp245, ssp585
#   umbral P : solo u0p5
#   descarga : ESC_00, ESC_25, ESC_50, ESC_75, ESC_100
#
# ENTRADAS:
#   - Precipitación corregida QDM (u0p5)
#   - ET corregido
#   - LULC + SOIL + shapefiles
#   - QM_model_monthly.csv
#   - Descargas proyectadas
#
# SALIDAS:
#   - TXT por bloques de 10 años
#   - Un Excel final por combinación
#   - Un Excel resumen global
#
# REANUDACIÓN:
#   - Si el proceso se corta, al reiniciar:
#       * detecta los bloques ya procesados
#       * borra los 2 últimos bloques
#       * los vuelve a procesar
#
# REGLA DE CORRECCIÓN DE VOLUMEN:
#   - 2019-12-31 -> 56.00 Hm3 útil
#   - 2025-12-31 -> 36.37 Hm3 útil
#   - Se convierte a volumen total del modelo:
#         V_total = VOL_MUERTO + V_util * 1e6
#   - Luego la simulación continúa normalmente
# ==========================================================

# ==========================================================
# 0) LIBRERÍAS
# ==========================================================
import os
import re
import json
import calendar
import gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)


# ==========================================================
# 1) CONFIGURACIÓN GENERAL
# ==========================================================
BASE_DIR = Path("/content/drive/MyDrive/Project001")

# -------------------------
# Carpetas estáticas / base
# -------------------------
LULC_DIR = BASE_DIR / "LULC_LC_Type1_WGS84"
SOIL_DIR = BASE_DIR / "SOIL_TEXTURE_b0_WGS84"
SHP_DIR  = BASE_DIR / "shp"

AOI_CUENCA_PATH  = SHP_DIR / "Cuenca-es.geojson"
AOI_EMBALSE_PATH = SHP_DIR / "Embalse.geojson"

QM_MODEL_PATH = BASE_DIR / "QM_model_monthly.csv"

# -------------------------
# Carpetas futuras corregidas
# -------------------------
CLIMA_DIR = BASE_DIR / "DATOS_CLIMATICOS_TOTALES"
PPT_CORR_DIR = CLIMA_DIR / "PRECIPITACION_CORREGIDA_QDM"
ET_CORR_DIR  = CLIMA_DIR / "ET_HARGREAVES_CORREGIDA_QDM"

# -------------------------
# Descargas proyectadas
# -------------------------
DESC_PROY_DIR = BASE_DIR / "proyeccion_descargas_ciclo_3anios_promedio_no_ceros_2020_2100"

# -------------------------
# Carpeta de salida
# -------------------------
OUT_DIR = BASE_DIR / "resultados_pane_futuro_u0p5_excel"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Temporales por bloques
# -------------------------
BLOCK_YEARS = 10
REPROCESS_LAST_N_BLOCKS = 2

TMP_DIR = OUT_DIR / "_tmp_bloques_txt"
TMP_DIR.mkdir(parents=True, exist_ok=True)

RESUMEN_GLOBAL_PARCIAL_TXT = OUT_DIR / "resumen_global_u0p5_parcial.txt"

# -------------------------
# Modelos / escenarios
# -------------------------
MODELOS = [
    "MPI_ESM1_2_LR",
    "EC_Earth3_Veg_LR",
    "EC_Earth3"
]

SSP_LIST = ["ssp245", "ssp585"]
ESCENARIOS_DESCARGA = ["ESC_00", "ESC_25", "ESC_50", "ESC_75", "ESC_100"]

UMBRAL_PRECIP = "u0p5"


# ==========================================================
# 2) PARÁMETROS DEL MODELO HIDROLÓGICO
# Basado en tu escenario ESC_05_G13_I6
# ==========================================================
PARAMS = {
    "descripcion": "GEN 13/20 - IND 6/42",
    "g7": 1,
    "g9": 2,
    "n": 4,
    "K": 4.313107873728896,
    "k_infil": 0.6953880721605649,
    "k_et": 0.8082241252469766,
    "Vs": 0.009594970137013482,
    "CN10_g7": 95.72124246470727,
    "CN16_g7": 85.71535662377057,
    "CN10_g9": 96.31605148608226,
    "CN16_g9": 66.28439128813265,
    "k_effP": 1.812739987216763,
    "k_q": 1.7089490774160854,
    "lambda_ia": 0.03914373443714243,
}


# ==========================================================
# 3) PARÁMETROS DEL EMBALSE
# ==========================================================
VOL_MUERTO = 41e6
VOL_MAX    = 148.80e6
AREA_CUENCA_TOTAL = 196.33e6
V_INICIAL = 57.80e6


# ==========================================================
# 4) FECHAS DE CORRECCIÓN DEL VOLUMEN
# ==========================================================
CORRECCIONES_UTIL_HM3 = {
    pd.Timestamp("2019-12-31"): 56.00,
    pd.Timestamp("2025-12-31"): 36.37,
}


# ==========================================================
# 5) UTILIDADES DE CONSOLA
# ==========================================================
def linea(char="─", n=80):
    print(char * n)

def header(txt, char="═"):
    linea(char)
    print(f"  {txt}")
    linea(char)

def ok(txt):
    print(f"  ✔ {txt}")

def warn(txt):
    print(f"  ⚠ {txt}")

def err(txt):
    print(f"  ✘ {txt}")

def print_paths_for_run(modelo, ssp, esc_desc, ppt_dir, et_dir, desc_csv):
    header(f"USANDO ENTRADAS | {modelo} | {ssp} | {esc_desc}", char="·")
    print(f"  Precipitación corregida : {ppt_dir}")
    print(f"  ET corregido            : {et_dir}")
    print(f"  Descarga proyectada     : {desc_csv}")
    print(f"  QM mensual              : {QM_MODEL_PATH}")
    print(f"  LULC                    : {LULC_DIR}")
    print(f"  SOIL                    : {SOIL_DIR}")
    print(f"  Cuenca                  : {AOI_CUENCA_PATH}")
    print(f"  Embalse                 : {AOI_EMBALSE_PATH}")
    print(f"  Temporales TXT          : {TMP_DIR}")
    linea("·")


# ==========================================================
# 6) UTILIDADES GENERALES
# ==========================================================
def load_geoms(geojson_path: Path):
    if not geojson_path.exists():
        raise FileNotFoundError(f"No existe AOI: {geojson_path}")
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        raise ValueError(f"AOI vacío: {geojson_path}")
    geom = gdf.geometry.union_all()
    return [geom]

def read_clip_stack(tif_path: Path, geoms):
    if not tif_path.exists():
        raise FileNotFoundError(f"No existe raster: {tif_path}")
    with rasterio.open(tif_path) as src:
        out_img, _ = mask(src, geoms, crop=True, filled=True)
        nodata = src.nodata
    return out_img, nodata

def nanmean_masked(a: np.ndarray, nodata=None) -> float:
    a = a.astype("float64", copy=False)
    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)
    return float(np.nanmean(a))

def find_single_tif(folder: Path) -> Path:
    hits = sorted(folder.glob("*.tif")) + sorted(folder.glob("*.tiff"))
    if not hits:
        raise FileNotFoundError(f"No encontré tif/tiff en: {folder}")
    return hits[0]

def load_qm_table(csv_path: Path):
    if not csv_path.exists():
        raise FileNotFoundError(f"No existe QM CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"mes", "x", "y"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"QM_model_monthly.csv debe tener columnas {required}. Tiene: {list(df.columns)}")

    qm = {}
    for _, r in df.iterrows():
        mes = int(r["mes"])
        x = json.loads(r["x"]) if isinstance(r["x"], str) else r["x"]
        y = json.loads(r["y"]) if isinstance(r["y"], str) else r["y"]

        x = np.array(x, dtype=float)
        y = np.array(y, dtype=float)

        if x.size < 2 or y.size < 2 or x.size != y.size:
            raise ValueError(f"QM inválido en mes {mes}: x={x.size}, y={y.size}")

        qm[mes] = (x, y)

    faltantes = [m for m in range(1, 13) if m not in qm]
    if faltantes:
        raise ValueError(f"Faltan meses en QM: {faltantes}")

    return qm

def qm_correct_array(arr: np.ndarray, month: int, QM_TABLE):
    x, y = QM_TABLE[month]
    idx = np.argsort(x)
    x2 = x[idx]
    y2 = y[idx]
    flat = arr.reshape(-1).astype("float64", copy=False)
    out = np.interp(flat, x2, y2).reshape(arr.shape)
    return np.maximum(out, 0.0)


# ==========================================================
# 7) PARSEO DE NOMBRES DE ARCHIVOS FUTUROS
# ==========================================================
def extraer_anio_mes_desde_nombre(nombre: str):
    m = re.search(r"_(\d{4})_(\d{2})(?:_|\.|$)", nombre)
    if not m:
        return None, None
    anio = int(m.group(1))
    mes = int(m.group(2))
    return anio, mes

def listar_tifs_por_anio_mes(folder: Path, suffix=".tif"):
    d = {}
    for p in sorted(folder.glob(f"*{suffix}")):
        anio, mes = extraer_anio_mes_desde_nombre(p.name)
        if anio is not None and mes is not None:
            d[(anio, mes)] = p
    return d


# ==========================================================
# 8) TABLAS CN
# ==========================================================
CN_TABLE = {
    1: {1:35, 2:25, 3:45, 4:39, 5:45, 6:49, 7:68, 8:36, 9:45, 10:30, 11:95, 12:67, 13:72, 14:63, 15:100, 16:74, 17:100},
    2: {1:50, 2:55, 3:66, 4:61, 5:66, 6:69, 7:79, 8:60, 9:66, 10:58, 11:95, 12:78, 13:82, 14:75, 15:100, 16:84, 17:100},
    3: {1:73, 2:70, 3:77, 4:74, 5:77, 6:79, 7:86, 8:73, 9:77, 10:71, 11:95, 12:85, 13:87, 14:83, 15:100, 16:90, 17:100},
    4: {1:79, 2:77, 3:83, 4:80, 5:83, 6:89, 7:89, 8:79, 9:83, 10:78, 11:95, 12:89, 13:89, 14:87, 15:100, 16:92, 17:100},
}

def soil_group_from_texture(soil_class, g7=2, g9=2):
    soil_grp = np.zeros_like(soil_class, dtype="int16")
    soil_grp = np.where(soil_class > 10, 1, soil_grp)
    soil_grp = np.where((soil_class > 4) & (soil_class <= 10), 2, soil_grp)
    soil_grp = np.where((soil_class > 1) & (soil_class <= 4), 3, soil_grp)
    soil_grp = np.where((soil_class > 0) & (soil_class <= 1), 4, soil_grp)

    soil_grp = np.where(soil_class == 7, int(g7), soil_grp)
    soil_grp = np.where(soil_class == 9, int(g9), soil_grp)
    return soil_grp

def reemplazar_cn_lulc10_16(params):
    cn_table_local = {k: v.copy() for k, v in CN_TABLE.items()}
    cn_table_local[int(params["g7"])][10] = float(params["CN10_g7"])
    cn_table_local[int(params["g7"])][16] = float(params["CN16_g7"])
    cn_table_local[int(params["g9"])][10] = float(params["CN10_g9"])
    cn_table_local[int(params["g9"])][16] = float(params["CN16_g9"])
    return cn_table_local

def build_cn_s_from_cache(LULC_CUENCA, SOIL_CUENCA, cn_table_local, params, f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4):
    lulc = LULC_CUENCA.astype("float64")
    soil = SOIL_CUENCA.astype("float64")
    soil_grp = soil_group_from_texture(soil, g7=params["g7"], g9=params["g9"])

    CN2 = np.zeros_like(lulc, dtype="float64")
    CN2 = np.where(soil_grp == 0, 100.0, CN2)

    for grp, lut in cn_table_local.items():
        for lc, cn in lut.items():
            CN2 = np.where((soil_grp == grp) & (lulc == lc), float(cn), CN2)

    CN2 = np.minimum(CN2 * float(f_cn2), 100.0)

    CN1 = CN2 / (2.281 - (CN2 * 0.0128))
    CN3 = CN2 / (0.427 + (CN2 * 0.00573))

    S1 = (25400.0 / CN1 - 254.0) * float(f_s1)
    S2 = (25400.0 / CN2 - 254.0) * float(f_s2)
    S3 = (25400.0 / CN3 - 254.0) * float(f_s3)

    return S1, S2, S3


# ==========================================================
# 9) ESCORRENTÍA Y RUTEO
# ==========================================================
def rolling_sum_5d(stack_TRC: np.ndarray, prev4=None):
    if prev4 is None:
        prev4 = np.zeros((0,) + stack_TRC.shape[1:], dtype=stack_TRC.dtype)

    combo = np.concatenate([prev4, stack_TRC], axis=0)
    cs = np.cumsum(combo, axis=0)

    amc = np.empty_like(combo, dtype="float64")
    for i in range(combo.shape[0]):
        if i < 4:
            amc[i] = np.sum(combo[:i+1], axis=0)
        else:
            amc[i] = cs[i] - cs[i-5]

    tail4 = combo[-4:] if combo.shape[0] >= 4 else combo
    return amc[-stack_TRC.shape[0]:], tail4

def runoff_scs(ppt, amc5, S1, S2, S3, lambda_ia=0.2):
    S = np.where(amc5 <= 13, S1, S2)
    S = np.where(amc5 > 28, S3, S)

    lam = float(np.clip(lambda_ia, 0.001, 0.8))
    Ia = lam * S

    numer = np.power(ppt - Ia, 2)
    denom = (ppt - Ia) + S

    with np.errstate(divide="ignore", invalid="ignore"):
        Q = np.where(ppt < Ia, 0.0, numer / denom)

    return np.where(np.isfinite(Q), Q, 0.0)

def route_linear_reservoir(runoff_mm, K=2.0):
    r = np.asarray(runoff_mm, dtype=float)
    q = np.zeros_like(r)
    alpha = np.exp(-1.0 / max(K, 1e-6))

    for t in range(len(r)):
        q[t] = (1 - alpha) * r[t] if t == 0 else alpha * q[t-1] + (1 - alpha) * r[t]
    return q

def route_nash_cascade(runoff_mm, n=3, K=2.0):
    q = np.asarray(runoff_mm, dtype=float)
    for _ in range(int(n)):
        q = route_linear_reservoir(q, K=K)
    return q

def route_nash_cascade_step(runoff_mm, states=None, n=3, K=2.0):
    n = int(n)
    alpha = np.exp(-1.0 / max(float(K), 1e-6))

    if states is None or len(states) != n:
        states = [0.0] * n

    x = float(runoff_mm)
    new_states = []

    for i in range(n):
        prev_out = float(states[i])
        out = alpha * prev_out + (1.0 - alpha) * x
        new_states.append(out)
        x = out

    return float(x), new_states


# ==========================================================
# 10) EMBALSE
# ==========================================================
def area_embalse(V):
    D366 = V / 1e6
    area_km2 = (
        8.91946288801159E-08 * D366**4
        - 0.0000346583770904819 * D366**3
        + 0.003919249094071 * D366**2
        - 0.0306477700093546 * D366
        + 2.02308232795992
    )
    return max(area_km2, 0) * 1e6

def volumen_auxiliar(V_prev, esc_mm, A_c, Vd):
    A_emb = area_embalse(V_prev)
    A_aporte = max(A_c - A_emb, 0)
    esc_m = esc_mm / 1000.0
    V_esc = esc_m * A_aporte
    return V_prev + V_esc - Vd

def area_media(V_prev, V_aux):
    return 0.5 * (area_embalse(V_prev) + area_embalse(V_aux))

def aplicar_restricciones(B_i):
    if B_i > VOL_MAX:
        return VOL_MAX, B_i - VOL_MAX
    if B_i < VOL_MUERTO:
        return VOL_MUERTO, 0.0
    return B_i, 0.0

def balance_embalse_diario(V_prev, esc_mm, precip_mm, et_mm, A_c, Vd, Vs_m_d, k_infil=1.0, k_et=1.0):
    V_aux = volumen_auxiliar(V_prev, esc_mm, A_c, Vd)
    A_i = area_media(V_prev, V_aux)

    esc_m = esc_mm / 1000.0
    precip_m = precip_mm / 1000.0
    et_m = (et_mm * k_et) / 1000.0

    V_esc = esc_m * max(A_c - A_i, 0.0)
    V_clima = (precip_m - et_m) * A_i
    Vs_vol = (float(Vs_m_d) * float(k_infil)) * A_i

    B_i = V_prev + V_esc + V_clima - Vd - Vs_vol
    V_emb, V_vertido = aplicar_restricciones(B_i)

    return V_emb, V_vertido, A_i, V_clima, V_esc, Vs_vol, B_i, Vd

def volumen_util_a_total_m3(v_util_hm3):
    return VOL_MUERTO + (float(v_util_hm3) * 1_000_000.0)


# ==========================================================
# 11) MÉTRICAS
# ==========================================================
def _compute_metrics(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    m = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[m]
    obs = obs[m]

    if len(obs) < 1:
        return {
            "n": int(len(obs)),
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan
        }

    if len(obs) >= 2 and np.std(obs) > 0 and np.std(sim) > 0:
        r = float(np.corrcoef(obs, sim)[0, 1])
        R2 = float(r ** 2)
    else:
        r = np.nan
        R2 = np.nan

    RMSE = float(np.sqrt(np.mean((sim - obs) ** 2)))

    denom = float(np.sum((obs - np.mean(obs)) ** 2))
    NSE = float(1 - np.sum((sim - obs) ** 2) / denom) if denom > 0 else np.nan

    sigma = float(np.std(obs))
    RMSE_n = float(RMSE / sigma) if sigma > 0 else np.nan

    mu_o = float(np.mean(obs))
    mu_s = float(np.mean(sim))
    Bias = float((mu_s - mu_o) / mu_o) if abs(mu_o) > 1e-12 else np.nan
    PBIAS = float(Bias * 100.0) if np.isfinite(Bias) else np.nan

    alpha = float(np.std(sim) / np.std(obs)) if np.std(obs) > 0 else np.nan
    beta = float(mu_s / mu_o) if abs(mu_o) > 1e-12 else np.nan

    if np.isfinite(r) and np.isfinite(alpha) and np.isfinite(beta):
        KGE = float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))
    else:
        KGE = np.nan

    return {
        "n": int(len(obs)),
        "R2": R2,
        "RMSE": RMSE,
        "NSE": NSE,
        "RMSE_n": RMSE_n,
        "Bias": Bias,
        "PBIAS": PBIAS,
        "KGE": KGE
    }


# ==========================================================
# 12) LECTURA DE DESCARGAS PROYECTADAS
# ==========================================================
def cargar_descarga_proyectada(csv_path: Path):
    if not csv_path.exists():
        raise FileNotFoundError(f"No existe descarga proyectada: {csv_path}")

    df = pd.read_csv(csv_path)
    cols_required = {"fecha", "descarga_proyectada_m3_s"}
    if not cols_required.issubset(df.columns):
        raise ValueError(f"{csv_path.name} debe tener columnas {cols_required}")

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce").dt.floor("D")
    df["descarga_proyectada_m3_s"] = pd.to_numeric(df["descarga_proyectada_m3_s"], errors="coerce")
    df = df.dropna(subset=["fecha", "descarga_proyectada_m3_s"]).sort_values("fecha").reset_index(drop=True)
    return df


# ==========================================================
# 13) CARGA DE DATOS FIJOS
# ==========================================================
header("CARGANDO DATOS FIJOS")

roi_cuenca_geoms  = load_geoms(AOI_CUENCA_PATH)
roi_embalse_geoms = load_geoms(AOI_EMBALSE_PATH)

QM_TABLE = load_qm_table(QM_MODEL_PATH)

LULC_TIF = find_single_tif(LULC_DIR)
SOIL_TIF = find_single_tif(SOIL_DIR)

ok(f"LULC: {LULC_TIF}")
ok(f"SOIL: {SOIL_TIF}")

lulc_stack, _ = read_clip_stack(LULC_TIF, roi_cuenca_geoms)
soil_stack, _ = read_clip_stack(SOIL_TIF, roi_cuenca_geoms)

LULC_CUENCA = lulc_stack[0].astype("int16")
SOIL_CUENCA = soil_stack[0].astype("int16")

print("LULC únicos (cuenca):", np.unique(LULC_CUENCA))
print("SOIL únicos (cuenca):", np.unique(SOIL_CUENCA))

cn_table_local = reemplazar_cn_lulc10_16(PARAMS)
S1_CU, S2_CU, S3_CU = build_cn_s_from_cache(
    LULC_CUENCA, SOIL_CUENCA,
    cn_table_local=cn_table_local,
    params=PARAMS,
    f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4
)


# ==========================================================
# 13-bis) UTILIDADES TXT / PROGRESO
# ==========================================================
def save_block_txt(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False, encoding="utf-8")

def read_block_txt(path: Path):
    return pd.read_csv(path, sep="\t", encoding="utf-8", parse_dates=["fecha"])

def save_resumen_txt(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False, encoding="utf-8")

def read_resumen_txt(path: Path):
    return pd.read_csv(path, sep="\t", encoding="utf-8")

def combo_name(modelo: str, ssp: str, esc_desc: str) -> str:
    return f"{modelo}_{ssp}_{UMBRAL_PRECIP}_{esc_desc}"

def combo_dirs(modelo: str, ssp: str, esc_desc: str):
    nombre = combo_name(modelo, ssp, esc_desc)
    combo_dir = TMP_DIR / nombre
    blocks_dir = combo_dir / "bloques_txt"
    states_dir = combo_dir / "estados"
    combo_dir.mkdir(parents=True, exist_ok=True)
    blocks_dir.mkdir(parents=True, exist_ok=True)
    states_dir.mkdir(parents=True, exist_ok=True)
    return nombre, combo_dir, blocks_dir, states_dir

def construir_bloques_10_anios(claves, block_years=10):
    if not claves:
        return []

    years = sorted({y for y, _ in claves})
    y_min = min(years)
    y_max = max(years)

    bloques = []
    y0 = y_min
    while y0 <= y_max:
        y1 = min(y0 + block_years - 1, y_max)
        claves_block = [(y, m) for (y, m) in claves if y0 <= y <= y1]
        if claves_block:
            bloques.append({
                "start_year": y0,
                "end_year": y1,
                "claves": claves_block
            })
        y0 = y1 + 1

    return bloques

def estado_inicial_combo():
    return {
        "V_emb": float(V_INICIAL),
        "routing_states": [0.0] * int(PARAMS["n"]),
        "last_date": None
    }

def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: Path, default=None):
    if not path.exists():
        return {} if default is None else default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def progress_path_for(combo_dir: Path):
    return combo_dir / "progress.json"

def load_progress(combo_dir: Path):
    p = progress_path_for(combo_dir)
    if not p.exists():
        prog = {
            "status": "new",
            "completed_blocks": []
        }
        save_json(p, prog)
        return prog
    return load_json(p, default={"status": "new", "completed_blocks": []})

def save_progress(combo_dir: Path, progress: dict):
    save_json(progress_path_for(combo_dir), progress)

def remove_file_if_exists(path_str):
    if not path_str:
        return
    p = Path(path_str)
    if p.exists():
        p.unlink()

def limpiar_ultimos_bloques_si_hay_reinicio(combo_dir: Path, final_xlsx: Path, n_remove=2):
    progress = load_progress(combo_dir)

    if final_xlsx.exists():
        progress["status"] = "finished"
        save_progress(combo_dir, progress)
        return progress

    completed = progress.get("completed_blocks", [])

    if len(completed) == 0:
        return progress

    # Si hay progreso previo y no existe Excel final, re-procesar últimos bloques
    n_remove = min(int(n_remove), len(completed))
    warn(f"Reanudación detectada. Se borrarán los últimos {n_remove} bloques para reprocesarlos.")

    for item in completed[-n_remove:]:
        remove_file_if_exists(item.get("block_file"))
        remove_file_if_exists(item.get("state_file"))
        remove_file_if_exists(item.get("prev4_file"))

    progress["completed_blocks"] = completed[:-n_remove]
    progress["status"] = "resumed_after_cleanup"
    save_progress(combo_dir, progress)
    return progress

def cargar_estado_desde_progreso(progress: dict):
    completed = progress.get("completed_blocks", [])
    if not completed:
        return estado_inicial_combo(), None

    last = completed[-1]
    state_file = Path(last["state_file"])
    prev4_file = Path(last["prev4_file"])

    estado = load_json(state_file, default=estado_inicial_combo())

    prev4 = None
    if prev4_file.exists():
        prev4 = np.load(prev4_file, allow_pickle=False)
        if prev4.size == 0:
            prev4 = None

    return estado, prev4

def guardar_estado_bloque(states_dir: Path, start_year: int, end_year: int, estado: dict, prev4):
    state_file = states_dir / f"state_{start_year}_{end_year}.json"
    prev4_file = states_dir / f"prev4_{start_year}_{end_year}.npy"

    estado_serializable = {
        "V_emb": float(estado["V_emb"]),
        "routing_states": [float(x) for x in estado["routing_states"]],
        "last_date": estado["last_date"]
    }
    save_json(state_file, estado_serializable)

    if prev4 is None:
        np.save(prev4_file, np.empty((0,), dtype="float32"), allow_pickle=False)
    else:
        np.save(prev4_file, prev4.astype("float32"), allow_pickle=False)

    return state_file, prev4_file

def leer_bloques_concatenados(progress: dict):
    bloques = progress.get("completed_blocks", [])
    dfs = []

    for item in bloques:
        block_file = Path(item["block_file"])
        if block_file.exists():
            dfb = read_block_txt(block_file)
            dfs.append(dfb)

    if not dfs:
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.floor("D")
    df = df.sort_values("fecha").drop_duplicates(subset=["fecha"], keep="last").reset_index(drop=True)
    return df

def actualizar_resumen_global_parcial(row_dict: dict):
    if RESUMEN_GLOBAL_PARCIAL_TXT.exists():
        df = read_resumen_txt(RESUMEN_GLOBAL_PARCIAL_TXT)
    else:
        df = pd.DataFrame()

    if not df.empty:
        mask = (
            (df["modelo"] == row_dict["modelo"]) &
            (df["ssp"] == row_dict["ssp"]) &
            (df["umbral_precip"] == row_dict["umbral_precip"]) &
            (df["escenario_descarga"] == row_dict["escenario_descarga"])
        )
        df = df.loc[~mask].copy()

    df = pd.concat([df, pd.DataFrame([row_dict])], ignore_index=True)
    save_resumen_txt(df, RESUMEN_GLOBAL_PARCIAL_TXT)


# ==========================================================
# 14) FUNCIÓN PRINCIPAL DE SIMULACIÓN POR BLOQUES TXT
# ==========================================================
def simular_futuro_combinacion_por_bloques(modelo: str, ssp: str, esc_desc: str):
    ppt_folder = PPT_CORR_DIR / f"{modelo}_{ssp}_{UMBRAL_PRECIP}"
    et_folder  = ET_CORR_DIR  / f"{modelo}_{ssp}"
    desc_csv   = DESC_PROY_DIR / f"proyeccion_{esc_desc}.csv"

    if not ppt_folder.exists():
        raise FileNotFoundError(f"No existe carpeta de precipitación: {ppt_folder}")
    if not et_folder.exists():
        raise FileNotFoundError(f"No existe carpeta de ET: {et_folder}")
    if not desc_csv.exists():
        raise FileNotFoundError(f"No existe CSV de descarga: {desc_csv}")

    print_paths_for_run(modelo, ssp, esc_desc, ppt_folder, et_folder, desc_csv)

    # Listar TIFF mensuales
    ppt_files = listar_tifs_por_anio_mes(ppt_folder)
    et_files  = listar_tifs_por_anio_mes(et_folder)

    claves = sorted(set(ppt_files.keys()) & set(et_files.keys()))
    if not claves:
        raise RuntimeError(f"No hay meses comunes entre precipitación y ET para {modelo}_{ssp}")

    bloques = construir_bloques_10_anios(claves, block_years=BLOCK_YEARS)

    # Leer descarga proyectada
    df_desc = cargar_descarga_proyectada(desc_csv)
    df_desc["fecha"] = pd.to_datetime(df_desc["fecha"]).dt.floor("D")
    desc_lookup = dict(zip(df_desc["fecha"], df_desc["descarga_proyectada_m3_s"]))

    nombre_combo, combo_dir, blocks_dir, states_dir = combo_dirs(modelo, ssp, esc_desc)
    out_xlsx = OUT_DIR / f"{nombre_combo}.xlsx"

    progress = limpiar_ultimos_bloques_si_hay_reinicio(
        combo_dir=combo_dir,
        final_xlsx=out_xlsx,
        n_remove=REPROCESS_LAST_N_BLOCKS
    )

    bloques_completados = {
        (b["start_year"], b["end_year"])
        for b in progress.get("completed_blocks", [])
    }

    estado, prev4 = cargar_estado_desde_progreso(progress)
    V_emb = float(estado["V_emb"])
    routing_states = [float(x) for x in estado["routing_states"]]

    for bloque in bloques:
        y0 = bloque["start_year"]
        y1 = bloque["end_year"]

        if (y0, y1) in bloques_completados:
            ok(f"Bloque ya existe, se omite: {y0}-{y1}")
            continue

        header(f"PROCESANDO BLOQUE {y0}-{y1} | {nombre_combo}", char="=")

        rows_block = []
        claves_block = bloque["claves"]

        barra_bloque = tqdm(
            claves_block,
            desc=f"{nombre_combo} | {y0}-{y1}",
            unit="mes",
            colour="green"
        )

        for (year, month) in barra_bloque:
            barra_bloque.set_postfix(fecha=f"{year}-{month:02d}")

            ppt_path = ppt_files[(year, month)]
            et_path  = et_files[(year, month)]

            # 1) Precipitación cuenca
            ppt_cu_stack, _ = read_clip_stack(ppt_path, roi_cuenca_geoms)
            ppt_cu = qm_correct_array(ppt_cu_stack.astype("float64"), month, QM_TABLE)
            ppt_cu_eff = ppt_cu * float(PARAMS["k_effP"])

            # 2) AMC + escorrentía
            amc5, prev4 = rolling_sum_5d(ppt_cu_eff, prev4=prev4)
            q_cu = runoff_scs(
                ppt_cu_eff, amc5, S1_CU, S2_CU, S3_CU,
                lambda_ia=float(PARAMS["lambda_ia"])
            )

            q_mean = [nanmean_masked(q_cu[i], None) for i in range(q_cu.shape[0])]
            q_mean = [v * float(PARAMS["k_q"]) for v in q_mean]

            ppt_cu_mean = [nanmean_masked(ppt_cu_eff[i], None) for i in range(ppt_cu_eff.shape[0])]

            # 3) Precipitación embalse
            ppt_em_stack, ppt_em_nodata = read_clip_stack(ppt_path, roi_embalse_geoms)
            ppt_em = qm_correct_array(ppt_em_stack.astype("float64"), month, QM_TABLE)
            ppt_em_mean = [nanmean_masked(ppt_em[i], ppt_em_nodata) for i in range(ppt_em.shape[0])]
            ppt_em_mean = [v * float(PARAMS["k_effP"]) for v in ppt_em_mean]

            # 4) ET embalse
            et_em_stack, et_em_nodata = read_clip_stack(et_path, roi_embalse_geoms)
            et_em = et_em_stack.astype("float64")
            et_em_mean = [nanmean_masked(et_em[i], et_em_nodata) for i in range(et_em.shape[0])]

            ndays = calendar.monthrange(year, month)[1]
            T = min(ndays, len(q_mean), len(ppt_em_mean), len(et_em_mean), len(ppt_cu_mean))

            for d in range(T):
                fecha = pd.Timestamp(datetime(year, month, d + 1)).floor("D")

                esc_sin_ruteo = float(q_mean[d])
                esc_ruteada, routing_states = route_nash_cascade_step(
                    esc_sin_ruteo,
                    states=routing_states,
                    n=int(PARAMS["n"]),
                    K=float(PARAMS["K"])
                )

                descarga_m3_s = desc_lookup.get(fecha, None)
                if descarga_m3_s is None:
                    continue

                Vd = float(descarga_m3_s) * 86400.0
                p_mm = float(ppt_em_mean[d])
                et_mm = float(et_em_mean[d])

                V_prev = float(V_emb)

                V_emb, V_vert, A_i, V_clima, V_esc, Vs_vol, B_i, Vd = balance_embalse_diario(
                    V_prev, esc_ruteada, p_mm, et_mm,
                    AREA_CUENCA_TOTAL, Vd,
                    Vs_m_d=float(PARAMS["Vs"]),
                    k_infil=float(PARAMS["k_infil"]),
                    k_et=float(PARAMS["k_et"])
                )

                # Corrección puntual del volumen
                if fecha in CORRECCIONES_UTIL_HM3:
                    V_emb = volumen_util_a_total_m3(CORRECCIONES_UTIL_HM3[fecha])
                    V_emb = max(VOL_MUERTO, min(V_emb, VOL_MAX))

                rows_block.append({
                    "fecha": fecha,
                    "modelo": modelo,
                    "ssp": ssp,
                    "umbral_precip": UMBRAL_PRECIP,
                    "escenario_descarga": esc_desc,
                    "ppt_cuenca_eff_mm": float(ppt_cu_mean[d]),
                    "esc_mm_sin_ruteo": esc_sin_ruteo,
                    "esc_mm_ruteada": float(esc_ruteada),
                    "precip_mm_embalse": p_mm,
                    "et_mm_embalse": et_mm,
                    "descarga_proyectada_m3_s": float(descarga_m3_s),
                    "descarga_proyectada_m3_dia": Vd,
                    "V_emb_m3": float(V_emb),
                    "V_emb_hm3": float(V_emb) / 1e6,
                    "V_emb_util_hm3": max((float(V_emb) - VOL_MUERTO) / 1e6, 0.0),
                    "V_vertido_m3": float(V_vert),
                    "area_embalse_m2": float(A_i),
                    "V_clima_m3": float(V_clima),
                    "V_esc_m3": float(V_esc),
                    "Vs_vol_m3": float(Vs_vol),
                    "balance_bruto_m3": float(B_i),
                    "corregido_volumen": int(fecha in CORRECCIONES_UTIL_HM3),
                    "vol_util_objetivo_hm3": CORRECCIONES_UTIL_HM3.get(fecha, np.nan)
                })

            # Liberar RAM por mes
            del ppt_cu_stack, ppt_cu, ppt_cu_eff, amc5, q_cu
            del ppt_em_stack, ppt_em
            del et_em_stack, et_em
            gc.collect()

        barra_bloque.close()

        df_block = pd.DataFrame(rows_block)
        if not df_block.empty:
            df_block["fecha"] = pd.to_datetime(df_block["fecha"]).dt.floor("D")
            df_block = df_block.sort_values("fecha").drop_duplicates(subset=["fecha"], keep="last").reset_index(drop=True)

        block_file = blocks_dir / f"serie_{y0}_{y1}.txt"
        save_block_txt(df_block, block_file)

        estado = {
            "V_emb": float(V_emb),
            "routing_states": [float(x) for x in routing_states],
            "last_date": None if df_block.empty else str(pd.to_datetime(df_block["fecha"].max()).date())
        }

        state_file, prev4_file = guardar_estado_bloque(
            states_dir=states_dir,
            start_year=y0,
            end_year=y1,
            estado=estado,
            prev4=prev4
        )

        progress = load_progress(combo_dir)
        progress["status"] = "running"
        progress["completed_blocks"].append({
            "start_year": y0,
            "end_year": y1,
            "block_file": str(block_file),
            "state_file": str(state_file),
            "prev4_file": str(prev4_file)
        })
        save_progress(combo_dir, progress)

        ok(f"Bloque TXT guardado: {block_file.name}")

        del rows_block, df_block
        gc.collect()

    # Unir bloques al final
    progress = load_progress(combo_dir)
    df_res = leer_bloques_concatenados(progress)

    if df_res.empty or len(df_res) < 100:
        raise RuntimeError(f"La serie final quedó vacía o demasiado corta para {nombre_combo}")

    df_ctrl = df_res[df_res["corregido_volumen"] == 1].copy()
    if len(df_ctrl) >= 1:
        met_ctrl = _compute_metrics(df_ctrl["V_emb_util_hm3"], df_ctrl["vol_util_objetivo_hm3"])
    else:
        met_ctrl = {k: np.nan for k in ["n", "R2", "RMSE", "NSE", "RMSE_n", "Bias", "PBIAS", "KGE"]}

    progress["status"] = "finished"
    save_progress(combo_dir, progress)

    return df_res, met_ctrl, ppt_folder, et_folder, desc_csv, combo_dir


# ==========================================================
# 15) EJECUCIÓN MULTICOMBINACIÓN
# ==========================================================
header("SIMULACIÓN FUTURA EL PAÑE | u0p5 | BLOQUES TXT")

combinaciones = []
for modelo in MODELOS:
    for ssp in SSP_LIST:
        for esc_desc in ESCENARIOS_DESCARGA:
            combinaciones.append((modelo, ssp, esc_desc))

if RESUMEN_GLOBAL_PARCIAL_TXT.exists():
    try:
        resumen_global = read_resumen_txt(RESUMEN_GLOBAL_PARCIAL_TXT).to_dict(orient="records")
    except Exception:
        resumen_global = []
else:
    resumen_global = []

barra_total = tqdm(combinaciones, desc="Combinaciones totales", unit="combo", colour="blue")

for modelo, ssp, esc_desc in barra_total:
    barra_total.set_postfix(modelo=modelo, ssp=ssp, esc=esc_desc)

    nombre_combo = combo_name(modelo, ssp, esc_desc)
    out_xlsx = OUT_DIR / f"{nombre_combo}.xlsx"

    try:
        # Si ya existe el Excel final, lo omite
        if out_xlsx.exists():
            ok(f"Ya existe Excel final, se omite: {out_xlsx.name}")

            row_resumen = {
                "modelo": modelo,
                "ssp": ssp,
                "umbral_precip": UMBRAL_PRECIP,
                "escenario_descarga": esc_desc,
                "excel": str(out_xlsx),
                "n": np.nan,
                "R2": np.nan,
                "RMSE": np.nan,
                "NSE": np.nan,
                "RMSE_n": np.nan,
                "Bias": np.nan,
                "PBIAS": np.nan,
                "KGE": np.nan,
                "estado": "ya_existia"
            }
            actualizar_resumen_global_parcial(row_resumen)
            continue

        df_res, met_ctrl, ppt_folder, et_folder, desc_csv, combo_dir = simular_futuro_combinacion_por_bloques(
            modelo, ssp, esc_desc
        )

        df_paths = pd.DataFrame([
            {"entrada": "ppt_corregida", "ruta": str(ppt_folder)},
            {"entrada": "et_corregido", "ruta": str(et_folder)},
            {"entrada": "descarga_proyectada", "ruta": str(desc_csv)},
            {"entrada": "qm_model", "ruta": str(QM_MODEL_PATH)},
            {"entrada": "lulc", "ruta": str(LULC_TIF)},
            {"entrada": "soil", "ruta": str(SOIL_TIF)},
            {"entrada": "cuenca", "ruta": str(AOI_CUENCA_PATH)},
            {"entrada": "embalse", "ruta": str(AOI_EMBALSE_PATH)},
            {"entrada": "tmp_bloques_txt", "ruta": str(combo_dir)},
        ])

        df_params = pd.DataFrame([
            {"parametro": k, "valor": v} for k, v in PARAMS.items()
        ])

        df_props = pd.DataFrame([
            {"propiedad": "VOL_MUERTO_m3", "valor": VOL_MUERTO},
            {"propiedad": "VOL_MAX_m3", "valor": VOL_MAX},
            {"propiedad": "AREA_CUENCA_TOTAL_m2", "valor": AREA_CUENCA_TOTAL},
            {"propiedad": "V_INICIAL_m3", "valor": V_INICIAL},
            {"propiedad": "UMBRAL_PRECIP", "valor": UMBRAL_PRECIP},
            {"propiedad": "BLOCK_YEARS", "valor": BLOCK_YEARS},
            {"propiedad": "REPROCESS_LAST_N_BLOCKS", "valor": REPROCESS_LAST_N_BLOCKS},
            {"propiedad": "correccion_2019_12_31_util_hm3", "valor": 56.00},
            {"propiedad": "correccion_2025_12_31_util_hm3", "valor": 36.37},
        ])

        df_metricas = pd.DataFrame([{
            "modelo": modelo,
            "ssp": ssp,
            "umbral_precip": UMBRAL_PRECIP,
            "escenario_descarga": esc_desc,
            **met_ctrl
        }])

        progress = load_progress(combo_dir)
        df_bloques = pd.DataFrame(progress.get("completed_blocks", []))

        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            df_metricas.to_excel(writer, sheet_name="metricas_control", index=False)
            df_props.to_excel(writer, sheet_name="propiedades_modelo", index=False)
            df_params.to_excel(writer, sheet_name="parametros_hidrologicos", index=False)
            df_paths.to_excel(writer, sheet_name="rutas_usadas", index=False)
            df_bloques.to_excel(writer, sheet_name="bloques_procesados", index=False)
            df_res.to_excel(writer, sheet_name="serie_simulada", index=False)

        row_resumen = {
            "modelo": modelo,
            "ssp": ssp,
            "umbral_precip": UMBRAL_PRECIP,
            "escenario_descarga": esc_desc,
            "excel": str(out_xlsx),
            **met_ctrl,
            "estado": "ok"
        }
        actualizar_resumen_global_parcial(row_resumen)

        ok(f"Excel generado: {out_xlsx.name}")

        del df_res, df_metricas, df_props, df_params, df_paths, df_bloques
        gc.collect()

    except Exception as e:
        err(f"{nombre_combo}: {e}")

        row_resumen = {
            "modelo": modelo,
            "ssp": ssp,
            "umbral_precip": UMBRAL_PRECIP,
            "escenario_descarga": esc_desc,
            "excel": str(out_xlsx),
            "n": np.nan,
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan,
            "estado": "error",
            "error": str(e),
        }
        actualizar_resumen_global_parcial(row_resumen)

barra_total.close()


# ==========================================================
# 16) RESUMEN GLOBAL FINAL
# ==========================================================
if RESUMEN_GLOBAL_PARCIAL_TXT.exists():
    df_resumen_global = read_resumen_txt(RESUMEN_GLOBAL_PARCIAL_TXT)
else:
    df_resumen_global = pd.DataFrame(resumen_global)

out_resumen = OUT_DIR / "resumen_global_u0p5.xlsx"

with pd.ExcelWriter(out_resumen, engine="openpyxl") as writer:
    df_resumen_global.to_excel(writer, sheet_name="resumen_global", index=False)

header("PROCESO COMPLETADO", char="█")
print(f"  Carpeta de salida           : {OUT_DIR}")
print(f"  Carpeta temporal TXT        : {TMP_DIR}")
print(f"  Resumen parcial TXT         : {RESUMEN_GLOBAL_PARCIAL_TXT}")
print(f"  Excel resumen global        : {out_resumen}")
print(f"  Total combinaciones         : {len(combinaciones)}")

## 5.4 · El Pañe Futuro — versión robusta con limpieza de NA y reanudación

Incluye:

- Reemplazo de píxeles NA/nodata por 0.0 en precipitación.
- Imputación de huecos en temperatura (vecinos diarios o media mensual).
- Reanudación automática tras cortes de ejecución.


In [ ]:
# ==========================================================
# EL PAÑE FUTURO
# MODELO HIDROLÓGICO + EMBALSE
# ----------------------------------------------------------
# CORRECCIONES IMPORTANTES INCLUIDAS EN ESTA VERSIÓN:
#   1) PRECIPITACIÓN:
#      - Todo pixel NA / nodata / no finito se reemplaza por 0.0
#      - Esto se aplica antes del cálculo hidrológico y antes del promedio
#
#   2) TEMPERATURA (rutina incluida para uso futuro):
#      - Si falta 1 solo día: promedio del día anterior y siguiente
#      - Si faltan varios días consecutivos: promedio mensual
#      - Si todo el mes falta: usa 0.0 por seguridad
#
#   3) SERIES DIARIAS:
#      - Se protegen contra NaN para que no colapse el volumen del embalse
#
#   4) REANUDACIÓN:
#      - Si el proceso se corta, al reiniciar:
#        * detecta los bloques ya procesados
#        * borra los 2 últimos bloques
#        * los vuelve a procesar
#
# SALIDAS:
#   - TXT por bloques de 10 años
#   - Excel final por combinación
#   - Excel resumen global
# ==========================================================

# ==========================================================
# 0) LIBRERÍAS
# ==========================================================
import os
import re
import json
import calendar
import gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)


# ==========================================================
# 1) CONFIGURACIÓN GENERAL
# ==========================================================
BASE_DIR = Path("/content/drive/MyDrive/Project001")

# -------------------------
# Carpetas estáticas / base
# -------------------------
LULC_DIR = BASE_DIR / "LULC_LC_Type1_WGS84"
SOIL_DIR = BASE_DIR / "SOIL_TEXTURE_b0_WGS84"
SHP_DIR  = BASE_DIR / "shp"

AOI_CUENCA_PATH  = SHP_DIR / "Cuenca-es.geojson"
AOI_EMBALSE_PATH = SHP_DIR / "Embalse.geojson"

QM_MODEL_PATH = BASE_DIR / "QM_model_monthly.csv"

# -------------------------
# Carpetas futuras corregidas
# -------------------------
CLIMA_DIR = BASE_DIR / "DATOS_CLIMATICOS_TOTALES"
PPT_CORR_DIR = CLIMA_DIR / "PRECIPITACION_CORREGIDA_QDM"
ET_CORR_DIR  = CLIMA_DIR / "ET_HARGREAVES_CORREGIDA_QDM"

# -------------------------
# Descargas proyectadas
# -------------------------
DESC_PROY_DIR = BASE_DIR / "proyeccion_descargas_ciclo_3anios_promedio_no_ceros_2020_2100"

# -------------------------
# Carpeta de salida
# -------------------------
OUT_DIR = BASE_DIR / "resultados_pane_futuro_u0p5_excel"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Temporales por bloques
# -------------------------
BLOCK_YEARS = 10
REPROCESS_LAST_N_BLOCKS = 2

TMP_DIR = OUT_DIR / "_tmp_bloques_txt"
TMP_DIR.mkdir(parents=True, exist_ok=True)

RESUMEN_GLOBAL_PARCIAL_TXT = OUT_DIR / "resumen_global_u0p5_parcial.txt"

# -------------------------
# Modelos / escenarios
# -------------------------
MODELOS = [
    "MPI_ESM1_2_LR",
    "EC_Earth3_Veg_LR",
    "EC_Earth3"
]

SSP_LIST = ["ssp245", "ssp585"]
ESCENARIOS_DESCARGA = ["ESC_00", "ESC_25", "ESC_50", "ESC_75", "ESC_100"]

UMBRAL_PRECIP = "u0p5"


# ==========================================================
# 2) PARÁMETROS DEL MODELO HIDROLÓGICO
# ==========================================================
PARAMS = {
    "descripcion": "GEN 13/20 - IND 6/42",
    "g7": 1,
    "g9": 2,
    "n": 4,
    "K": 4.313107873728896,
    "k_infil": 0.6953880721605649,
    "k_et": 0.8082241252469766,
    "Vs": 0.009594970137013482,
    "CN10_g7": 95.72124246470727,
    "CN16_g7": 85.71535662377057,
    "CN10_g9": 96.31605148608226,
    "CN16_g9": 66.28439128813265,
    "k_effP": 1.812739987216763,
    "k_q": 1.7089490774160854,
    "lambda_ia": 0.03914373443714243,
}


# ==========================================================
# 3) PARÁMETROS DEL EMBALSE
# ==========================================================
VOL_MUERTO = 41e6
VOL_MAX    = 148.80e6
AREA_CUENCA_TOTAL = 196.33e6
V_INICIAL = 57.80e6


# ==========================================================
# 4) FECHAS DE CORRECCIÓN DEL VOLUMEN
# ==========================================================
CORRECCIONES_UTIL_HM3 = {
    pd.Timestamp("2019-12-31"): 56.00,
    pd.Timestamp("2025-12-31"): 36.37,
}


# ==========================================================
# 5) UTILIDADES DE CONSOLA
# ==========================================================
def linea(char="─", n=80):
    print(char * n)

def header(txt, char="═"):
    linea(char)
    print(f"  {txt}")
    linea(char)

def ok(txt):
    print(f"  ✔ {txt}")

def warn(txt):
    print(f"  ⚠ {txt}")

def err(txt):
    print(f"  ✘ {txt}")

def print_paths_for_run(modelo, ssp, esc_desc, ppt_dir, et_dir, desc_csv):
    header(f"USANDO ENTRADAS | {modelo} | {ssp} | {esc_desc}", char="·")
    print(f"  Precipitación corregida : {ppt_dir}")
    print(f"  ET corregido            : {et_dir}")
    print(f"  Descarga proyectada     : {desc_csv}")
    print(f"  QM mensual              : {QM_MODEL_PATH}")
    print(f"  LULC                    : {LULC_DIR}")
    print(f"  SOIL                    : {SOIL_DIR}")
    print(f"  Cuenca                  : {AOI_CUENCA_PATH}")
    print(f"  Embalse                 : {AOI_EMBALSE_PATH}")
    print(f"  Temporales TXT          : {TMP_DIR}")
    linea("·")


# ==========================================================
# 6) UTILIDADES GENERALES
# ==========================================================
def load_geoms(geojson_path: Path):
    if not geojson_path.exists():
        raise FileNotFoundError(f"No existe AOI: {geojson_path}")
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        raise ValueError(f"AOI vacío: {geojson_path}")
    geom = gdf.geometry.union_all()
    return [geom]

def read_clip_stack(tif_path: Path, geoms):
    if not tif_path.exists():
        raise FileNotFoundError(f"No existe raster: {tif_path}")
    with rasterio.open(tif_path) as src:
        out_img, _ = mask(src, geoms, crop=True, filled=True)
        nodata = src.nodata
    return out_img, nodata

def to_nan(arr: np.ndarray, nodata=None) -> np.ndarray:
    """
    Convierte nodata / no finitos a NaN.
    No modifica el original.
    """
    a = np.asarray(arr, dtype="float64").copy()

    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)

    a[~np.isfinite(a)] = np.nan
    return a

def safe_num(x, default=0.0) -> float:
    """
    Convierte cualquier valor a float seguro.
    Si no es finito o falla, devuelve default.
    """
    try:
        x = float(x)
        if not np.isfinite(x):
            return float(default)
        return float(x)
    except Exception:
        return float(default)

def nanmean_masked(a: np.ndarray, nodata=None, fill_all_nan=np.nan) -> float:
    """
    Promedio ignorando nodata.
    Si toda la matriz es NaN, devuelve fill_all_nan.
    """
    a = to_nan(a, nodata=nodata)

    if np.all(np.isnan(a)):
        return float(fill_all_nan)

    return float(np.nanmean(a))

def fill_precip_stack(arr: np.ndarray, nodata=None) -> np.ndarray:
    """
    REGLA PARA PRECIPITACIÓN:
    Todo NA / nodata / no finito se vuelve 0.0
    y además se asegura que no haya valores negativos.
    """
    a = to_nan(arr, nodata=nodata)
    a = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
    a = np.maximum(a, 0.0)
    return a

def fill_temperature_series(values, default_if_all_nan=0.0):
    """
    REGLA PARA TEMPERATURA:
      - 1 día faltante aislado -> promedio del día anterior y siguiente
      - varios días consecutivos faltantes -> promedio mensual
      - si todo el mes falta -> default_if_all_nan

    NOTA:
    Esta rutina queda lista para usarse si luego incorporas tasmin/tasmax.
    """
    s = pd.to_numeric(pd.Series(values), errors="coerce").astype(float)
    vals = s.to_numpy()

    if np.all(np.isnan(vals)):
        return [float(default_if_all_nan)] * len(vals)

    monthly_mean = float(np.nanmean(vals))
    i = 0
    n = len(vals)

    while i < n:
        if np.isfinite(vals[i]):
            i += 1
            continue

        j = i
        while j < n and not np.isfinite(vals[j]):
            j += 1

        run_len = j - i

        if run_len == 1:
            k = i
            prev_ok = (k - 1 >= 0) and np.isfinite(vals[k - 1])
            next_ok = (k + 1 < n) and np.isfinite(vals[k + 1])

            if prev_ok and next_ok:
                vals[k] = 0.5 * (vals[k - 1] + vals[k + 1])
            else:
                vals[k] = monthly_mean
        else:
            vals[i:j] = monthly_mean

        i = j

    vals = np.where(np.isfinite(vals), vals, monthly_mean)
    return [float(v) for v in vals]

def fill_temperature_stack(arr: np.ndarray, nodata=None, default_if_all_nan=0.0) -> np.ndarray:
    """
    Relleno temporal por pixel para un stack de temperatura (días, filas, columnas).
    Se deja incorporado para uso futuro si agregas tasmax/tasmin.
    """
    a = to_nan(arr, nodata=nodata)

    if a.ndim != 3:
        return np.nan_to_num(a, nan=default_if_all_nan, posinf=default_if_all_nan, neginf=default_if_all_nan)

    t, h, w = a.shape
    flat = a.reshape(t, -1)

    for col in range(flat.shape[1]):
        flat[:, col] = np.array(
            fill_temperature_series(flat[:, col], default_if_all_nan=default_if_all_nan),
            dtype="float64"
        )

    return flat.reshape(t, h, w)

def fill_generic_daily_series(values, default_if_all_nan=0.0):
    """
    Para variables que NO son precipitación y NO son temperatura,
    rellena NA con el promedio mensual.
    Si todo el mes falta -> default_if_all_nan.
    """
    s = pd.to_numeric(pd.Series(values), errors="coerce").astype(float)

    if s.notna().sum() == 0:
        return [float(default_if_all_nan)] * len(s)

    monthly_mean = float(s.mean(skipna=True))
    s = s.fillna(monthly_mean)
    s = s.replace([np.inf, -np.inf], monthly_mean)
    s = s.fillna(monthly_mean)
    return [float(v) for v in s.tolist()]

def find_single_tif(folder: Path) -> Path:
    hits = sorted(folder.glob("*.tif")) + sorted(folder.glob("*.tiff"))
    if not hits:
        raise FileNotFoundError(f"No encontré tif/tiff en: {folder}")
    return hits[0]

def load_qm_table(csv_path: Path):
    if not csv_path.exists():
        raise FileNotFoundError(f"No existe QM CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"mes", "x", "y"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"QM_model_monthly.csv debe tener columnas {required}. Tiene: {list(df.columns)}")

    qm = {}
    for _, r in df.iterrows():
        mes = int(r["mes"])
        x = json.loads(r["x"]) if isinstance(r["x"], str) else r["x"]
        y = json.loads(r["y"]) if isinstance(r["y"], str) else r["y"]

        x = np.array(x, dtype=float)
        y = np.array(y, dtype=float)

        if x.size < 2 or y.size < 2 or x.size != y.size:
            raise ValueError(f"QM inválido en mes {mes}: x={x.size}, y={y.size}")

        qm[mes] = (x, y)

    faltantes = [m for m in range(1, 13) if m not in qm]
    if faltantes:
        raise ValueError(f"Faltan meses en QM: {faltantes}")

    return qm

def qm_correct_array(arr: np.ndarray, month: int, QM_TABLE):
    """
    Corrección QM/QDM para arreglos.
    Si entran NaN, luego se vuelven a sanear externamente según la variable.
    """
    x, y = QM_TABLE[month]
    idx = np.argsort(x)
    x2 = x[idx]
    y2 = y[idx]

    flat = arr.reshape(-1).astype("float64", copy=False)
    out = np.interp(flat, x2, y2).reshape(arr.shape)
    return np.maximum(out, 0.0)


# ==========================================================
# 7) PARSEO DE NOMBRES DE ARCHIVOS FUTUROS
# ==========================================================
def extraer_anio_mes_desde_nombre(nombre: str):
    m = re.search(r"_(\d{4})_(\d{2})(?:_|\.|$)", nombre)
    if not m:
        return None, None
    anio = int(m.group(1))
    mes = int(m.group(2))
    return anio, mes

def listar_tifs_por_anio_mes(folder: Path, suffix=".tif"):
    d = {}
    for p in sorted(folder.glob(f"*{suffix}")):
        anio, mes = extraer_anio_mes_desde_nombre(p.name)
        if anio is not None and mes is not None:
            d[(anio, mes)] = p
    return d


# ==========================================================
# 8) TABLAS CN
# ==========================================================
CN_TABLE = {
    1: {1:35, 2:25, 3:45, 4:39, 5:45, 6:49, 7:68, 8:36, 9:45, 10:30, 11:95, 12:67, 13:72, 14:63, 15:100, 16:74, 17:100},
    2: {1:50, 2:55, 3:66, 4:61, 5:66, 6:69, 7:79, 8:60, 9:66, 10:58, 11:95, 12:78, 13:82, 14:75, 15:100, 16:84, 17:100},
    3: {1:73, 2:70, 3:77, 4:74, 5:77, 6:79, 7:86, 8:73, 9:77, 10:71, 11:95, 12:85, 13:87, 14:83, 15:100, 16:90, 17:100},
    4: {1:79, 2:77, 3:83, 4:80, 5:83, 6:89, 7:89, 8:79, 9:83, 10:78, 11:95, 12:89, 13:89, 14:87, 15:100, 16:92, 17:100},
}

def soil_group_from_texture(soil_class, g7=2, g9=2):
    soil_grp = np.zeros_like(soil_class, dtype="int16")
    soil_grp = np.where(soil_class > 10, 1, soil_grp)
    soil_grp = np.where((soil_class > 4) & (soil_class <= 10), 2, soil_grp)
    soil_grp = np.where((soil_class > 1) & (soil_class <= 4), 3, soil_grp)
    soil_grp = np.where((soil_class > 0) & (soil_class <= 1), 4, soil_grp)

    soil_grp = np.where(soil_class == 7, int(g7), soil_grp)
    soil_grp = np.where(soil_class == 9, int(g9), soil_grp)
    return soil_grp

def reemplazar_cn_lulc10_16(params):
    cn_table_local = {k: v.copy() for k, v in CN_TABLE.items()}
    cn_table_local[int(params["g7"])][10] = float(params["CN10_g7"])
    cn_table_local[int(params["g7"])][16] = float(params["CN16_g7"])
    cn_table_local[int(params["g9"])][10] = float(params["CN10_g9"])
    cn_table_local[int(params["g9"])][16] = float(params["CN16_g9"])
    return cn_table_local

def build_cn_s_from_cache(LULC_CUENCA, SOIL_CUENCA, cn_table_local, params, f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4):
    lulc = LULC_CUENCA.astype("float64")
    soil = SOIL_CUENCA.astype("float64")
    soil_grp = soil_group_from_texture(soil, g7=params["g7"], g9=params["g9"])

    CN2 = np.zeros_like(lulc, dtype="float64")
    CN2 = np.where(soil_grp == 0, 100.0, CN2)

    for grp, lut in cn_table_local.items():
        for lc, cn in lut.items():
            CN2 = np.where((soil_grp == grp) & (lulc == lc), float(cn), CN2)

    CN2 = np.minimum(CN2 * float(f_cn2), 100.0)

    CN1 = CN2 / (2.281 - (CN2 * 0.0128))
    CN3 = CN2 / (0.427 + (CN2 * 0.00573))

    S1 = (25400.0 / CN1 - 254.0) * float(f_s1)
    S2 = (25400.0 / CN2 - 254.0) * float(f_s2)
    S3 = (25400.0 / CN3 - 254.0) * float(f_s3)

    return S1, S2, S3


# ==========================================================
# 9) ESCORRENTÍA Y RUTEO
# ==========================================================
def rolling_sum_5d(stack_TRC: np.ndarray, prev4=None):
    if prev4 is None:
        prev4 = np.zeros((0,) + stack_TRC.shape[1:], dtype=stack_TRC.dtype)

    combo = np.concatenate([prev4, stack_TRC], axis=0)
    cs = np.cumsum(combo, axis=0)

    amc = np.empty_like(combo, dtype="float64")
    for i in range(combo.shape[0]):
        if i < 4:
            amc[i] = np.sum(combo[:i+1], axis=0)
        else:
            amc[i] = cs[i] - cs[i-5]

    tail4 = combo[-4:] if combo.shape[0] >= 4 else combo
    return amc[-stack_TRC.shape[0]:], tail4

def runoff_scs(ppt, amc5, S1, S2, S3, lambda_ia=0.2):
    S = np.where(amc5 <= 13, S1, S2)
    S = np.where(amc5 > 28, S3, S)

    lam = float(np.clip(lambda_ia, 0.001, 0.8))
    Ia = lam * S

    numer = np.power(ppt - Ia, 2)
    denom = (ppt - Ia) + S

    with np.errstate(divide="ignore", invalid="ignore"):
        Q = np.where(ppt < Ia, 0.0, numer / denom)

    return np.where(np.isfinite(Q), Q, 0.0)

def route_linear_reservoir(runoff_mm, K=2.0):
    r = np.asarray(runoff_mm, dtype=float)
    q = np.zeros_like(r)
    alpha = np.exp(-1.0 / max(K, 1e-6))

    for t in range(len(r)):
        q[t] = (1 - alpha) * r[t] if t == 0 else alpha * q[t-1] + (1 - alpha) * r[t]
    return q

def route_nash_cascade(runoff_mm, n=3, K=2.0):
    q = np.asarray(runoff_mm, dtype=float)
    for _ in range(int(n)):
        q = route_linear_reservoir(q, K=K)
    return q

def route_nash_cascade_step(runoff_mm, states=None, n=3, K=2.0):
    n = int(n)
    alpha = np.exp(-1.0 / max(float(K), 1e-6))

    if states is None or len(states) != n:
        states = [0.0] * n

    x = float(runoff_mm)
    new_states = []

    for i in range(n):
        prev_out = float(states[i])
        out = alpha * prev_out + (1.0 - alpha) * x
        new_states.append(out)
        x = out

    return float(x), new_states


# ==========================================================
# 10) EMBALSE
# ==========================================================
def area_embalse(V):
    D366 = V / 1e6
    area_km2 = (
        8.91946288801159E-08 * D366**4
        - 0.0000346583770904819 * D366**3
        + 0.003919249094071 * D366**2
        - 0.0306477700093546 * D366
        + 2.02308232795992
    )
    return max(area_km2, 0) * 1e6

def volumen_auxiliar(V_prev, esc_mm, A_c, Vd):
    A_emb = area_embalse(V_prev)
    A_aporte = max(A_c - A_emb, 0)
    esc_m = esc_mm / 1000.0
    V_esc = esc_m * A_aporte
    return V_prev + V_esc - Vd

def area_media(V_prev, V_aux):
    return 0.5 * (area_embalse(V_prev) + area_embalse(V_aux))

def aplicar_restricciones(B_i):
    if B_i > VOL_MAX:
        return VOL_MAX, B_i - VOL_MAX
    if B_i < VOL_MUERTO:
        return VOL_MUERTO, 0.0
    return B_i, 0.0

def balance_embalse_diario(V_prev, esc_mm, precip_mm, et_mm, A_c, Vd, Vs_m_d, k_infil=1.0, k_et=1.0):
    V_aux = volumen_auxiliar(V_prev, esc_mm, A_c, Vd)
    A_i = area_media(V_prev, V_aux)

    esc_m = esc_mm / 1000.0
    precip_m = precip_mm / 1000.0
    et_m = (et_mm * k_et) / 1000.0

    V_esc = esc_m * max(A_c - A_i, 0.0)
    V_clima = (precip_m - et_m) * A_i
    Vs_vol = (float(Vs_m_d) * float(k_infil)) * A_i

    B_i = V_prev + V_esc + V_clima - Vd - Vs_vol
    V_emb, V_vertido = aplicar_restricciones(B_i)

    return V_emb, V_vertido, A_i, V_clima, V_esc, Vs_vol, B_i, Vd

def volumen_util_a_total_m3(v_util_hm3):
    return VOL_MUERTO + (float(v_util_hm3) * 1_000_000.0)


# ==========================================================
# 11) MÉTRICAS
# ==========================================================
def _compute_metrics(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    m = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[m]
    obs = obs[m]

    if len(obs) < 1:
        return {
            "n": int(len(obs)),
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan
        }

    if len(obs) >= 2 and np.std(obs) > 0 and np.std(sim) > 0:
        r = float(np.corrcoef(obs, sim)[0, 1])
        R2 = float(r ** 2)
    else:
        r = np.nan
        R2 = np.nan

    RMSE = float(np.sqrt(np.mean((sim - obs) ** 2)))

    denom = float(np.sum((obs - np.mean(obs)) ** 2))
    NSE = float(1 - np.sum((sim - obs) ** 2) / denom) if denom > 0 else np.nan

    sigma = float(np.std(obs))
    RMSE_n = float(RMSE / sigma) if sigma > 0 else np.nan

    mu_o = float(np.mean(obs))
    mu_s = float(np.mean(sim))
    Bias = float((mu_s - mu_o) / mu_o) if abs(mu_o) > 1e-12 else np.nan
    PBIAS = float(Bias * 100.0) if np.isfinite(Bias) else np.nan

    alpha = float(np.std(sim) / np.std(obs)) if np.std(obs) > 0 else np.nan
    beta = float(mu_s / mu_o) if abs(mu_o) > 1e-12 else np.nan

    if np.isfinite(r) and np.isfinite(alpha) and np.isfinite(beta):
        KGE = float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))
    else:
        KGE = np.nan

    return {
        "n": int(len(obs)),
        "R2": R2,
        "RMSE": RMSE,
        "NSE": NSE,
        "RMSE_n": RMSE_n,
        "Bias": Bias,
        "PBIAS": PBIAS,
        "KGE": KGE
    }


# ==========================================================
# 12) LECTURA DE DESCARGAS PROYECTADAS
# ==========================================================
def cargar_descarga_proyectada(csv_path: Path):
    if not csv_path.exists():
        raise FileNotFoundError(f"No existe descarga proyectada: {csv_path}")

    df = pd.read_csv(csv_path)
    cols_required = {"fecha", "descarga_proyectada_m3_s"}
    if not cols_required.issubset(df.columns):
        raise ValueError(f"{csv_path.name} debe tener columnas {cols_required}")

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce").dt.floor("D")
    df["descarga_proyectada_m3_s"] = pd.to_numeric(df["descarga_proyectada_m3_s"], errors="coerce")
    df = df.dropna(subset=["fecha", "descarga_proyectada_m3_s"]).sort_values("fecha").reset_index(drop=True)
    return df


# ==========================================================
# 13) CARGA DE DATOS FIJOS
# ==========================================================
header("CARGANDO DATOS FIJOS")

roi_cuenca_geoms  = load_geoms(AOI_CUENCA_PATH)
roi_embalse_geoms = load_geoms(AOI_EMBALSE_PATH)

QM_TABLE = load_qm_table(QM_MODEL_PATH)

LULC_TIF = find_single_tif(LULC_DIR)
SOIL_TIF = find_single_tif(SOIL_DIR)

ok(f"LULC: {LULC_TIF}")
ok(f"SOIL: {SOIL_TIF}")

lulc_stack, _ = read_clip_stack(LULC_TIF, roi_cuenca_geoms)
soil_stack, _ = read_clip_stack(SOIL_TIF, roi_cuenca_geoms)

LULC_CUENCA = lulc_stack[0].astype("int16")
SOIL_CUENCA = soil_stack[0].astype("int16")

print("LULC únicos (cuenca):", np.unique(LULC_CUENCA))
print("SOIL únicos (cuenca):", np.unique(SOIL_CUENCA))

cn_table_local = reemplazar_cn_lulc10_16(PARAMS)
S1_CU, S2_CU, S3_CU = build_cn_s_from_cache(
    LULC_CUENCA, SOIL_CUENCA,
    cn_table_local=cn_table_local,
    params=PARAMS,
    f_cn2=1.0, f_s1=1.3, f_s2=1.2, f_s3=1.4
)


# ==========================================================
# 13-bis) UTILIDADES TXT / PROGRESO
# ==========================================================
def save_block_txt(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False, encoding="utf-8")

def read_block_txt(path: Path):
    return pd.read_csv(path, sep="\t", encoding="utf-8", parse_dates=["fecha"])

def save_resumen_txt(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False, encoding="utf-8")

def read_resumen_txt(path: Path):
    return pd.read_csv(path, sep="\t", encoding="utf-8")

def combo_name(modelo: str, ssp: str, esc_desc: str) -> str:
    return f"{modelo}_{ssp}_{UMBRAL_PRECIP}_{esc_desc}"

def combo_dirs(modelo: str, ssp: str, esc_desc: str):
    nombre = combo_name(modelo, ssp, esc_desc)
    combo_dir = TMP_DIR / nombre
    blocks_dir = combo_dir / "bloques_txt"
    states_dir = combo_dir / "estados"
    combo_dir.mkdir(parents=True, exist_ok=True)
    blocks_dir.mkdir(parents=True, exist_ok=True)
    states_dir.mkdir(parents=True, exist_ok=True)
    return nombre, combo_dir, blocks_dir, states_dir

def construir_bloques_10_anios(claves, block_years=10):
    if not claves:
        return []

    years = sorted({y for y, _ in claves})
    y_min = min(years)
    y_max = max(years)

    bloques = []
    y0 = y_min
    while y0 <= y_max:
        y1 = min(y0 + block_years - 1, y_max)
        claves_block = [(y, m) for (y, m) in claves if y0 <= y <= y1]
        if claves_block:
            bloques.append({
                "start_year": y0,
                "end_year": y1,
                "claves": claves_block
            })
        y0 = y1 + 1

    return bloques

def estado_inicial_combo():
    return {
        "V_emb": float(V_INICIAL),
        "routing_states": [0.0] * int(PARAMS["n"]),
        "last_date": None
    }

def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: Path, default=None):
    if not path.exists():
        return {} if default is None else default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def progress_path_for(combo_dir: Path):
    return combo_dir / "progress.json"

def load_progress(combo_dir: Path):
    p = progress_path_for(combo_dir)
    if not p.exists():
        prog = {
            "status": "new",
            "completed_blocks": []
        }
        save_json(p, prog)
        return prog
    return load_json(p, default={"status": "new", "completed_blocks": []})

def save_progress(combo_dir: Path, progress: dict):
    save_json(progress_path_for(combo_dir), progress)

def remove_file_if_exists(path_str):
    if not path_str:
        return
    p = Path(path_str)
    if p.exists():
        p.unlink()

def limpiar_ultimos_bloques_si_hay_reinicio(combo_dir: Path, final_xlsx: Path, n_remove=2):
    progress = load_progress(combo_dir)

    if final_xlsx.exists():
        progress["status"] = "finished"
        save_progress(combo_dir, progress)
        return progress

    completed = progress.get("completed_blocks", [])

    if len(completed) == 0:
        return progress

    n_remove = min(int(n_remove), len(completed))
    warn(f"Reanudación detectada. Se borrarán los últimos {n_remove} bloques para reprocesarlos.")

    for item in completed[-n_remove:]:
        remove_file_if_exists(item.get("block_file"))
        remove_file_if_exists(item.get("state_file"))
        remove_file_if_exists(item.get("prev4_file"))

    progress["completed_blocks"] = completed[:-n_remove]
    progress["status"] = "resumed_after_cleanup"
    save_progress(combo_dir, progress)
    return progress

def cargar_estado_desde_progreso(progress: dict):
    completed = progress.get("completed_blocks", [])
    if not completed:
        return estado_inicial_combo(), None

    last = completed[-1]
    state_file = Path(last["state_file"])
    prev4_file = Path(last["prev4_file"])

    estado = load_json(state_file, default=estado_inicial_combo())

    prev4 = None
    if prev4_file.exists():
        prev4 = np.load(prev4_file, allow_pickle=False)
        if prev4.size == 0:
            prev4 = None

    return estado, prev4

def guardar_estado_bloque(states_dir: Path, start_year: int, end_year: int, estado: dict, prev4):
    state_file = states_dir / f"state_{start_year}_{end_year}.json"
    prev4_file = states_dir / f"prev4_{start_year}_{end_year}.npy"

    estado_serializable = {
        "V_emb": float(estado["V_emb"]),
        "routing_states": [float(x) for x in estado["routing_states"]],
        "last_date": estado["last_date"]
    }
    save_json(state_file, estado_serializable)

    if prev4 is None:
        np.save(prev4_file, np.empty((0,), dtype="float32"), allow_pickle=False)
    else:
        np.save(prev4_file, prev4.astype("float32"), allow_pickle=False)

    return state_file, prev4_file

def leer_bloques_concatenados(progress: dict):
    bloques = progress.get("completed_blocks", [])
    dfs = []

    for item in bloques:
        block_file = Path(item["block_file"])
        if block_file.exists():
            dfb = read_block_txt(block_file)
            dfs.append(dfb)

    if not dfs:
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.floor("D")
    df = df.sort_values("fecha").drop_duplicates(subset=["fecha"], keep="last").reset_index(drop=True)
    return df

def actualizar_resumen_global_parcial(row_dict: dict):
    if RESUMEN_GLOBAL_PARCIAL_TXT.exists():
        df = read_resumen_txt(RESUMEN_GLOBAL_PARCIAL_TXT)
    else:
        df = pd.DataFrame()

    if not df.empty:
        mask = (
            (df["modelo"] == row_dict["modelo"]) &
            (df["ssp"] == row_dict["ssp"]) &
            (df["umbral_precip"] == row_dict["umbral_precip"]) &
            (df["escenario_descarga"] == row_dict["escenario_descarga"])
        )
        df = df.loc[~mask].copy()

    df = pd.concat([df, pd.DataFrame([row_dict])], ignore_index=True)
    save_resumen_txt(df, RESUMEN_GLOBAL_PARCIAL_TXT)


# ==========================================================
# 14) FUNCIÓN PRINCIPAL DE SIMULACIÓN POR BLOQUES TXT
# ==========================================================
def simular_futuro_combinacion_por_bloques(modelo: str, ssp: str, esc_desc: str):
    ppt_folder = PPT_CORR_DIR / f"{modelo}_{ssp}_{UMBRAL_PRECIP}"
    et_folder  = ET_CORR_DIR  / f"{modelo}_{ssp}"
    desc_csv   = DESC_PROY_DIR / f"proyeccion_{esc_desc}.csv"

    if not ppt_folder.exists():
        raise FileNotFoundError(f"No existe carpeta de precipitación: {ppt_folder}")
    if not et_folder.exists():
        raise FileNotFoundError(f"No existe carpeta de ET: {et_folder}")
    if not desc_csv.exists():
        raise FileNotFoundError(f"No existe CSV de descarga: {desc_csv}")

    print_paths_for_run(modelo, ssp, esc_desc, ppt_folder, et_folder, desc_csv)

    # Listar TIFF mensuales
    ppt_files = listar_tifs_por_anio_mes(ppt_folder)
    et_files  = listar_tifs_por_anio_mes(et_folder)

    claves = sorted(set(ppt_files.keys()) & set(et_files.keys()))
    if not claves:
        raise RuntimeError(f"No hay meses comunes entre precipitación y ET para {modelo}_{ssp}")

    bloques = construir_bloques_10_anios(claves, block_years=BLOCK_YEARS)

    # Leer descarga proyectada
    df_desc = cargar_descarga_proyectada(desc_csv)
    df_desc["fecha"] = pd.to_datetime(df_desc["fecha"]).dt.floor("D")
    desc_lookup = dict(zip(df_desc["fecha"], df_desc["descarga_proyectada_m3_s"]))

    nombre_combo, combo_dir, blocks_dir, states_dir = combo_dirs(modelo, ssp, esc_desc)
    out_xlsx = OUT_DIR / f"{nombre_combo}.xlsx"

    progress = limpiar_ultimos_bloques_si_hay_reinicio(
        combo_dir=combo_dir,
        final_xlsx=out_xlsx,
        n_remove=REPROCESS_LAST_N_BLOCKS
    )

    bloques_completados = {
        (b["start_year"], b["end_year"])
        for b in progress.get("completed_blocks", [])
    }

    estado, prev4 = cargar_estado_desde_progreso(progress)
    V_emb = safe_num(estado["V_emb"], V_INICIAL)
    routing_states = [safe_num(x, 0.0) for x in estado["routing_states"]]

    for bloque in bloques:
        y0 = bloque["start_year"]
        y1 = bloque["end_year"]

        if (y0, y1) in bloques_completados:
            ok(f"Bloque ya existe, se omite: {y0}-{y1}")
            continue

        header(f"PROCESANDO BLOQUE {y0}-{y1} | {nombre_combo}", char="=")

        rows_block = []
        claves_block = bloque["claves"]

        barra_bloque = tqdm(
            claves_block,
            desc=f"{nombre_combo} | {y0}-{y1}",
            unit="mes",
            colour="green"
        )

        for (year, month) in barra_bloque:
            barra_bloque.set_postfix(fecha=f"{year}-{month:02d}")

            ppt_path = ppt_files[(year, month)]
            et_path  = et_files[(year, month)]

            # --------------------------------------------------
            # 1) PRECIPITACIÓN EN CUENCA
            # REGLA: todo NA/nodata -> 0.0
            # --------------------------------------------------
            ppt_cu_stack_raw, ppt_cu_nodata = read_clip_stack(ppt_path, roi_cuenca_geoms)
            ppt_cu_stack = fill_precip_stack(ppt_cu_stack_raw, nodata=ppt_cu_nodata)

            # Corrección QM/QDM y saneamiento final
            ppt_cu = qm_correct_array(ppt_cu_stack, month, QM_TABLE)
            ppt_cu = np.nan_to_num(ppt_cu, nan=0.0, posinf=0.0, neginf=0.0)
            ppt_cu = np.maximum(ppt_cu, 0.0)

            ppt_cu_eff = ppt_cu * float(PARAMS["k_effP"])
            ppt_cu_eff = np.nan_to_num(ppt_cu_eff, nan=0.0, posinf=0.0, neginf=0.0)
            ppt_cu_eff = np.maximum(ppt_cu_eff, 0.0)

            # --------------------------------------------------
            # 2) AMC + ESCORRENTÍA
            # --------------------------------------------------
            amc5, prev4 = rolling_sum_5d(ppt_cu_eff, prev4=prev4)
            q_cu = runoff_scs(
                ppt_cu_eff, amc5, S1_CU, S2_CU, S3_CU,
                lambda_ia=float(PARAMS["lambda_ia"])
            )
            q_cu = np.nan_to_num(q_cu, nan=0.0, posinf=0.0, neginf=0.0)
            q_cu = np.maximum(q_cu, 0.0)

            q_mean = [nanmean_masked(q_cu[i], None, fill_all_nan=0.0) for i in range(q_cu.shape[0])]
            q_mean = [safe_num(v, 0.0) * float(PARAMS["k_q"]) for v in q_mean]

            ppt_cu_mean = [nanmean_masked(ppt_cu_eff[i], None, fill_all_nan=0.0) for i in range(ppt_cu_eff.shape[0])]
            ppt_cu_mean = [safe_num(v, 0.0) for v in ppt_cu_mean]

            # --------------------------------------------------
            # 3) PRECIPITACIÓN EN EMBALSE
            # REGLA: todo NA/nodata -> 0.0
            # --------------------------------------------------
            ppt_em_stack_raw, ppt_em_nodata = read_clip_stack(ppt_path, roi_embalse_geoms)
            ppt_em_stack = fill_precip_stack(ppt_em_stack_raw, nodata=ppt_em_nodata)

            ppt_em = qm_correct_array(ppt_em_stack, month, QM_TABLE)
            ppt_em = np.nan_to_num(ppt_em, nan=0.0, posinf=0.0, neginf=0.0)
            ppt_em = np.maximum(ppt_em, 0.0)

            ppt_em_mean = [nanmean_masked(ppt_em[i], None, fill_all_nan=0.0) for i in range(ppt_em.shape[0])]
            ppt_em_mean = [safe_num(v, 0.0) * float(PARAMS["k_effP"]) for v in ppt_em_mean]

            # --------------------------------------------------
            # 4) ET EN EMBALSE
            # No es temperatura, así que si falta un día:
            # usa promedio mensual del mismo mes; si todo el mes falta -> 0
            # --------------------------------------------------
            et_em_stack_raw, et_em_nodata = read_clip_stack(et_path, roi_embalse_geoms)
            et_em_stack = to_nan(et_em_stack_raw, nodata=et_em_nodata)

            et_em_mean_raw = [nanmean_masked(et_em_stack[i], None, fill_all_nan=np.nan) for i in range(et_em_stack.shape[0])]
            et_em_mean = fill_generic_daily_series(et_em_mean_raw, default_if_all_nan=0.0)

            ndays = calendar.monthrange(year, month)[1]
            T = min(ndays, len(q_mean), len(ppt_em_mean), len(et_em_mean), len(ppt_cu_mean))

            for d in range(T):
                fecha = pd.Timestamp(datetime(year, month, d + 1)).floor("D")

                esc_sin_ruteo = safe_num(q_mean[d], 0.0)
                esc_ruteada, routing_states = route_nash_cascade_step(
                    esc_sin_ruteo,
                    states=routing_states,
                    n=int(PARAMS["n"]),
                    K=float(PARAMS["K"])
                )
                esc_ruteada = safe_num(esc_ruteada, 0.0)

                descarga_m3_s = desc_lookup.get(fecha, None)
                if descarga_m3_s is None:
                    continue

                descarga_m3_s = safe_num(descarga_m3_s, 0.0)
                Vd = descarga_m3_s * 86400.0

                # En precipitación: si falta -> 0.0
                p_mm = safe_num(ppt_em_mean[d], 0.0)

                # ET segura por promedio mensual si faltó
                et_mm = safe_num(et_em_mean[d], 0.0)

                # Evita heredar NaN en volumen
                if not np.isfinite(V_emb):
                    warn(f"V_emb no finito antes del balance en {fecha}. Se reinicia desde V_INICIAL.")
                    V_prev = float(V_INICIAL)
                else:
                    V_prev = float(V_emb)

                V_emb, V_vert, A_i, V_clima, V_esc, Vs_vol, B_i, Vd = balance_embalse_diario(
                    V_prev, esc_ruteada, p_mm, et_mm,
                    AREA_CUENCA_TOTAL, Vd,
                    Vs_m_d=float(PARAMS["Vs"]),
                    k_infil=float(PARAMS["k_infil"]),
                    k_et=float(PARAMS["k_et"])
                )

                # Protecciones finales contra NaN
                if not np.isfinite(V_emb):
                    warn(f"V_emb no finito en {fecha}. Se reemplaza por V_prev.")
                    V_emb = float(V_prev)

                if not np.isfinite(V_vert):
                    V_vert = 0.0
                if not np.isfinite(A_i):
                    A_i = area_embalse(V_prev)
                if not np.isfinite(V_clima):
                    V_clima = 0.0
                if not np.isfinite(V_esc):
                    V_esc = 0.0
                if not np.isfinite(Vs_vol):
                    Vs_vol = 0.0
                if not np.isfinite(B_i):
                    B_i = float(V_emb)

                # Corrección puntual del volumen
                if fecha in CORRECCIONES_UTIL_HM3:
                    V_emb = volumen_util_a_total_m3(CORRECCIONES_UTIL_HM3[fecha])
                    V_emb = max(VOL_MUERTO, min(V_emb, VOL_MAX))

                rows_block.append({
                    "fecha": fecha,
                    "modelo": modelo,
                    "ssp": ssp,
                    "umbral_precip": UMBRAL_PRECIP,
                    "escenario_descarga": esc_desc,
                    "ppt_cuenca_eff_mm": safe_num(ppt_cu_mean[d], 0.0),
                    "esc_mm_sin_ruteo": esc_sin_ruteo,
                    "esc_mm_ruteada": safe_num(esc_ruteada, 0.0),
                    "precip_mm_embalse": p_mm,
                    "et_mm_embalse": et_mm,
                    "descarga_proyectada_m3_s": descarga_m3_s,
                    "descarga_proyectada_m3_dia": safe_num(Vd, 0.0),
                    "V_emb_m3": safe_num(V_emb, V_prev),
                    "V_emb_hm3": safe_num(V_emb, V_prev) / 1e6,
                    "V_emb_util_hm3": max((safe_num(V_emb, V_prev) - VOL_MUERTO) / 1e6, 0.0),
                    "V_vertido_m3": safe_num(V_vert, 0.0),
                    "area_embalse_m2": safe_num(A_i, area_embalse(V_prev)),
                    "V_clima_m3": safe_num(V_clima, 0.0),
                    "V_esc_m3": safe_num(V_esc, 0.0),
                    "Vs_vol_m3": safe_num(Vs_vol, 0.0),
                    "balance_bruto_m3": safe_num(B_i, safe_num(V_emb, V_prev)),
                    "corregido_volumen": int(fecha in CORRECCIONES_UTIL_HM3),
                    "vol_util_objetivo_hm3": CORRECCIONES_UTIL_HM3.get(fecha, np.nan)
                })

            # Liberar RAM por mes
            del ppt_cu_stack_raw, ppt_cu_stack, ppt_cu, ppt_cu_eff
            del amc5, q_cu
            del ppt_em_stack_raw, ppt_em_stack, ppt_em
            del et_em_stack_raw, et_em_stack
            gc.collect()

        barra_bloque.close()

        df_block = pd.DataFrame(rows_block)
        if not df_block.empty:
            df_block["fecha"] = pd.to_datetime(df_block["fecha"]).dt.floor("D")
            df_block = df_block.sort_values("fecha").drop_duplicates(subset=["fecha"], keep="last").reset_index(drop=True)

        block_file = blocks_dir / f"serie_{y0}_{y1}.txt"
        save_block_txt(df_block, block_file)

        estado = {
            "V_emb": safe_num(V_emb, V_INICIAL),
            "routing_states": [safe_num(x, 0.0) for x in routing_states],
            "last_date": None if df_block.empty else str(pd.to_datetime(df_block["fecha"].max()).date())
        }

        state_file, prev4_file = guardar_estado_bloque(
            states_dir=states_dir,
            start_year=y0,
            end_year=y1,
            estado=estado,
            prev4=prev4
        )

        progress = load_progress(combo_dir)
        progress["status"] = "running"
        progress["completed_blocks"].append({
            "start_year": y0,
            "end_year": y1,
            "block_file": str(block_file),
            "state_file": str(state_file),
            "prev4_file": str(prev4_file)
        })
        save_progress(combo_dir, progress)

        ok(f"Bloque TXT guardado: {block_file.name}")

        del rows_block, df_block
        gc.collect()

    # Unir bloques al final
    progress = load_progress(combo_dir)
    df_res = leer_bloques_concatenados(progress)

    if df_res.empty or len(df_res) < 100:
        raise RuntimeError(f"La serie final quedó vacía o demasiado corta para {nombre_combo}")

    df_ctrl = df_res[df_res["corregido_volumen"] == 1].copy()
    if len(df_ctrl) >= 1:
        met_ctrl = _compute_metrics(df_ctrl["V_emb_util_hm3"], df_ctrl["vol_util_objetivo_hm3"])
    else:
        met_ctrl = {k: np.nan for k in ["n", "R2", "RMSE", "NSE", "RMSE_n", "Bias", "PBIAS", "KGE"]}

    progress["status"] = "finished"
    save_progress(combo_dir, progress)

    return df_res, met_ctrl, ppt_folder, et_folder, desc_csv, combo_dir


# ==========================================================
# 15) EJECUCIÓN MULTICOMBINACIÓN
# ==========================================================
header("SIMULACIÓN FUTURA EL PAÑE | u0p5 | BLOQUES TXT | CORREGIDO")

combinaciones = []
for modelo in MODELOS:
    for ssp in SSP_LIST:
        for esc_desc in ESCENARIOS_DESCARGA:
            combinaciones.append((modelo, ssp, esc_desc))

if RESUMEN_GLOBAL_PARCIAL_TXT.exists():
    try:
        resumen_global = read_resumen_txt(RESUMEN_GLOBAL_PARCIAL_TXT).to_dict(orient="records")
    except Exception:
        resumen_global = []
else:
    resumen_global = []

barra_total = tqdm(combinaciones, desc="Combinaciones totales", unit="combo", colour="blue")

for modelo, ssp, esc_desc in barra_total:
    barra_total.set_postfix(modelo=modelo, ssp=ssp, esc=esc_desc)

    nombre_combo = combo_name(modelo, ssp, esc_desc)
    out_xlsx = OUT_DIR / f"{nombre_combo}.xlsx"

    try:
        if out_xlsx.exists():
            ok(f"Ya existe Excel final, se omite: {out_xlsx.name}")

            row_resumen = {
                "modelo": modelo,
                "ssp": ssp,
                "umbral_precip": UMBRAL_PRECIP,
                "escenario_descarga": esc_desc,
                "excel": str(out_xlsx),
                "n": np.nan,
                "R2": np.nan,
                "RMSE": np.nan,
                "NSE": np.nan,
                "RMSE_n": np.nan,
                "Bias": np.nan,
                "PBIAS": np.nan,
                "KGE": np.nan,
                "estado": "ya_existia"
            }
            actualizar_resumen_global_parcial(row_resumen)
            continue

        df_res, met_ctrl, ppt_folder, et_folder, desc_csv, combo_dir = simular_futuro_combinacion_por_bloques(
            modelo, ssp, esc_desc
        )

        df_paths = pd.DataFrame([
            {"entrada": "ppt_corregida", "ruta": str(ppt_folder)},
            {"entrada": "et_corregido", "ruta": str(et_folder)},
            {"entrada": "descarga_proyectada", "ruta": str(desc_csv)},
            {"entrada": "qm_model", "ruta": str(QM_MODEL_PATH)},
            {"entrada": "lulc", "ruta": str(LULC_TIF)},
            {"entrada": "soil", "ruta": str(SOIL_TIF)},
            {"entrada": "cuenca", "ruta": str(AOI_CUENCA_PATH)},
            {"entrada": "embalse", "ruta": str(AOI_EMBALSE_PATH)},
            {"entrada": "tmp_bloques_txt", "ruta": str(combo_dir)},
        ])

        df_params = pd.DataFrame([
            {"parametro": k, "valor": v} for k, v in PARAMS.items()
        ])

        df_props = pd.DataFrame([
            {"propiedad": "VOL_MUERTO_m3", "valor": VOL_MUERTO},
            {"propiedad": "VOL_MAX_m3", "valor": VOL_MAX},
            {"propiedad": "AREA_CUENCA_TOTAL_m2", "valor": AREA_CUENCA_TOTAL},
            {"propiedad": "V_INICIAL_m3", "valor": V_INICIAL},
            {"propiedad": "UMBRAL_PRECIP", "valor": UMBRAL_PRECIP},
            {"propiedad": "BLOCK_YEARS", "valor": BLOCK_YEARS},
            {"propiedad": "REPROCESS_LAST_N_BLOCKS", "valor": REPROCESS_LAST_N_BLOCKS},
            {"propiedad": "correccion_2019_12_31_util_hm3", "valor": 56.00},
            {"propiedad": "correccion_2025_12_31_util_hm3", "valor": 36.37},
        ])

        df_metricas = pd.DataFrame([{
            "modelo": modelo,
            "ssp": ssp,
            "umbral_precip": UMBRAL_PRECIP,
            "escenario_descarga": esc_desc,
            **met_ctrl
        }])

        progress = load_progress(combo_dir)
        df_bloques = pd.DataFrame(progress.get("completed_blocks", []))

        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            df_metricas.to_excel(writer, sheet_name="metricas_control", index=False)
            df_props.to_excel(writer, sheet_name="propiedades_modelo", index=False)
            df_params.to_excel(writer, sheet_name="parametros_hidrologicos", index=False)
            df_paths.to_excel(writer, sheet_name="rutas_usadas", index=False)
            df_bloques.to_excel(writer, sheet_name="bloques_procesados", index=False)
            df_res.to_excel(writer, sheet_name="serie_simulada", index=False)

        row_resumen = {
            "modelo": modelo,
            "ssp": ssp,
            "umbral_precip": UMBRAL_PRECIP,
            "escenario_descarga": esc_desc,
            "excel": str(out_xlsx),
            **met_ctrl,
            "estado": "ok"
        }
        actualizar_resumen_global_parcial(row_resumen)

        ok(f"Excel generado: {out_xlsx.name}")

        del df_res, df_metricas, df_props, df_params, df_paths, df_bloques
        gc.collect()

    except Exception as e:
        err(f"{nombre_combo}: {e}")

        row_resumen = {
            "modelo": modelo,
            "ssp": ssp,
            "umbral_precip": UMBRAL_PRECIP,
            "escenario_descarga": esc_desc,
            "excel": str(out_xlsx),
            "n": np.nan,
            "R2": np.nan,
            "RMSE": np.nan,
            "NSE": np.nan,
            "RMSE_n": np.nan,
            "Bias": np.nan,
            "PBIAS": np.nan,
            "KGE": np.nan,
            "estado": "error",
            "error": str(e),
        }
        actualizar_resumen_global_parcial(row_resumen)

barra_total.close()


# ==========================================================
# 16) RESUMEN GLOBAL FINAL
# ==========================================================
if RESUMEN_GLOBAL_PARCIAL_TXT.exists():
    df_resumen_global = read_resumen_txt(RESUMEN_GLOBAL_PARCIAL_TXT)
else:
    df_resumen_global = pd.DataFrame(resumen_global)

out_resumen = OUT_DIR / "resumen_global_u0p5.xlsx"

with pd.ExcelWriter(out_resumen, engine="openpyxl") as writer:
    df_resumen_global.to_excel(writer, sheet_name="resumen_global", index=False)

header("PROCESO COMPLETADO", char="█")
print(f"  Carpeta de salida           : {OUT_DIR}")
print(f"  Carpeta temporal TXT        : {TMP_DIR}")
print(f"  Resumen parcial TXT         : {RESUMEN_GLOBAL_PARCIAL_TXT}")
print(f"  Excel resumen global        : {out_resumen}")
print(f"  Total combinaciones         : {len(combinaciones)}")

## 5.5 · Post-procesamiento — agregar temperatura a los Excels generados

Estrategia:

1. Extraer la serie de temperatura a TXT.
2. Unir TXT + Excel original.
3. Generar el Excel final con todas las variables.


In [ ]:
# ==========================================================
# AGREGAR TEMPERATURA A EXCELS YA GENERADOS
# ESTRATEGIA:
#   1) EXTRAER TEMPERATURA A TXT
#   2) UNIR TXT + EXCEL ORIGINAL
#   3) GENERAR NUEVO EXCEL FINAL
#
# REANUDACIÓN:
#   - si se corta en extracción TXT, borra el último TXT
#   - si se corta en unión Excel, borra el último Excel
# ==========================================================

import re
import gc
import json
import calendar
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")


# ==========================================================
# 1) CONFIGURACIÓN
# ==========================================================
BASE_DIR = Path("/content/drive/MyDrive/Project001")

SHP_DIR = BASE_DIR / "shp"
AOI_CUENCA_PATH  = SHP_DIR / "Cuenca-es.geojson"
AOI_EMBALSE_PATH = SHP_DIR / "Embalse.geojson"

CLIMA_DIR = BASE_DIR / "DATOS_CLIMATICOS_TOTALES"
IN_DIR    = BASE_DIR / "resultados_pane_futuro_u0p5_excel"

CANDIDATAS_TMAX = [
    CLIMA_DIR / "TEMPERATURA_MAXIMA_CORREGIDA_QDM",
    CLIMA_DIR / "TEMPERATURA_MAX_CORREGIDA_QDM",
]

CANDIDATAS_TMIN = [
    CLIMA_DIR / "TEMPERATURA_MINIMA_CORREGIDA_QDM",
    CLIMA_DIR / "TEMPERATURA_MIN_CORREGIDA_QDM",
]

OUT_DIR = BASE_DIR / "resultados_pane_futuro_u0p5_excel_con_temperatura"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TMP_DIR = OUT_DIR / "_tmp_temperatura_txt"
TMP_DIR.mkdir(parents=True, exist_ok=True)

CTRL_DIR = OUT_DIR / "_tmp_control"
CTRL_DIR.mkdir(parents=True, exist_ok=True)

PROGRESS_JSON = CTRL_DIR / "progress_temperatura.json"

SOBREESCRIBIR = False


# ==========================================================
# 2) UTILIDADES
# ==========================================================
def header(txt, char="="):
    print("\n" + char * 90)
    print(txt)
    print(char * 90)

def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: Path, default=None):
    if not path.exists():
        return {} if default is None else default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_progress():
    default = {
        "status": "new",
        "completed_txt": [],
        "completed_excel": [],
        "last_txt": None,
        "last_excel": None,
        "phase": None
    }
    if not PROGRESS_JSON.exists():
        save_json(PROGRESS_JSON, default)
        return default
    return load_json(PROGRESS_JSON, default=default)

def save_progress(progress):
    save_json(PROGRESS_JSON, progress)

def remove_file_if_exists(path_str):
    if not path_str:
        return
    p = Path(path_str)
    if p.exists():
        p.unlink()
        print(f"[RESTART] Se borró: {p.name}")

def limpiar_reinicio():
    progress = load_progress()

    if progress.get("status") == "running":
        fase = progress.get("phase")

        if fase == "txt":
            remove_file_if_exists(progress.get("last_txt"))
            progress["completed_txt"] = [
                x for x in progress.get("completed_txt", [])
                if x != progress.get("last_txt")
            ]
            progress["last_txt"] = None

        elif fase == "excel":
            remove_file_if_exists(progress.get("last_excel"))
            progress["completed_excel"] = [
                x for x in progress.get("completed_excel", [])
                if x != progress.get("last_excel")
            ]
            progress["last_excel"] = None

        progress["status"] = "resumed_after_cleanup"
        progress["phase"] = None
        save_progress(progress)

    return progress

def primera_ruta_existente(lista_rutas):
    for ruta in lista_rutas:
        if ruta.exists():
            return ruta
    return None

DIR_TMAX = primera_ruta_existente(CANDIDATAS_TMAX)
DIR_TMIN = primera_ruta_existente(CANDIDATAS_TMIN)

if DIR_TMAX is None:
    raise FileNotFoundError("No se encontró carpeta de tasmax corregida.")
if DIR_TMIN is None:
    raise FileNotFoundError("No se encontró carpeta de tasmin corregida.")


# ==========================================================
# 3) GEOMETRÍAS Y RASTERS
# ==========================================================
def load_geoms(geojson_path: Path):
    if not geojson_path.exists():
        raise FileNotFoundError(f"No existe AOI: {geojson_path}")
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        raise ValueError(f"AOI vacío: {geojson_path}")
    geom = gdf.geometry.union_all()
    return [geom]

def read_clip_stack(tif_path: Path, geoms):
    if not tif_path.exists():
        raise FileNotFoundError(f"No existe raster: {tif_path}")
    with rasterio.open(tif_path) as src:
        out_img, _ = mask(src, geoms, crop=True, filled=True)
        nodata = src.nodata
        descriptions = list(src.descriptions) if src.descriptions else []
    return out_img, nodata, descriptions

def to_nan(arr: np.ndarray, nodata=None) -> np.ndarray:
    a = np.asarray(arr, dtype="float64").copy()
    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)
    a[~np.isfinite(a)] = np.nan
    return a

def nanmean_masked(a: np.ndarray, nodata=None, fill_all_nan=np.nan) -> float:
    a = to_nan(a, nodata=nodata)
    if np.all(np.isnan(a)):
        return float(fill_all_nan)
    return float(np.nanmean(a))

def fill_temperature_series(values, default_if_all_nan=0.0):
    s = pd.to_numeric(pd.Series(values), errors="coerce").astype(float)
    vals = s.to_numpy()

    if np.all(np.isnan(vals)):
        return [float(default_if_all_nan)] * len(vals)

    monthly_mean = float(np.nanmean(vals))
    i = 0
    n = len(vals)

    while i < n:
        if np.isfinite(vals[i]):
            i += 1
            continue

        j = i
        while j < n and not np.isfinite(vals[j]):
            j += 1

        run_len = j - i

        if run_len == 1:
            k = i
            prev_ok = (k - 1 >= 0) and np.isfinite(vals[k - 1])
            next_ok = (k + 1 < n) and np.isfinite(vals[k + 1])

            if prev_ok and next_ok:
                vals[k] = 0.5 * (vals[k - 1] + vals[k + 1])
            else:
                vals[k] = monthly_mean
        else:
            vals[i:j] = monthly_mean

        i = j

    vals = np.where(np.isfinite(vals), vals, monthly_mean)
    return [float(v) for v in vals]


# ==========================================================
# 4) FECHAS / PARSEO
# ==========================================================
def extraer_anio_mes(nombre_archivo):
    m = re.search(r'(\d{4})_(\d{2})(?:\D|$)', nombre_archivo)
    if m:
        anio = int(m.group(1))
        mes = int(m.group(2))
        if 1 <= mes <= 12:
            return anio, mes
    return None, None

def listar_tifs_por_anio_mes(carpeta):
    salida = {}
    for p in sorted(carpeta.glob("*.tif")):
        anio, mes = extraer_anio_mes(p.name)
        if anio is not None:
            salida[(anio, mes)] = p
    return salida

def fechas_desde_descripciones_o_mes(descriptions, n_bandas, anio, mes):
    fechas = []

    if len(descriptions) == n_bandas and all(d is not None and str(d).strip() != "" for d in descriptions):
        ok = True
        for d in descriptions:
            try:
                fechas.append(pd.Timestamp(datetime.strptime(str(d), "%Y%m%d")).floor("D"))
            except Exception:
                ok = False
                break
        if ok:
            return fechas

    n_dias_mes = calendar.monthrange(anio, mes)[1]
    if n_bandas > n_dias_mes:
        raise ValueError(f"El archivo tiene {n_bandas} bandas, pero {anio}-{mes:02d} solo tiene {n_dias_mes} días.")

    return [pd.Timestamp(datetime(anio, mes, d)).floor("D") for d in range(1, n_bandas + 1)]

def buscar_subcarpeta_escenario(base_dir: Path, nombre_escenario: str) -> Path:
    path = base_dir / nombre_escenario
    if path.exists() and path.is_dir():
        return path
    raise FileNotFoundError(f"No existe carpeta de escenario: {path}")

def extraer_combo_desde_nombre_excel(nombre_archivo: str):
    patron = r"^(.*?)_(ssp245|ssp585)_(u0p1|u0p5)_(ESC_\d+)\.xlsx$"
    m = re.match(patron, nombre_archivo)
    if not m:
        return None
    return {
        "modelo": m.group(1),
        "ssp": m.group(2),
        "umbral": m.group(3),
        "escenario_descarga": m.group(4)
    }


# ==========================================================
# 5) EXTRACCIÓN DE TEMPERATURA
# ==========================================================
def leer_temperatura_promedio_diaria(ruta_tif: Path, geoms):
    stack, nodata, descriptions = read_clip_stack(ruta_tif, geoms)
    anio, mes = extraer_anio_mes(ruta_tif.name)

    if anio is None:
        raise ValueError(f"No se pudo inferir año/mes del archivo: {ruta_tif.name}")

    fechas = fechas_desde_descripciones_o_mes(descriptions, stack.shape[0], anio, mes)

    valores = [
        nanmean_masked(stack[i], nodata=nodata, fill_all_nan=np.nan)
        for i in range(stack.shape[0])
    ]

    valores = fill_temperature_series(valores, default_if_all_nan=0.0)

    df = pd.DataFrame({
        "fecha": pd.to_datetime(fechas).floor("D"),
        "valor": valores
    })
    return df

def construir_serie_temperatura_escenario(dir_esc_tif: Path, geoms, nombre_columna: str):
    files = listar_tifs_por_anio_mes(dir_esc_tif)
    claves = sorted(files.keys())

    if not claves:
        raise RuntimeError(f"No hay TIFF en {dir_esc_tif}")

    dfs = []
    barra = tqdm(claves, desc=nombre_columna, unit="mes", colour="green")

    for clave in barra:
        ruta_tif = files[clave]
        barra.set_postfix(archivo=ruta_tif.name)

        df_mes = leer_temperatura_promedio_diaria(ruta_tif, geoms)
        df_mes = df_mes.rename(columns={"valor": nombre_columna})
        dfs.append(df_mes)

    barra.close()

    df = pd.concat(dfs, ignore_index=True)
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.floor("D")
    df = df.sort_values("fecha").drop_duplicates(subset=["fecha"], keep="last").reset_index(drop=True)
    return df

def txt_path_para_excel(excel_in: Path):
    return TMP_DIR / f"{excel_in.stem}_temperatura.txt"

def extraer_temperatura_a_txt(excel_in: Path):
    combo = extraer_combo_desde_nombre_excel(excel_in.name)
    if combo is None:
        raise ValueError(f"No coincide con patrón esperado: {excel_in.name}")

    modelo = combo["modelo"]
    ssp = combo["ssp"]
    nombre_escenario_temp = f"{modelo}_{ssp}"

    dir_tmax_esc = buscar_subcarpeta_escenario(DIR_TMAX, nombre_escenario_temp)
    dir_tmin_esc = buscar_subcarpeta_escenario(DIR_TMIN, nombre_escenario_temp)

    df_tmax_cu = construir_serie_temperatura_escenario(dir_tmax_esc, roi_cuenca_geoms,  "tasmax_cuenca_C")
    df_tmin_cu = construir_serie_temperatura_escenario(dir_tmin_esc, roi_cuenca_geoms,  "tasmin_cuenca_C")
    df_tmax_em = construir_serie_temperatura_escenario(dir_tmax_esc, roi_embalse_geoms, "tasmax_embalse_C")
    df_tmin_em = construir_serie_temperatura_escenario(dir_tmin_esc, roi_embalse_geoms, "tasmin_embalse_C")

    df_temp = df_tmax_cu.merge(df_tmin_cu, on="fecha", how="outer")
    df_temp = df_temp.merge(df_tmax_em, on="fecha", how="outer")
    df_temp = df_temp.merge(df_tmin_em, on="fecha", how="outer")
    df_temp = df_temp.sort_values("fecha").reset_index(drop=True)

    txt_out = txt_path_para_excel(excel_in)
    df_temp.to_csv(txt_out, sep="\t", index=False, encoding="utf-8")

    del df_tmax_cu, df_tmin_cu, df_tmax_em, df_tmin_em, df_temp
    gc.collect()

    return txt_out


# ==========================================================
# 6) UNIÓN TXT + EXCEL
# ==========================================================
def unir_txt_con_excel(excel_in: Path, txt_temp: Path, excel_out: Path):
    combo = extraer_combo_desde_nombre_excel(excel_in.name)
    if combo is None:
        raise ValueError(f"No coincide con patrón esperado: {excel_in.name}")

    xls = pd.ExcelFile(excel_in)
    hojas = xls.sheet_names

    if "serie_simulada" not in hojas:
        raise ValueError(f"{excel_in.name} no contiene hoja 'serie_simulada'")

    df_serie = pd.read_excel(excel_in, sheet_name="serie_simulada")
    df_serie["fecha"] = pd.to_datetime(df_serie["fecha"], errors="coerce").dt.floor("D")
    df_serie = df_serie.dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)

    df_temp = pd.read_csv(txt_temp, sep="\t", encoding="utf-8")
    df_temp["fecha"] = pd.to_datetime(df_temp["fecha"], errors="coerce").dt.floor("D")
    df_temp = df_temp.dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)

    df_serie_temp = df_serie.merge(df_temp, on="fecha", how="left")

    df_temp_resumen = pd.DataFrame([{
        "modelo": combo["modelo"],
        "ssp": combo["ssp"],
        "umbral_precip": combo["umbral"],
        "escenario_descarga": combo["escenario_descarga"],
        "fecha_inicio_serie": df_serie_temp["fecha"].min(),
        "fecha_fin_serie": df_serie_temp["fecha"].max(),
        "n_registros": len(df_serie_temp),
        "tasmax_cuenca_media_C": pd.to_numeric(df_serie_temp["tasmax_cuenca_C"], errors="coerce").mean(),
        "tasmin_cuenca_media_C": pd.to_numeric(df_serie_temp["tasmin_cuenca_C"], errors="coerce").mean(),
        "tasmax_embalse_media_C": pd.to_numeric(df_serie_temp["tasmax_embalse_C"], errors="coerce").mean(),
        "tasmin_embalse_media_C": pd.to_numeric(df_serie_temp["tasmin_embalse_C"], errors="coerce").mean(),
    }])

    with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
        for hoja in hojas:
            df_tmp = pd.read_excel(excel_in, sheet_name=hoja)
            df_tmp.to_excel(writer, sheet_name=hoja, index=False)

        df_temp.to_excel(writer, sheet_name="temperatura_diaria", index=False)
        df_temp_resumen.to_excel(writer, sheet_name="temperatura_resumen", index=False)
        df_serie_temp.to_excel(writer, sheet_name="serie_simulada_temp", index=False)

    del df_serie, df_temp, df_serie_temp, df_temp_resumen
    gc.collect()


# ==========================================================
# 7) CARGA DE GEOMETRÍAS
# ==========================================================
roi_cuenca_geoms  = load_geoms(AOI_CUENCA_PATH)
roi_embalse_geoms = load_geoms(AOI_EMBALSE_PATH)


# ==========================================================
# 8) LIMPIEZA DE REINICIO
# ==========================================================
progress = limpiar_reinicio()


# ==========================================================
# 9) LISTA DE EXCELS
# ==========================================================
exceles = sorted([
    p for p in IN_DIR.glob("*.xlsx")
    if p.name != "resumen_global_u0p5.xlsx"
])

if not exceles:
    raise RuntimeError(f"No se encontraron Excel en {IN_DIR}")

header("FASE 1: EXTRACCIÓN DE TEMPERATURA A TXT")

completed_txt = set(load_progress().get("completed_txt", []))
barra_txt = tqdm(exceles, desc="Extracción TXT", unit="archivo", colour="blue")

for excel_in in barra_txt:
    txt_out = txt_path_para_excel(excel_in)
    barra_txt.set_postfix(archivo=excel_in.name)

    if str(txt_out) in completed_txt and txt_out.exists() and not SOBREESCRIBIR:
        continue

    if txt_out.exists() and not SOBREESCRIBIR:
        continue

    progress = load_progress()
    progress["status"] = "running"
    progress["phase"] = "txt"
    progress["last_txt"] = str(txt_out)
    save_progress(progress)

    try:
        extraer_temperatura_a_txt(excel_in)

        progress = load_progress()
        done = set(progress.get("completed_txt", []))
        done.add(str(txt_out))
        progress["completed_txt"] = sorted(done)
        progress["last_txt"] = None
        progress["status"] = "idle"
        progress["phase"] = None
        save_progress(progress)

    except Exception as e:
        print(f"[ERROR TXT] {excel_in.name}: {e}")
        raise

barra_txt.close()


# ==========================================================
# 10) FASE 2: UNIÓN FINAL AL EXCEL
# ==========================================================
header("FASE 2: UNIÓN TXT + EXCEL")

completed_excel = set(load_progress().get("completed_excel", []))
barra_excel = tqdm(exceles, desc="Generación Excel final", unit="archivo", colour="magenta")

for excel_in in barra_excel:
    excel_out = OUT_DIR / excel_in.name
    txt_temp = txt_path_para_excel(excel_in)

    barra_excel.set_postfix(archivo=excel_in.name)

    if not txt_temp.exists():
        raise FileNotFoundError(f"No existe TXT temporal para {excel_in.name}: {txt_temp}")

    if str(excel_out) in completed_excel and excel_out.exists() and not SOBREESCRIBIR:
        continue

    if excel_out.exists() and not SOBREESCRIBIR:
        continue

    progress = load_progress()
    progress["status"] = "running"
    progress["phase"] = "excel"
    progress["last_excel"] = str(excel_out)
    save_progress(progress)

    try:
        unir_txt_con_excel(excel_in, txt_temp, excel_out)

        progress = load_progress()
        done = set(progress.get("completed_excel", []))
        done.add(str(excel_out))
        progress["completed_excel"] = sorted(done)
        progress["last_excel"] = None
        progress["status"] = "idle"
        progress["phase"] = None
        save_progress(progress)

    except Exception as e:
        print(f"[ERROR EXCEL] {excel_in.name}: {e}")
        raise

barra_excel.close()


# ==========================================================
# 11) FIN
# ==========================================================
progress = load_progress()
progress["status"] = "finished"
progress["phase"] = None
progress["last_txt"] = None
progress["last_excel"] = None
save_progress(progress)

header("PROCESO COMPLETADO", char="█")
print(f"Entrada Excel : {IN_DIR}")
print(f"Salida Excel  : {OUT_DIR}")
print(f"TXT temporal  : {TMP_DIR}")
print(f"Control JSON  : {PROGRESS_JSON}")